# ProstT5 Probabilistic Folding Drafter Benchmark: AA -> 3Di

This notebook runs the full 100-protein folding-direction benchmark for the probabilistic/HMM-style drafter.

The workflow is intentionally single-mode now:

- Load the paired AA/3Di benchmark data.
- Build one family-specific drafter per benchmark protein.
- Run plain ProstT5 folding as the baseline.
- Run static and prefix-aware HMM-assisted folding.
- Save raw result CSVs, summaries, plots, and the query/family AA+3Di alignment export under the local `prostT5/results/prostT5_probabilistic_drafter_folding_100` directory.


In [ ]:
#@title Full 100-protein benchmark settings. { display-mode: "form" }

# Single full-evaluation mode. `MODE` is kept only as a label for saved summaries.
MODE = "evaluate"
BENCHMARK_PROTEIN_LIMIT = 100
RESULTS_SUBDIR = "prostT5_probabilistic_drafter_folding_100"
PART2_MSA_CACHE_SUBDIR = "prostT5_benchmarks/msa_hmms"
RUN_BENCHMARKS = True

# Draft lengths tested by the static and prefix-aware HMM assistants.
K_VALUES = [1, 3, 5, 8, 11, 15]

# Prefix context lengths built for the prefix-aware drafter. A later comparison
# cell may restrict the timed prefix benchmark to p=3 and p=5.
P_VALUES_OVERRIDE = [1, 2, 3, 4, 5]

USE_FP16 = True
NUM_WARMUP = 0
NUM_REPEATS = 1
HELD_OUT_N = 2
MAX_CONTEXT_P = 5

FAMILY_MSA_MODE = "env"
FAMILY_MSA_MAX_SEQS = 16
FAMILY_MSA_MIN_SEQ_LEN = 40
FAMILY_API_DELAY_S = 2.0

print(f"MODE label: {MODE}")
print(f"Benchmark protein limit: {BENCHMARK_PROTEIN_LIMIT}")
print(f"K values: {K_VALUES}")
print(f"Prefix context values built: {P_VALUES_OVERRIDE}")
print(f"Results subdir: {RESULTS_SUBDIR}")
print(f"Part 2 MSA cache: {PART2_MSA_CACHE_SUBDIR}")


MODE label: evaluate
Benchmark protein limit: 100
K values: [1, 3, 5, 8, 11, 15]
Prefix context values built: [1, 2, 3, 4, 5]
Results subdir: prostT5_probabilistic_drafter_folding_100
Part 2 MSA cache: prostT5_benchmarks/msa_hmms


## Setup

Install/import dependencies, choose the runtime device, and fix random seeds for reproducible stochastic behavior.


### What this cell does

Sets up the notebook runtime.

- Imports Python, PyTorch, Transformers, plotting/data packages, and helper libraries.
- Fixes the random seed for reproducibility.
- Selects the runtime device.
- Requires CUDA/GPU, because this folding benchmark is intended for Colab GPU.


In [ ]:
#@title Imports + seed. { display-mode: "form" }

import os, sys, time, statistics, json, importlib.util, subprocess, io, tarfile, zipfile, gc
from pathlib import Path
from collections import defaultdict

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Colab often misses these two packages. Install only when needed.
def _ensure_package(import_name: str, pip_name: str | None = None):
    if importlib.util.find_spec(import_name) is None:
        pip_name = pip_name or import_name
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

_ensure_package("sentencepiece")
_ensure_package("accelerate")
_ensure_package("requests")

import numpy as np
import pandas as pd
import torch
import requests
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM, GenerationConfig, PreTrainedModel, T5Config
from transformers.generation.utils import GenerationMixin
from transformers.modeling_outputs import Seq2SeqLMOutput

SEED = 0
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

REQUIRE_CUDA_RUNTIME = True
print(f"torch={torch.__version__}  device={device}  seed={SEED}")
if REQUIRE_CUDA_RUNTIME and device.type != "cuda":
    raise RuntimeError(
        "This notebook is configured for Colab GPU/CUDA. "
        "In Colab, switch Runtime -> Change runtime type -> GPU, then rerun."
    )
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True


torch=2.11.0+cu128  device=cuda:0  seed=0


## Configuration

This cell resolves benchmark data and output paths from the current checkout. Set `PROSTT5_DIR` or `PROST_DIR` if the kernel is running outside the repository; otherwise it searches the current working directory, parents, and common Colab checkout locations.


In [ ]:
#@title Local benchmark paths and output directories. { display-mode: "form" }

NOTEBOOK_DIR = Path.cwd().resolve()
EMBEDDED_BENCHMARK_AA_FASTA = '>P01308\nMALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN\n>P02798\nMDPNCSCASDGSCSCAGACKCKQCKCTSCKKSCCSCCPVGCAKCSQGCICKEASDKCSCCA\n>P62944\nMTDSKYFTTNKKGEIFELKAELNNEKKEKRKEAVKKVIAAMTVGKDVSSLFPDVVNCMQTDNLELKKLVYLYLMNYAKSQPDMAIMAVNSFVKDCEDPNPLIRALAVRTMGCIRVDKITEYLCEPLRKCLKDEDPYVRKTAAVCVAKLHDINAQMVEDQGFLDSLRDLIADSNPMVVANAVAALSEISESHPNSNLLDLNPQNINKLLTALNECTEWGQIFILDCLSNYNPKDDREAQSICERVTPRLSHANSAVVLSAVKVLMKFLELLPKDSDYYNMLLKKLAPPLVTLLSGEPEVQYVALRNINLIVQKRPEILKQEIKVFFVKYNDPIYVKLEKLDIMIRLASQANIAQVLAELKEYATEVDVDFVRKAVRAIGRCAIKVEQSAERCVSTLLDLIQTKVNYVVQEAIVVIRDIFRKYPNKYESIIATLCENLDSLDEPDARAAMIWIVGEYAERIDNADELLESFLEGFHDESTQVQLTLLTAIVKLFLKKPSETQELVQQVLSLATQDSDNPDLRDRGYIYWRLLSTDPVTAKEVVLSEKPLISEETDLIEPTLLDELICHIGSLASVYHKPPNAFVEGSHGIHRKHLPIHHGSTDAGDSPVGTTTATNLEQPQVIPSQGDLLGDLLNLDLGPPVNVPQVSSMQMGAVDLLGGGLDSLVGQSFIPSSVPATFAPSPTPAVVSSGLNDLFELSTGIGMAPGGYVAPKAVWLPAVKAKGLEISGTFTHRQGHIYMEMNFTNKALQHMTDFAIQFNKNSFGVIPSTPLAIHTPLMPNQSIDVSLPLNTLGPVMKMEPLNNLQVAVKNNIDVFYFSCLIPLNVLFVEDGKMERQVFLATWKDIPNENELQFQIKECHLNADTVSSKLQNNNVYTIAKRNVEGQDMLYQSLKLTNGIWILAELRIQPGNPNYTLSLKCRAPEVSQYIYQVYDSILKN\n>P01542\nTTCCPSIVARSNFNVCRLPGTPEALCATYTGCIIIPGATCPGDYAN\n>P43408\nMKNKVVVVTGVPGVGGTTLTQKTIEKLKEEGIEYKMVNFGTVMFEVAKEEGLVEDRDQMRKLDPDTQKRIQKLAGRKIAEMAKESNVIVDTHSTVKTPKGYLAGLPIWVLEELNPDIIVIVETSSDEILMRRLGDATRNRDIELTSDIDEHQFMNRCAAMAYGVLTGATVKIIKNRDGLLDKAVEELISVLK\n>P0AFL6\nMPIHDKSPRPQEFAAVDLGSNSFHMVIARVVDGAMQIIGRLKQRVHLADGLGPDNMLSEEAMTRGLNCLSLFAERLQGFSPASVCIVGTHTLRQALNATDFLKRAEKVIPYPIEIISGNEEARLIFMGVEHTQPEKGRKLVIDIGGGSTELVIGENFEPILVESRRMGCVSFAQLYFPGGVINKENFQRARMAAAQKLETLTWQFRIQGWNVAMGASGTIKAAHEVLMEMGEKDGIITPERLEKLVKEVLRHRNFASLSLPGLSEERKTVFVPGLAILCGVFDALAIRELRLSDGALREGVLYEMEGRFRHQDVRSRTASSLANQYHIDSEQARRVLDTTMQMYEQWREQQPKLAHPQLEALLRWAAMLHEVGLNINHSGLHRHSAYILQNSDLPGFNQEQQLMMATLVRYHRKAIKLDDLPRFTLFKKKQFLPLIQLLRLGVLLNNQRQATTTPPTLTLITDDSHWTLRFPHDWFSQNALVLLDLEKEQEYWEGVAGWRLKIEEESTPEIAA\n>P0ACJ8\nMVLGKPQTDPTLEWFLSHCHIHKYPSKSTLIHQGEKAETLYYIVKGSVAVLIKDEEGKEMILSYLNQGDFIGELGLFEEGQERSAWVRAKTACEVAEISYKKFRQLIQVNPDILMRLSAQMARRLQVTSEKVGNLAFLDVTGRIAQTLLNLAKQPDAMTHPDGMQIKITRQEIGQIVGCSRETVGRILKMLEDQNLISAHGKTIVVYGTR\n>P0A988\nMKFTVEREHLLKPLQQVSGPLGGRPTLPILGNLLLQVADGTLSLTGTDLEMEMVARVALVQPHEPGATTVPARKFFDICRGLPEGAEIAVQLEGERMLVRSGRSRFSLSTLPAADFPNLDDWQSEVEFTLPQATMKRLIEATQFSMAHQDVRYYLNGMLFETEGEELRTVATDGHRLAVCSMPIGQSLPSHSVIVPRKGVIELMRMLDGGDNPLRVQIGSNNIRAHVGDFIFTSKLVDGRFPDYRRVLPKNPDKHLEAGCDLLKQAFARAAILSNEKFRGVRLYVSENQLKITANNPEQEEAEEILDVTYSGAEMEIGFNVSYVLDVLNALKCENVRMMLTDSVSSVQIEDAASQSAAYVVMPMRL\n>P0A7Y4\nMLKQVEIFTDGSCLGNPGPGGYGAILRYRGREKTFSAGYTRTTNNRMELMAAIVALEALKEHCEVILSTDSQYVRQGITQWIHNWKKRGWKTADKKPVKNVDLWQRLDAALGQHQIKWEWVKGHAGHPENERCDELARAAAMNPTLEDTGYQVEV\n>Q57733\nMFGRDPFDSLFERMFKEFFATPMTGTTMIQSSTGIQISGKGFMPISIIEGDQHIKVIAWLPGVNKEDIILNAVGDTLEIRAKRSPLMITESERIIYSEIPEEEEIYRTIKLPATVKEENASAKFENGVLSVILPKAESSIKKGINIE\n>P22579\nMSQVWHNSNSQSNDVATSNDATGSNERNEKEPSLQGNKPGFVQQQQRITLPSLSALSTKEEDRRDSNGQQALTSHAAHILGYPPPHSNAMPSIATDSALKQPHEYHPRPKSSSSSPSINASLMNAGPAPLPTVGAASFSLSRFDNPLPIKAPVHTEEPKSYNGLQEEEKATQRPQDCKEVPAGVQPADAPDPSSNHADANDDNNNNENSHDEDADYRPLNVKDALSYLEQVKFQFSSRPDIYNLFLDIMKDFKSQAIDTPGVIERVSTLFRGYPILIQGFNTFLPQGYRIECSSNPDDPIRVTTPMGTTTVNNNISPSGRGTTDAQELGSFPESDGNGVQQPSNVPMVPSSVYQSEQNQDQQQSLPLLATSSGLPSIQQPEMPAHRQIPQSQSLVPQEDAKKNVDVEFSQAISYVNKIKTRFADQPDIYKHFLEILQTYQREQKPINEVYAQVTHLFQNAPDLLEDFKKFLPDSSASANQQVQHAQQHAQQQHEAQMHAQAQAQAQAQAQVEQQKQQQQFLYPASGYYGHPSNRGIPQQNLPPIGSFSPPTNGSTVHEAYQDQQHMQPPHFMPLPSIVQHGPNMVHQGIANENPPLSDLRTSLTEQYAPSSIQHQQQHPQSISPIANTQYGDIPVRPEIDLDPSIVPVVPEPTEPIENNISLNEEVTFFEKAKRYIGNKHLYTEFLKILNLYSQDILDLDDLVEKVDFYLGSNKELFTWFKNFVGYQEKTKCIENIVHEKHRLDLDLCEAFGPSYKRLPKSDTFMPCSGRDDMCWEVLNDEWVGHPVWASEDSGFIAHRKNQYEETLFKIEEERHEYDFYIESNLRTIQCLETIVNKIENMTENEKANFKLPPGLGHTSMTIYKKVIRKVYDKERGFEIIDALHEHPAVTAPVVLKRLKQKDEEWRRAQREWNKVWRELEQKVFFKSLDHLGLTFKQADKKLLTTKQLISEISSIKVDQTNKKIHWLTPKPKSQLDFDFPDKNIFYDILCLADTFITHTTAYSNPDKERLKDLLKYFISLFFSISFEKIEESLYSHKQNVSESSGSDDGSSIASRKRPYQQEMSLLDILHRSRYQKLKRSNDEDGKVPQLSEPPEEEPNTIEEEELIDEEAKNPWLTGNLVEEANSQGIIQNRSIFNLFANTNIYIFFRHWTTIYERLLEIKQMNERVTKEINTRSTVTFAKDLDLLSSQLSEMGLDFVGEDAYKQVLRLSRRLINGDLEHQWFEESLRQAYNNKAFKLYTIDKVTQSLVKHAHTLMTDAKTAEIMALFVKDRNASTTSAKDQIIYRLQVRSHMSNTENMFRIEFDKRTLHVSIQYIALDDLTLKEPKADEDKWKYYVTSYALPHPTEGIPHEKLKIPFLERLIEFGQDIDGTEVDEEFSPEGISVSTLKIKIQPITYQLHIENGSYDVFTRKATNKYPTIANDNTQKGMVSQKKELISKFLDCAVGLRNNLDEAQKLSMQKKWENLKDSIAKTSAGNQGIESETEKGKITKQEQSDNLDSSTASVLPASITTVPQDDNIETTGNTESSDKGAKIQ\n>P15378\nMLSILFIFGLILGSFYYTAGCRIPLHLSIIAPRSSCPFCRRTLTPAELIPILSFLFQKGKCKSCGHRISFMYPAAELVTACLFAAAGIRFGISLELFPAVVFISLLIIVAVTDIHFMLIPNRILIFFLPFLAAARLISPLDSWYAGLLGAAAGFLFLAVIAAITHGGVGGGDIKLFAVIGFVLGVKMLAAAFFFSVLIGALYGAAAVLTGRLAKRQPLPFAPAIAAGSILAYLYGDSIISFYIKMALG\n>P34690\nMREVISIHVGQAGVQIGNACWELYCLEHGIQPDGTMPTQSTNEGESFTTFFSDTGSGRYVPRSIFVDLEPTVVDEIRTGTYKKLFHPEQMITGKEDAANNYARGHYTVGKELIDTVLDRIRRLADNCSGLQGFFVFHSFGGGTGSGFTSLLMERLSVDYGKKSKLEFSIYPAPQVSTAVVEPYNSILTTHTTLEHSDCAFMVDNEAIYDICRRNLDVERPSYTNLNRIISQVVSSITASLRFDGALNVDLNEFQTNLVPYPRIHFPLAAYTPLISAEKAYHEALSVSDITNSCFEPANQMVKCDPRHGKYMAVCLLYRGDVVPKDVNTAIAAIKTKRTIQFVDWCPTGFKVGINYQPPTVVPGGDLAKVPRAVCMLSNTTAIAEAWSRLDYKFDLMYAKRAFVHWYVGEGMEEGEFTEAREDLAALEKDYEEVGADSNEGGEEEGEEY\n>P0A7U3\nMPRSLKKGPFIDLHLLKKVEKAVESGDKKPLRTWSRRSTIFPNMIGLTIAVHNGRQHVPVFVTDEMVGHKLGEFAPTRTYRGHAADKKAKKK\n>P63165\nMSDQEAKPSTEDLGDKKEGEYIKLKVIGQDSSEIHFKVKMTTHLKKLKESYCQRQGVPMNSLRFLFEGQRIADNHTPKELGMEEEDVIEVYQEQTGGHSTV\n>P0A7N4\nMAVQQNKPTRSKRGMRRSHDALTAVTSLSVDKTSGEKHLRHHITADGYYRGRKVIAK\n>P24311\nMFPLVKSALNRLQVRSIQQTMARQSHQKRTPDFHDKYGNAVLASGATFCIVTWTYVATQVGIEWNLSPVGRVTPKEWRNQ\n>P56252\nSITKVFARTIFDSRGNPTVEVDLYTSKGLFRAAVPSGASTGVHEALEMRDGDKSKYHGKSVFNAVKNVNDVIVPEIIKSGLKVTQQKECDEFMCKLDGTENKSSLGANAILGVSLAICKAGAAELGIPLYRHIANLANYDEVILPVPAFNVINGGSHAGNKLAMQEFMILPTGATSFTEAMRMGTEVYHHLKAVIKARFGLDATAVGDEGGFAPNILNNKDALDLIQEAIKKAGYTGKIEIGMDVAASEFYKQNNIYDLDFKTANNDGSQKISGDQLRDMYMEFCKDFPIVSIEDPFDQDDWETWSKMTSGTTIQIVGDDLTVTNPKRITTAVEKKACKCLLLKVNQIGSVTESIDAHLLAKKNGWGTMVSHRSGETEDCFIADLVVGLCTGQIKTGAPCRSERLAKYNQILRIEEELGSGAKFAGKNFRAPS\n>P18253\nMSNCFFDVIANGQPLGRIVFKLFDDVVPKTAANFRALCTGEKGYGYAGSTFHRVIPQFMLQGGDFTRGNGTGGKSIYGEKFPDENFALKHNKPGLLSMANAGPNTNGSQFFITTVVTPWLDGKHVVFGEVTEGMDVVKKVESLGSNSGATRARIVIDKCGTV\n>A0A6G0XC32\nMEKLSSLSFAYCIKLTKVIIESLSLQELDVTACSDLAQLRLQCPVLSSLDCSWCRHIHVKNVIVGTSVPKLSQLALRGWDPSGGSLDTQLGLLLRSFPLVCSLNLSHVALSDRGLFSVFVLANSLETLVLSQPQSNVWVDGTWTLDLLTMWKERRPNVHVVLQ\n>P42212\nMSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK\n>P61626\nMKALIVLGLVLLSVTVQGKVFERCELARTLKRLGMDGYRGISLANWMCLAKWESGYNTRATNYNAGDRSTDYGIFQINSRYWCNDGKTPGAVNACHLSCSALLQDNIADAVACAKRVVRDPQGIRAWVAWRNRCQNRDVRQYVQGCGV\n>P68871\nMVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH\n>P37840\nMDVFMKGLSKAKEGVVAAAEKTKQGVAEAAGKTKEGVLYVGSKTKEGVVHGVATVAEKTKEQVTNVGGAVVTGVTAVAQKTVEGAGSIAAATGFVKKDQLGKNEEGAPQEGILEDMPVDPDNEAYEMPSEEGYQDYEPEA\n>P02686\nMGNHAGKRELNAEKASTNSETNRGESEKKRNLGELSRTTSEDNEVFGEADANQNNGTSSQDTAVTDSKRTADPKNAWQDAHPADPGSRPHLIRLFSRDAPGREDNTFKDRPSESDELQTIQEDSAATSESLDVMASQKRPSQRHGSKYLATASTMDHARHGFLPRHRDTGILDSIGRFFGGDRGAPKRGSGKDSHHPARTAHYGSLPQKSHGRTQDENPVVHFFKNIVTPRTPPPSQGKGRGLSLSRFSWGAEGQRPGFGYGGRASDYKSAHKGFKGVDAQGTLSKIFKLGGRDSRSGSPMARR\n>P61769\nMSRSVALAVLALLSLSGLEAIQRTPKIQVYSRHPAENGKSNFLNCYVSGFHPSDIEVDLLKNGERIEKVEHSDLSFSKDWSFYLLYYTEFTPTEKDEYACRVNHVTLSQPKIVKWDRDM\n>P0ABH7\nMADTKAKLTLNGDTAVELDVLKGTLGQDVIDIRTLGSKGVFTFDPGFTSTASCESKITFIDGDEGILLHRGFPIDQLATDSNYLEVCYILLNGEKPTQEQYDEFKTTVTRHTMIHEQITRLFHAFRRDSHPMAVMCGITGALAAFYHDSLDVNNPRHREIAAFRLLSKMPTMAAMCYKYSIGQPFVYPRNDLSYAGNFLNMMFSTPCEPYEVNPILERAMDRILILHADHEQNASTSTVRTAGSSGANPFACIAAGIASLWGPAHGGANEAALKMLEEISSVKHIPEFVRRAKDKNDSFRLMGFGHRVYKNYDPRATVMRETCHEVLKELGTKDDLLEVAMELENIALNDPYFIEKKLYPNVDFYSGIILKAMGIPSSMFTVIFAMARTVGWIAHWSEMHSDGMKIARPRQLYTGYEKRDFKSDIKR\n>P0A9Q7\nMAVTNVAELNALVERVKKAQREYASFTQEQVDKIFRAAALAAADARIPLAKMAVAESGMGIVEDKVIKNHFASEYIYNAYKDEKTCGVLSEDDTFGTITIAEPIGIICGIVPTTNPTSTAIFKSLISLKTRNAIIFSPHPRAKDATNKAADIVLQAAIAAGAPKDLIGWIDQPSVELSNALMHHPDINLILATGGPGMVKAAYSSGKPAIGVGAGNTPVVIDETADIKRAVASVLMSKTFDNGVICASEQSVVVVDSVYDAVRERFATHGGYLLQGKELKAVQDVILKNGALNAAIVGQPAYKIAELAGFSVPENTKILIGEVTVVDESEPFAHEKLSPTLAMYRAKDFEDAVEKAEKLVAMGGIGHTSCLYTDQDNQPARVSYFGQKMKTARILINTPASQGGIGDLYNFKLAPSLTLGCGSWGGNSISENVGPKHLINKKTVAKRAENMLWHKLPKSIYFRRGSLPIALDEVITDGHKRALIVTDRFLFNNGYADQITSVLKAAGVETEVFFEVEADPTLSIVRKGAELANSFKPDVIIALGGGSPMDAAKIMWVMYEHPETHFEELALRFMDIRKRIYKFPKMGVKAKMIAVTTTSGTGSEVTPFAVVTDDATGQKYPLADYALTPDMAIVDANLVMDMPKSLCAFGGLDAVTHAMEAYVSVLASEFSDGQALQALKLLKEYLPASYHEGSKNPVARERVHSAATIAGIAFANAFLGVCHSMAHKLGSQFHIPHGLANALLICNVIRYNANDNPTKQTAFSQYDRPQARRRYAEIADHLGLSAPGDRTAAKIEKLLAWLETLKAELGIPKSIREAGVQEADFLANVDKLSEDAFDDQCTGANPRYPLISELKQILLDTYYGRDYVEGETAAKKEAAPAKAEKKAKKSA\n>P0A953\nMKRAVITGLGIVSSIGNNQQEVLASLREGRSGITFSQELKDSGMRSHVWGNVKLDTTGLIDRKVVRFMSDASIYAFLSMEQAIADAGLSPEAYQNNPRVGLIAGSGGGSPRFQVFGADAMRGPRGLKAVGPYVVTKAMASGVSACLATPFKIHGVNYSISSACATSAHCIGNAVEQIQLGKQDIVFAGGGEELCWEMACEFDAMGALSTKYNDTPEKASRTYDAHRDGFVIAGGGGMVVVEELEHALARGAHIYAEIVGYGATSDGADMVAPSGEGAVRCMKMAMHGVDTPIDYLNSHGTSTPVGDVKELAAIREVFGDKSPAISATKAMTGHSLGAAGVQEAIYSLLMLEHGFIAPSINIEELDEQAAGLNIVTETTDRELTTVMSNSFGFGGTNATLVMRKLKD\n>P00698\nMRSLLILVLCFLPLAALGKVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL\n>P0DP27\nMADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTEAELQDMINEVDADGNGTIDFPEFLTMMARKMKDTDSEEEIREAFRVFDKDGNGYISAAELRHVMTNLGEKLTDEEVDEMIREADIDGDGQVNYEEFVQMMTAK\n>P62937\nMVNPTVFFDIAVDGEPLGRVSFELFADKVPKTAENFRALSTGEKGFGYKGSCFHRIIPGFMCQGGDFTRHNGTGGKSIYGEKFEDENFILKHTGPGILSMANAGPNTNGSQFFICTAKTEWLDGKHVVFGKVKEGMNIVEAMERFGSRNGKTSKKITIADCGQLE\n>P10599\nMVKQIESKTAFQEALDAAGDKLVVVDFSATWCGPCKMIKPFFHSLSEKYSNVIFLEVDVDDCQDVASECEVKCMPTFQFFKKGQKVGEFSGANKEKLEATINELV\n>P56201\nMKCPGVWGMLTVTMCVVFLGCPQAQELQGHVSVILLGATGDLAKKYLWQGLFQLFLDEAGKGHSFSFHGAALTAPKQGQELMAKALESLSCPRDMAPSRCAELQAQFLRLSRYRQLKTAEDYQALGRDIEAQVQQEGLREAGRMFYFSVPPFAYADIARNINSSCRPGPGAWLRVVLEKPFGHDHLSAQQLATELGSFFQEEEMYRVDHYLGKQAVAQILPFRDQNRRALDSLWNRHHVERVEIIMKETVDAEGRTSFYEEYGVIRDTLQNHLTEILTLVAMELPANVSCSEAVLRHKLQAFRALRRLQRGSAVVGQYQTYSEQVRRELRKPAGSPSLTPTFAGVLVHVDNLRWEGVPFILMSGKALDERVGYVRVLFKNQAFCAQSEKHWAPAQSRCLPRQIIFYIGHGELGHPAVLVSRNLFRPFLPAQSWREVEDRPGLQLFGRPLSDFYAFSPVKERDAYSILLSHIFHARKESFVPTEHLLASWVFWTPLLESLAREVPRLYPGGADSGRLLDFEFSGSHLSFSLGQPEQLVPGPGSTPRPSDFQVLGAKYRESPLISAWPDELISKLASDIEAAAVQAVRRVGTFHLALSGGSSPIALFQQLASGHYGFPWAHTHLWLVDERCVPLSDPESNFQGLQAHLLQHVRVPYYNIHPMPVNLHQRLCAEEDRGAQAYASEISALVTNCSFDLVLLGMGTDGHTASLFPQSPTGLDGEQLVVLTESPSRPHQRMSLSLPLINRAKKVAVLVMGRTKRDITLLVSRVGREPKKWPISGVLPTSGQLVWYMDYEAFLG\n>P16949\nMASSDIQVKELEKRASGQAFELILSPRSKESVPEFPLSPPKKKDLSLEEIQKKLEAAEERRKSHEAEVLKQLAEKREHEKEVLQKAIEENNNFSKMAEEKLTHKMEANKENREAQMAAKLERLREKDKHIEEVRKNKESKDPADETEAD\n>P68082\nMGLSDGEWQQVLNVWGKVEADIAGHGQEVLIRLFTGHPETLEKFDKFKHLKTEAEMKASEDLKKHGTVVLTALGGILKKKGHHEAELKPLAQSHATKHKIPIKYLEFISDAIIHVLHSKHPGDFGADAQGAMTKALELFRNDIAAKYKELGFQG\n>P63101\nMDKNELVQKAKLAEQAERYDDMAACMKSVTEQGAELSNEERNLLSVAYKNVVGARRSSWRVVSSIEQKTEGAEKKQQMAREYREKIETELRDICNDVLSLLEKFLIPNASQPESKVFYLKMKGDYYRYLAEVAAGDDKKGIVDQSQQAYQEAFEISKKEMQPTHPIRLGLALNFSVFYYEILNSPEKACSLAKTAFDEAIAELDTLSEESYKDSTLIMQLLRDNLTLWTSDTQGDEAEAGEGGEN\n>Q9R0Q7\nMQPASAKWYDRRDYVFIEFCVEDSKDVNVNFEKSKLTFSCLGGSDNFKHLNEIDLFHCIDPNDSKHKRTDRSILCCLRKGESGQSWPRLTKERAKLNWLSVDFNNWKDWEDDSDEDMSNFDRFSEMMDHMGGDEDVDLPEVDGADDDSQDSDDEKMPDLE\n>P04637\nMEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGPDEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYQGSYGFRLGFLHSGTAKSVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPPGTRVRAMAIYKQSQHMTEVVRRCPHHERCSDSDGLAPPQHLIRVEGNLRVEYLDDRNTFRHSVVVPYEPPEVGSDCTTIHYNYMCNSSCMGGMNRRPILTIITLEDSSGNLLGRNSFEVRVCACPGRDRRTEEENLRKKGEPHHELPPGSTKRALPNNTSSSPQPKKKPLDGEYFTLQIRGRERFEMFRELNEALELKDAQAGKEPGGSRAHSSHLKSKKGQSTSRHKKLMFKTEGPDSD\n>P0DTC9\nMSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA\n>P0AEX9\nMKIKTGARILALSALTTMMFSASALAKIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPDKAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSKVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELAKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDEALKDAQTRITK\n>P00918\nMSHHWGYGKHNGPEHWHKDFPIAKGERQSPVDIDTHTAKYDPSLKPLSVSYDQATSLRILNNGHAFNVEFDDSQDKAVLKGGPLDGTYRLIQFHFHWGSLDGQGSEHTVDKKKYAAELHLVHWNTKYGDFGKAVQQPDGLAVLGIFLKVGSAKPGLQKVVDVLDSIKTKGKSADFTNFDPRGLLPESLDYWTYPGSLTTPPLLECVTWIVLKEPISVSSEQVLKFRKLNFNGEGEPEELMVDNWRPAQPLKNRQIKASFK\n>P0A855\nMKQALRVAFGFLILWASVLHAEVRIVIDSGVDSGRPIGVVPFQWAGPGAAPEDIGGIVAADLRNSGKFNPLDRARLPQQPGSAQEVQPAAWSALGIDAVVVGQVTPNPDGSYNVAYQLVDTGGAPGTVLAQNSYKVNKQWLRYAGHTASDEVFEKLTGIKGAFRTRIAYVVQTNGGQFPYELRVSDYDGYNQFVVHRSPQPLMSPAWSPDGSKLAYVTFESGRSALVIQTLANGAVRQVASFPRHNGAPAFSPDGSKLAFALSKTGSLNLYVMDLASGQIRQVTDGRSNNTEPTWFPDSQNLAFTSDQAGRPQVYKVNINGGAPQRITWEGSQNQDADVSSDGKFMVMVSSNGGQQHIAKQDLATGGVQVLSSTFLDETPSLAPNGTMVIYSSSQGMGSVLNLVSTDGRFKARLPATDGQVKFPAWSPYL\n>P00784\nMAMIPSISKLLFVAICLFVYMGLSFGDFSIVGYSQNDLTSTERLIQLFESWMLKHNKIYKNIDEKIYRFEIFKDNLKYIDETNKKNNSYWLGLNVFADMSNDEFKEKYTGSIAGNYTTTELSYEEVLNDGDVNIPEYVDWRQKGAVTPVKNQGSCGSCWAFSAVVTIEGIIKIRTGNLNEYSEQELLDCDRRSYGCNGGYPWSALQLVAQYGIHYRNTYPYEGVQRYCRSREKGPYAAKTDGVRQVQPYNEGALLYSIANQPVSVVLEAAGKDFQLYRGGIFVGPCGNKVDHAVAAVGYGPNYILIKNSWGTGWGENGYIRIKRGTGNSYGVCGLYTSSFYPVKN\n>P69441\nMRIILLGAPGAGKGTQAQFIMEKYGIPQISTGDMLRAAVKSGSELGKQAKDIMDAGKLVTDELVIALVKERIAQEDCRNGFLLDGFPRTIPQADAMKEAGINVDYVLEFDVPDELIVDRIVGRRVHAPSGRVYHVKFNPPKVEGKDDVTGEELTTRKDDQEETVRKRLVEYHQMTAPLIGYYSKEAEAGNTKYAKVDGTKPVAEVRADLEKILG\n>P02572\nMCDEEVAALVVDNGSGMCKAGFAGDDAPRAVFPSIVGRPRHQGVMVGMGQKDSYVGDEAQSKRGILTLKYPIEHGIVTNWDDMEKIWHHTFYNELRVAPEEHPVLLTEAPLNPKANREKMTQIMFETFNTPAMYVAIQAVLSLYASGRTTGIVLDSGDGVSHTVPIYEGYALPHAILRLDLAGRDLTDYLMKILTERGYSFTTTAEREIVRDIKEKLCYVALDFEQEMATAASSSSLEKSYELPDGQVITIGNERFRCPESLFQPSFLGMEACGIHETTYNSIMKCDVDIRKDLYANTVLSGGTTMYPGIADRMQKEITALAPSTMKIKIVAPPERKYSVWIGGSILASLSTFQQMWISKQEYDESGPSIVHRKCF\n>P63244\nMTEQMTLRGTLKGHNGWVTQIATTPQFPDMILSASRDKTIIMWKLTRDETNYGIPQRALRGHSHFVSDVVISSDGQFALSGSWDGTLRLWDLTTGTTTRRFVGHTKDVLSVAFSSDNRQIVSGSRDKTIKLWNTLGVCKYTVQDESHSEWVSCVRFSPNSSNPIIVSCGWDKLVKVWNLANCKLKTNHIGHTGYLNTVTVSPDGSLCASGGKDGQAMLWDLNEGKHLYTLDGGDIINALCFSPNRYWLCAATGPSIKIWDLEGKIIVDELKQEVISTSSKAEPPQCTSLAWSADGQTLFAGYTDNLVRVWQVTIGTR\n>P00439\nMSTAVLENPGLGRKLSDFGQETSYIEDNCNQNGAISLIFSLKEEVGALAKVLRLFEENDVNLTHIESRPSRLKKDEYEFFTHLDKRSLPALTNIIKILRHDIGATVHELSRDKKKDTVPWFPRTIQELDRFANQILSYGAELDADHPGFKDPVYRARRKQFADIAYNYRHGQPIPRVEYMEEEKKTWGTVFKTLKSLYKTHACYEYNHIFPLLEKYCGFHEDNIPQLEDVSQFLQTCTGFRLRPVAGLLSSRDFLGGLAFRVFHCTQYIRHGSKPMYTPEPDICHELLGHVPLFSDRSFAQFSQEIGLASLGAPDEYIEKLATIYWFTVEFGLCKQGDSIKAYGAGLLSSFGELQYCLSEKPKLLPLELEKTAIQNYTVTEFQPLYYVAESFNDAKEKVRNFAATIPRPFSVRYDPYTQRIEVLDNTQQLKILADSINSEIGILCSALQKIK\n>P62593\nMSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIELDLNSGKILESFRPEERFPMMSTFKVLLCGAVLSRVDAGQEQLGRRIHYSQNDLVEYSPVTEKHLTDGMTVRELCSAAITMSDNTAANLLLTTIGGPKELTAFLHNMGDHVTRLDRWEPELNEAIPNDERDTTMPAAMATTLRKLLTGELLTLASRQQLIDWMEADKVAGPLLRSALPAGWFIADKSGAGERGSRGIIAALGPDGKPSRIVVIYTTGSQATMDERNRQIAEIGASLIKHW\n>P00766\nCGVPAIQPVLSGLSRIVNGEEAVPGSWPWQVSLQDKTGFHFCGGSLINENWVVTAAHCGVTTSDVVVAGEFDQGSSSEKIQKLKIAKVFKNSKYNSLTINNDITLLKLSTAASFSQTVSAVCLPSASDDFAAGTTCVTTGWGLTRYTNANTPDRLQQASLPLLSNTNCKKYWGTKIKDAMICAGASGVSSCMGDSGGPLVCKKNGAWTLVGIVSWGSSTCSTSTPGVYARVTALVNWVQQTLAAN\n>P0A6P9\nMSKIVKIIGREIIDSRGNPTVEAEVHLEGGFVGMAAAPSGASTGSREALELRDGDKSRFLGKGVTKAVAAVNGPIAQALIGKDAKDQAGIDKIMIDLDGTENKSKFGANAILAVSLANAKAAAAAKGMPLYEHIAELNGTPGKYSMPVPMMNIINGGEHADNNVDIQEFMIQPVGAKTVKEAIRMGSEVFHHLAKVLKAKGMNTAVGDEGGYAPNLGSNAEALAVIAEAVKAAGYELGKDITLAMDCAASEFYKDGKYVLAGEGNKAFTSEEFTHFLEELTKQYPIVSIEDGLDESDWDGFAYQTKVLGDKIQLVGDDLFVTNTKILKEGIEKGIANSILIKFNQIGSLTETLAAIKMAKDAGYTAVISHRSGETEDATIADLAVGTAAGQIKTGSMSRSDRVAKYNQLIRIEEALGEKAPYNGRKEIKGQA\n>P04406\nMGKVKVGVNGFGRIGRLVTRAAFNSGKVDIVAINDPFIDLNYMVYMFQYDSTHGKFHGTVKAENGKLVINGNPITIFQERDPSKIKWGDAGAEYVVESTGVFTTMEKAGAHLQGGAKRVIISAPSADAPMFVMGVNHEKYDNSLKIISNASCTTNCLAPLAKVIHDNFGIVEGLMTTVHAITATQKTVDGPSGKLWRDGRGALQNIIPASTGAAKAVGKVIPELNGKLTGMAFRVPTANVSVVDLTCRLEKPAKYDDIKKVVKQASEGPLKGILGYTEHQVVSSDFNSDTHSSTFDAGAGIALNDHFVKLISWYDNEFGYSNRVVDLMAHMASKE\n>P24941\nMENFQKVEKIGEGTYGVVYKARNKLTGEVVALKKIRLDTETEGVPSTAIREISLLKELNHPNIVKLLDVIHTENKLYLVFEFLHQDLKKFMDASALTGIPLPLIKSYLFQLLQGLAFCHSHRVLHRDLKPQNLLINTEGAIKLADFGLARAFGVPVRTYTHEVVTLWYRAPEILLGCKYYSTAVDIWSLGCIFAEMVTRRALFPGDSEIDQLFRIFRTLGTPDEVVWPGVTSMPDYKPSFPKWARQDFSKVVPPLDEDGRSLLSQMLHYDPNKRISAKAALAHPFFQDVTKPVPHLRL\n>P02994\nMGKEKSHINVVVIGHVDSGKSTTTGHLIYKCGGIDKRTIEKFEKEAAELGKGSFKYAWVLDKLKAERERGITIDIALWKFETPKYQVTVIDAPGHRDFIKNMITGTSQADCAILIIAGGVGEFEAGISKDGQTREHALLAFTLGVRQLIVAVNKMDSVKWDESRFQEIVKETSNFIKKVGYNPKTVPFVPISGWNGDNMIEATTNAPWYKGWEKETKAGVVKGKTLLEAIDAIEQPSRPTDKPLRLPLQDVYKIGGIGTVPVGRVETGVIKPGMVVTFAPAGVTTEVKSVEMHHEQLEQGVPGDNVGFNVKNVSVKEIRRGNVCGDAKNDPPKGCASFNATVIVLNHPGQISAGYSPVLDCHTAHIACRFDELLEKNDRRSGKKLEDHPKFLKSGDAALVKFVPSKPMCVEAFSEYPPLGRFAVRDMRQTVAVGVIKSVDKTEKAAKVTKAAQKAAKK\n>P61583\nMNPSEMQRKGPPQRWCLQVYPTAPKRQRPSRTGHDDDGGFVEKKRGKCGEKQERSDCYCVCVERSRHRRLHFVLY\n>P05089\nMSAKSRTIGIIGAPFSKGQPRGGVEEGPTVLRKAGLLEKLKEQECDVKDYGDLPFADIPNDSPFQIVKNPRSVGKASEQLAGKVAEVKKNGRISLVLGGDHSLAIGSISGHARVHPDLGVIWVDAHTDINTPLTTTSGNLHGQPVSFLLKELKGKIPDVPGFSWVTPCISAKDIVYIGLRDVDPGEHYILKTLGIKYFSMTEVDRLGIGKVMEETLSYLLGRKKRPIHLSFDVDGLDPSFTPATGTPVVGGLTYREGLYITEEIYKTGLLSGLDIMEVNPSLGKTPEEVTRTVNTAVAITLACFGLAREGNHKPIDYLNPPK\n>P07688\nMWRLLATLSCLLVLTSARSSLYFPPLSDELVNFVNKQNTTWKAGHNFYNVDLSYVKKLCGAILGGPKLPQRDAFAADVVLPESFDAREQWPNCPTIKEIRDQGSCGSCWAFGAVEAISDRICIHSNGRVNVEVSAEDMLTCCGGECGDGCNGGFPSGAWNFWTKKGLVSGGLYNSHVGCRPYSIPPCEHHVNGSRPPCTGEGDTPKCSKTCEPGYSPSYKEDKHFGCSSYSVANNEKEIMAEIYKNGPVEGAFSVYSDFLLYKSGVYQHVSGEIMGGHAIRILGWGVENGTPYWLVGNSWNTDWGDNGFFKILRGQDHCGIESEIVAGMPCTHQY\n>P06280\nMQLRNPELHLGCALALRFLALVSWDIPGARALDNGLARTPTMGWLHWERFMCNLDCQEEPDSCISEKLFMEMAELMVSEGWKDAGYEYLCIDDCWMAPQRDSEGRLQADPQRFPHGIRQLANYVHSKGLKLGIYADVGNKTCAGFPGSFGYYDIDAQTFADWGVDLLKFDGCYCDSLENLADGYKHMSLALNRTGRSIVYSCEWPLYMWPFQKPNYTEIRQYCNHWRNFADIDDSWKSIKSILDWTSFNQERIVDVAGPGGWNDPDMLVIGNFGLSWNQQVTQMALWAIMAAPLFMSNDLRHISPQAKALLQDKDVIAINQDPLGKQGYQLRQGDNFEVWERPLSGLAWAVAMINRQEIGGPRSYTIAVASLGKGVACNPACFITQLLPVKRKLGFYEWTSRLRSHINPTGTVLLQLENTMQMSLKDLL\n>P19367\nMIAAQLLAYYFTELKDDQVKKIDKYLYAMRLSDETLIDIMTRFRKEMKNGLSRDFNPTATVKMLPTFVRSIPDGSEKGDFIALDLGGSSFRILRVQVNHEKNQNVHMESEVYDTPENIVHGSGSQLFDHVAECLGDFMEKRKIKDKKLPVGFTFSFPCQQSKIDEAILITWTKRFKASGVEGADVVKLLNKAIKKRGDYDANIVAVVNDTVGTMMTCGYDDQHCEVGLIIGTGTNACYMEELRHIDLVEGDEGRMCINTEWGAFGDDGSLEDIRTEFDREIDRGSLNPGKQLFEKMVSGMYLGELVRLILVKMAKEGLLFEGRITPELLTRGKFNTSDVSAIEKNKEGLHNAKEILTRLGVEPSDDDCVSVQHVCTIVSFRSANLVAATLGAILNRLRDNKGTPRLRTTVGVDGSLYKTHPQYSRRFHKTLRRLVPDSDVRFLLSESGSGKGAAMVTAVAYRLAEQHRQIEETLAHFHLTKDMLLEVKKRMRAEMELGLRKQTHNNAVVKMLPSFVRRTPDGTENGDFLALDLGGTNFRVLLVKIRSGKKRTVEMHNKIYAIPIEIMQGTGEELFDHIVSCISDFLDYMGIKGPRMPLGFTFSFPCQQTSLDAGILITWTKGFKATDCVGHDVVTLLRDAIKRREEFDLDVVAVVNDTVGTMMTCAYEEPTCEVGLIVGTGSNACYMEEMKNVEMVEGDQGQMCINMEWGAFGDNGCLDDIRTHYDRLVDEYSLNAGKQRYEKMISGMYLGEIVRNILIDFTKKGFLFRGQISETLKTRGIFETKFLSQIESDRLALLQVRAILQQLGLNSTCDDSILVKTVCGVVSRRAAQLCGAGMAAVVDKIRENRGLDRLNVTVGVDGTLYKLHPHFSRIMHQTVKELSPKCNVSFLLSEDGSGKGAALITAVGVRLRTEASS\n>P0A6F5\nMAAKDVKFGNDARVKMLRGVNVLADAVKVTLGPKGRNVVLDKSFGAPTITKDGVSVAREIELEDKFENMGAQMVKEVASKANDAAGDGTTTATVLAQAIITEGLKAVAAGMNPMDLKRGIDKAVTAAVEELKALSVPCSDSKAIAQVGTISANSDETVGKLIAEAMDKVGKEGVITVEDGTGLQDELDVVEGMQFDRGYLSPYFINKPETGAVELESPFILLADKKISNIREMLPVLEAVAKAGKPLLIIAEDVEGEALATLVVNTMRGIVKVAAVKAPGFGDRRKAMLQDIATLTGGTVISEEIGMELEKATLEDLGQAKRVVINKDTTTIIDGVGEEAAIQGRVAQIRQQIEEATSDYDREKLQERVAKLAGGVAVIKVGAATEVEMKEKKARVEDALHATRAAVEEGVVAGGGVALIRVASKLADLRGQNEDQNVGIKVALRAMEAPLRQIVLNCGEEPSVVANTVKGGDGNYGYNAATEEYGNMIDMGILDPTKVTRSALQYAASVAGLMITTECMVTDLPKNDAADLGAAGGMGGMGGMGGMM\n>P02768\nMKWVTFISLLFLFSSAYSRGVFRRDAHKSEVAHRFKDLGEENFKALVLIAFAQYLQQCPFEDHVKLVNEVTEFAKTCVADESAENCDKSLHTLFGDKLCTVATLRETYGEMADCCAKQEPERNECFLQHKDDNPNLPRLVRPEVDVMCTAFHDNEETFLKKYLYEIARRHPYFYAPELLFFAKRYKAAFTECCQAADKAACLLPKLDELRDEGKASSAKQRLKCASLQKFGERAFKAWAVARLSQRFPKAEFAEVSKLVTDLTKVHTECCHGDLLECADDRADLAKYICENQDSISSKLKECCEKPLLEKSHCIAEVENDEMPADLPSLAADFVESKDVCKNYAEAKDVFLGMFLYEYARRHPDYSVVLLLRLAKTYETTLEKCCAAADPHECYAKVFDEFKPLVEEPQNLIKQNCELFEQLGEYKFQNALLVRYTKKVPQVSTPTLVEVSRNLGKVGSKCCKHPEAKRMPCAEDYLSVVLNQLCVLHEKTPVSDRVTKCCTESLVNRRPCFSALEVDETYVPKEFNAETFTFHADICTLSEKERQIKKQTALVELVKHKPKATKEQLKAVMDDFAAFVEKCCKADDKETCFAEEGKKLVAASQAALGL\n>P08684\nMALIPDLAMETWLLLAVSLVLLYLYGTHSHGLFKKLGIPGPTPLPFLGNILSYHKGFCMFDMECHKKYGKVWGFYDGQQPVLAITDPDMIKTVLVKECYSVFTNRRPFGPVGFMKSAISIAEDEEWKRLRSLLSPTFTSGKLKEMVPIIAQYGDVLVRNLRREAETGKPVTLKDVFGAYSMDVITSTSFGVNIDSLNNPQDPFVENTKKLLRFDFLDPFFLSITVFPFLIPILEVLNICVFPREVTNFLRKSVKRMKESRLEDTQKHRVDFLQLMIDSQNSKETESHKALSDLELVAQSIIFIFAGYETTSSVLSFIMYELATHPDVQQKLQEEIDAVLPNKAPPTYDTVLQMEYLDMVVNETLRLFPIAMRLERVCKKDVEINGMFIPKGVVVMIPSYALHRDPKYWTEPEKFLPERFSKKNKDNIDPYIYTPFGSGPRNCIGMRFALMNMKLALIRVLQNFSFKPCKETQIPLKLSLGGLLQPEKPVVLKVESRDGTVSGA\n>P35354\nMLARALLLCAVLALSHTANPCCSHPCQNRGVCMSVGFDQYKCDCTRTGFYGENCSTPEFLTRIKLFLKPTPNTVHYILTHFKGFWNVVNNIPFLRNAIMSYVLTSRSHLIDSPPTYNADYGYKSWEAFSNLSYYTRALPPVPDDCPTPLGVKGKKQLPDSNEIVEKLLLRRKFIPDPQGSNMMFAFFAQHFTHQFFKTDHKRGPAFTNGLGHGVDLNHIYGETLARQRKLRLFKDGKMKYQIIDGEMYPPTVKDTQAEMIYPPQVPEHLRFAVGQEVFGLVPGLMMYATIWLREHNRVCDVLKQEHPEWGDEQLFQTSRLILIGETIKIVIEDYVQHLSGYHFKLKFDPELLFNKQFQYQNRIAAEFNTLYHWHPLLPDTFQIHDQKYNYQQFIYNNSILLEHGITQFVESFTRQIAGRVAGGRNVPPAVQKVSQASIDQSRQMKYQSFNEYRKRFMLKPYESFEELTGEKEMSAELEALYGDIDAVELYPALLVEKPRPDAIFGETMVEVGAPFSLKGLMGNVICSPAYWKPSTFGGEVGFQIINTASIQSLICNNVKGCPFTSFSVPDPELIKTVTINASSSRSGLDDINPTVLLKERSTEL\n>P04150\nMDSKESLTPGREENPSSVLAQERGDVMDFYKTLRGGATVKVSASSPSLAVASQSDSKQRRLLVDFPKGSVSNAQQPDLSKAVSLSMGLYMGETETKVMGNDLGFPQQGQISLSSGETDLKLLEESIANLNRSTSVPENPKSSASTAVSAAPTEKEFPKTHSDVSSEQQHLKGQTGTNGGNVKLYTTDQSTFDILQDLEFSSGSPGKETNESPWRSDLLIDENCLLSPLAGEDDSFLLEGNSNEDCKPLILPDTKPKIKDNGDLVLSSPSNVTLPQVKTEKEDFIELCTPGVIKQEKLGTVYCQASFPGANIIGNKMSAISVHGVSTSGGQMYHYDMNTASLSQQQDQKPIFNVIPPIPVGSENWNRCQGSGDDNLTSLGTLNFPGRTVFSNGYSSPSMRPDVSSPPSSSSTATTGPPPKLCLVCSDEASGCHYGVLTCGSCKVFFKRAVEGQHNYLCAGRNDCIIDKIRRKNCPACRYRKCLQAGMNLEARKTKKKIKGIQQATTGVSQETSENPGNKTIVPATLPQLTPTLVSLLEVIEPEVLYAGYDSSVPDSTWRIMTTLNMLGGRQVIAAVKWAKAIPGFRNLHLDDQMTLLQYSWMFLMAFALGWRSYRQSSANLLCFAPDLIINEQRMTLPCMYDQCKHMLYVSSELHRLQVSYEEYLCMKTLLLLSSVPKDGLKSQELFDEIRMTYIKELGKAIVKREGNSSQNWQRFYQLTKLLDSMHEVVENLLNYCFQTFLDKTMSIEFPEMLAEIITNQIPKYSNGNIKKLLFHQK\n>P0AG67\nMTESFAQLFEESLKEIETRPGSIVRGVVVAIDKDVVLVDAGLKSESAIPAEQFKNAQGELEIQVGDEVDVALDAVEDGFGETLLSREKAKRHEAWITLEKAYEDAETVTGVINGKVKGGFTVELNGIRAFLPGSLVDVRPVRDTLHLEGKELEFKVIKLDQKRNNVVVSRRAVIESENSAERDQLLENLQEGMEVKGIVKNLTDYGAFVDLGGVDGLLHITDMAWKRVKHPSEIVNVGDEITVKVLKFDRERTRVSLGLKQLGEDPWVAIAKRYPEGTKLTGRVTNLTDYGCFVEIEEGVEGLVHVSEMDWTNKNIHPSKVVNVGDVVEVMVLDIDEERRRISLGLKQCKANPWQQFAETHNKGDRVEGKIKSITDFGIFIGLDGGIDGLVHLSDISWNVAGEEAVREYKKGDEIAAVVLQVDAERERISLGVKQLAEDPFNNWVALNKKGAIVTGKVTAVDAKGATVELADGVEGYLRASEASRDRVEDATLVLSVGDEVEAKFTGVDRKNRAISLSVRAKDEADEKDAIATVNKQEDANFSNNAMAEAFKAAKGE\n>P15056\nMAALSGGGGGGAEPGQALFNGDMEPEAGAGAGAAASSAADPAIPEEVWNIKQMIKLTQEHIEALLDKFGGEHNPPSIYLEAYEEYTSKLDALQQREQQLLESLGNGTDFSVSSSASMDTVTSSSSSSLSVLPSSLSVFQNPTDVARSNPKSPQKPIVRVFLPNKQRTVVPARCGVTVRDSLKKALMMRGLIPECCAVYRIQDGEKKPIGWDTDISWLTGEELHVEVLENVPLTTHNFVRKTFFTLAFCDFCRKLLFQGFRCQTCGYKFHQRCSTEVPLMCVNYDQLDLLFVSKFFEHHPIPQEEASLAETALTSGSSPSAPASDSIGPQILTSPSPSKSIPIPQPFRPADEDHRNQFGQRDRSSSAPNVHINTIEPVNIDDLIRDQGFRGDGGSTTGLSATPPASLPGSLTNVKALQKSPGPQRERKSSSSSEDRNRMKTLGRRDSSDDWEIPDGQITVGQRIGSGSFGTVYKGKWHGDVAVKMLNVTAPTPQQLQAFKNEVGVLRKTRHVNILLFMGYSTKPQLAIVTQWCEGSSLYHHLHIIETKFEMIKLIDIARQTAQGMDYLHAKSIIHRDLKSNNIFLHEDLTVKIGDFGLATVKSRWSGSHQFEQLSGSILWMAPEVIRMQDKNPYSFQSDVYAFGIVLYELMTGQLPYSNINNRDQIIFMVGRGYLSPDLSKVRSNCPKAMKRLMAECLKKKRDERPLFPQILASIELLARSLPKIHRSASEPSLNRAGFQTEDFSLYACASPKTPIQAGGYGAFPVH\n>P05067\nMLPGLALLLLAAWTARALEVPTDGNAGLLAEPQIAMFCGRLNMHMNVQNGKWDSDPSGTKTCIDTKEGILQYCQEVYPELQITNVVEANQPVTIQNWCKRGRKQCKTHPHFVIPYRCLVGEFVSDALLVPDKCKFLHQERMDVCETHLHWHTVAKETCSEKSTNLHDYGMLLPCGIDKFRGVEFVCCPLAEESDNVDSADAEEDDSDVWWGGADTDYADGSEDKVVEVAEEEEVAEVEEEEADDDEDDEDGDEVEEEAEEPYEEATERTTSIATTTTTTTESVEEVVREVCSEQAETGPCRAMISRWYFDVTEGKCAPFFYGGCGGNRNNFDTEEYCMAVCGSAMSQSLLKTTQEPLARDPVKLPTTAASTPDAVDKYLETPGDENEHAHFQKAKERLEAKHRERMSQVMREWEEAERQAKNLPKADKKAVIQHFQEKVESLEQEAANERQQLVETHMARVEAMLNDRRRLALENYITALQAVPPRPRHVFNMLKKYVRAEQKDRQHTLKHFEHVRMVDPKKAAQIRSQVMTHLRVIYERMNQSLSLLYNVPAVAEEIQDEVDELLQKEQNYSDDVLANMISEPRISYGNDALMPSLTETKTTVELLPVNGEFSLDDLQPWHSFGADSVPANTENEVEPVDARPAADRGLTTRPGSGLTNIKTEEISEVKMDAEFRHDSGYEVHHQKLVFFAEDVGSNKGAIIGLMVGGVVIATVIVITLVMLKKKQYTSIHHGVVEVDAAVTPEERHLSKMQQNGYENPTYKFFEQMQN\n>P0ABB0\nMQLNSTEISELIKQRIAQFNVVSEAHNEGTIVSVSDGVIRIHGLADCMQGEMISLPGNRYAIALNLERDSVGAVVMGPYADLAEGMKVKCTGRILEVPVGRGLLGRVVNTLGAPIDGKGPLDHDGFSAVEAIAPGVIERQSVDQPVQTGYKAVDSMIPIGRGQRELIIGDRQTGKTALAIDAIINQRDSGIKCIYVAIGQKASTISNVVRKLEEHGALANTIVVVATASESAALQYLAPYAGCAMGEYFRDRGEDALIIYDDLSKQAVAYRQISLLLRRPPGREAFPGDVFYLHSRLLERAARVNAEYVEAFTKGEVKGKTGSLTALPIIETQAGDVSAFVPTNVISITDGQIFLETNLFNAGIRPAVNPGISVSRVGGAAQTKIMKKLSGGIRTALAQYRELAAFSQFASDLDDATRKQLDHGQKVTELLKQKQYAPMSVAQQSLVLFAAERGYLADVELSKIGSFEAALLAYVDRDHAPLMQEINQTGGYNDEIEGKLKGILDSFKATQSW\n>P23246\nMSRDRFRSRGGGGGGFHRRGGGGGRGGLHDFRSPPPGMGLNQNRGPMGPGPGQSGPKPPIPPPPPHQQQQQPPPQQPPPQQPPPHQPPPHPQPHQQQQPPPPPQDSSKPVVAQGPGPAPGVGSAPPASSSAPPATPPTSGAPPGSGPGPTPTPPPAVTSAPPGAPPPTPPSSGVPTTPPQAGGPPPPPAAVPGPGPGPKQGPGPGGPKGGKMPGGPKPGGGPGLSTPGGHPKPPHRGGGEPRGGRQHHPPYHQQHHQGPPPGGPGGRSEEKISDSEGFKANLSLLRRPGEKTYTQRCRLFVGNLPADITEDEFKRLFAKYGEPGEVFINKGKGFGFIKLESRALAEIAKAELDDTPMRGRQLRVRFATHAAALSVRNLSPYVSNELLEEAFSQFGPIERAVVIVDDRGRSTGKGIVEFASKPAARKAFERCSEGVFLLTTTPRPVIVEPLEQLDDEDGLPEKLAQKNPMYQKERETPPRFAQHGTFEYEYSQRWKSLDEMEKQQREQVEKNMKDAKDKLESEMEDAYHEHQANLLRQDLMRRQEELRRMEELHNQEMQKRKEMQLRQEEERRRREEEMMIRQREMEEQMRRQREESYSRMGYMDPRERDMRMGGGGAMNMGDPYGSGGQKFPPLGGGGGIGYEANPGVPPATMSGSMMGSDMRTERFGQGGAGPVGGQGPRGMGPGTPAGYGRGREEYEGPNKKPRF\n>P21802\nMVSWGRFICLVVVTMATLSLARPSFSLVEDTTLEPEEPPTKYQISQPEVYVAAPGESLEVRCLLKDAAVISWTKDGVHLGPNNRTVLIGEYLQIKGATPRDSGLYACTASRTVDSETWYFMVNVTDAISSGDDEDDTDGAEDFVSENSNNKRAPYWTNTEKMEKRLHAVPAANTVKFRCPAGGNPMPTMRWLKNGKEFKQEHRIGGYKVRNQHWSLIMESVVPSDKGNYTCVVENEYGSINHTYHLDVVERSPHRPILQAGLPANASTVVGGDVEFVCKVYSDAQPHIQWIKHVEKNGSKYGPDGLPYLKVLKAAGVNTTDKEIEVLYIRNVTFEDAGEYTCLAGNSIGISFHSAWLTVLPAPGREKEITASPDYLEIAIYCIGVFLIACMVVTVILCRMKNTTKKPDFSSQPAVHKLTKRIPLRRQVTVSAESSSSMNSNTPLVRITTRLSSTADTPMLAGVSEYELPEDPKWEFPRDKLTLGKPLGEGCFGQVVMAEAVGIDKDKPKEAVTVAVKMLKDDATEKDLSDLVSEMEMMKMIGKHKNIINLLGACTQDGPLYVIVEYASKGNLREYLRARRPPGMEYSYDINRVPEEQMTFKDLVSCTYQLARGMEYLASQKCIHRDLAARNVLVTENNVMKIADFGLARDINNIDYYKKTTNGRLPVKWMAPEALFDRVYTHQSDVWSFGVLMWEIFTLGGSPYPGIPVEELFKLLKEGHRMDKPANCTNELYMMMRDCWHAVPSQRPTFKQLVEDLDRILTLTTNEEYLDLSQPLEQYSPSYPDTRSSCSSGDDSVFSPDPMPYEPCLPQYPHINGSVKT\n>P11413\nMAEQVALSRTQVCGILREELFQGDAFHQSDTHIFIIMGASGDLAKKKIYPTIWWLFRDGLLPENTFIVGYARSRLTVADIRKQSEPFFKATPEEKLKLEDFFARNSYVAGQYDDAASYQRLNSHMNALHLGSQANRLFYLALPPTVYEAVTKNIHESCMSQIGWNRIIVEKPFGRDLQSSDRLSNHISSLFREDQIYRIDHYLGKEMVQNLMVLRFANRIFGPIWNRDNIACVILTFKEPFGTEGRGGYFDEFGIIRDVMQNHLLQMLCLVAMEKPASTNSDDVRDEKVKVLKCISEVQANNVVLGQYVGNPDGEGEATKGYLDDPTVPRGSTTATFAAVVLYVENERWDGVPFILRCGKALNERKAEVRLQFHDVAGDIFHQQCKRNELVIRVQPNEAVYTKMMTKKPGMFFNPEESELDLTYGNRYKNVKLPDAYERLILDVFCGSQMHFVRSDELREAWRIFTPLLHQIELEKPKPIPYIYGSRGPTEADELMKRVGFQYEGTYKWVNPHKL\n>P07901\nMPEETQTQDQPMEEEEVETFAFQAEIAQLMSLIINTFYSNKEIFLRELISNSSDALDKIRYESLTDPSKLDSGKELHINLIPSKQDRTLTIVDTGIGMTKADLINNLGTIAKSGTKAFMEALQAGADISMIGQFGVGFYSAYLVAEKVTVITKHNDDEQYAWESSAGGSFTVRTDTGEPMGRGTKVILHLKEDQTEYLEERRIKEIVKKHSQFIGYPITLFVEKERDKEVSDDEAEEKEEKEEEKEKEEKESDDKPEIEDVGSDEEEEEKKDGDKKKKKKIKEKYIDQEELNKTKPIWTRNPDDITNEEYGEFYKSLTNDWEEHLAVKHFSVEGQLEFRALLFVPRRAPFDLFENRKKKNNIKLYVRRVFIMDNCEELIPEYLNFIRGVVDSEDLPLNISREMLQQSKILKVIRKNLVKKCLELFTELAEDKENYKKFYEQFSKNIKLGIHEDSQNRKKLSELLRYYTSASGDEMVSLKDYCTRMKENQKHIYFITGETKDQVANSAFVERLRKHGLEVIYMIEPIDEYCVQQLKEFEGKTLVSVTKEGLELPEDEEEKKKQEEKKTKFENLCKIMKDILEKKVEKVVVSNRLVTSPCCIVTSTYGWTANMERIMKAQALRDNSTMGYMAAKKHLEINPDHSIIETLRQKAEADKNDKSVKDLVILLYETALLSSGFSLEDPQTHANRIYRMIKLGLGIDEDDPTVDDTSAAVTEEMPPLEGDDDTSRMEEVD\n>Q02880\nMAKSGGCGAGAGVGGGNGALTWVTLFDQNNAAKKEESETANKNDSSKKLSVERVYQKKTQLEHILLRPDTYIGSVEPLTQFMWVYDEDVGMNCREVTFVPGLYKIFDEILVNAADNKQRDKNMTCIKVSIDPESNIISIWNNGKGIPVVEHKVEKVYVPALIFGQLLTSSNYDDDEKKVTGGRNGYGAKLCNIFSTKFTVETACKEYKHSFKQTWMNNMMKTSEAKIKHFDGEDYTCITFQPDLSKFKMEKLDKDIVALMTRRAYDLAGSCRGVKVMFNGKKLPVNGFRSYVDLYVKDKLDETGVALKVIHELANERWDVCLTLSEKGFQQISFVNSIATTKGGRHVDYVVDQVVGKLIEVVKKKNKAGVSVKPFQVKNHIWVFINCLIENPTFDSQTKENMTLQPKSFGSKCQLSEKFFKAASNCGIVESILNWVKFKAQTQLNKKCSSVKYSKIKGIPKLDDANDAGGKHSLECTLILTEGDSAKSLAVSGLGVIGRDRYGVFPLRGKILNVREASHKQIMENAEINNIIKIVGLQYKKSYDDAESLKTLRYGKIMIMTDQDQDGSHIKGLLINFIHHNWPSLLKHGFLEEFITPIVKASKNKQELSFYSIPEFDEWKKHIENQKAWKIKYYKGLGTSTAKEAKEYFADMERHRILFRYAGPEDDAAITLAFSKKKIDDRKEWLTNFMEDRRQRRLHGLPEQFLYGTATKHLTYNDFINKELILFSNSDNERSIPSLVDGFKPGQRKVLFTCFKRNDKREVKVAQLAGSVAEMSAYHHGEQALMMTIVNLAQNFVGSNNINLLQPIGQFGTRLHGGKDAASPRYIFTMLSTLARLLFPAVDDNLLKFLYDDNQRVEPEWYIPIIPMVLINGAEGIGTGWACKLPNYDAREIVNNVRRMLDGLDPHPMLPNYKNFKGTIQELGQNQYAVSGEIFVVDRNTVEITELPVRTWTQVYKEQVLEPMLNGTDKTPALISDYKEYHTDTTVKFVVKMTEEKLAQAEAAGLHKVFKLQTTLTCNSMVLFDHMGCLKKYETVQDILKEFFDLRLSYYGLRKEWLVGMLGAESTKLNNQARFILEKIQGKITIENRSKKDLIQMLVQRGYESDPVKAWKEAQEKAAEEDETQNQHDDSSSDSGTPSGPDFNYILNMSLWSLTKEKVEELIKQRDAKGREVNDLKRKSPSDLWKEDLAAFVEELDKVESQEREDVLAGMSGKAIKGKVGKPKVKKLQLEETMPSPYGRRIIPEITAMKADASKKLLKKKKGDLDTAAVKVEFDEEFSGAPVEGAGEEALTPSVPINKGPKPKREKKEPGTRVRKTPTSSGKPSAKKVKKRNPWSDDESKSESDLEETEPVVIPRDSLLRRAAAERPKYTFDFSEEEDDDADDDDDDNNDLEELKVKASPITNDGEDEFVPSDGLDKDEYTFSPGKSKATPEKSLHDKKSQDFGNLFSFPSYSQKSEDDSAKFDSNEEDSASVFSPSFGLKQTDKVPSKTVAAKKGKPSSDTVPKPKRAPKQKKVVEAVNSDSDSEFGIPKKTTTPKGKGRGAKKRKASGSENEGDYNPGRKTSKTTSKKPKKTSFDQDSDVDIFPSDFPTEPPSLPRTGRARKEVKYFAESDEEEDDVDFAMFN\n>P14598\nMGDTFIRHIALLGFEKRFVPSQHYVYMFLVKWQDLSEKVVYRRFTEIYEFHKTLKEMFPIEAGAINPENRIIPHLPAPKWFDGQRAAENRQGTLTEYCSTLMSLPTKISRCPHLLDFFKVRPDDLKLPTDNQTKKPETYLMPKDGKSTATDITGPIILQTYRAIANYEKTSGSEMALSTGDVVEVVEKSESGWWFCQMKAKRGWIPASFLEPLDSPDETEDPEPNYAGEPYVAIKAYTAVEGDEVSLLEGEAVEVIHKLLDGWWVIRKDDVTGYFPSMYLQKSGQDVSQAQRQIKRGAPPRRSSIRNAHSIHQRSRKRLSQDAYRRNSVRFLQQRRRQARPGPQSPGSPLEEERQTQRSKPQPAVPPRPSADLILNRCSESTKRKLASAV\n>P10591\nMSKAVGIDLGTTYSCVAHFANDRVDIIANDQGNRTTPSFVAFTDTERLIGDAAKNQAAMNPSNTVFDAKRLIGRNFNDPEVQADMKHFPFKLIDVDGKPQIQVEFKGETKNFTPEQISSMVLGKMKETAESYLGAKVNDAVVTVPAYFNDSQRQATKDAGTIAGLNVLRIINEPTAAAIAYGLDKKGKEEHVLIFDLGGGTFDVSLLSIEDGIFEVKATAGDTHLGGEDFDNRLVNHFIQEFKRKNKKDLSTNQRALRRLRTACERAKRTLSSSAQTSVEIDSLFEGIDFYTSITRARFEELCADLFRSTLDPVEKVLRDAKLDKSQVDEIVLVGGSTRIPKVQKLVTDYFNGKEPNRSINPDEAVAYGAAVQAAILTGDESSKTQDLLLLDVAPLSLGIETAGGVMTKLIPRNSTIPTKKSEIFSTYADNQPGVLIQVFEGERAKTKDNNLLGKFELSGIPPAPRGVPQIEVTFDVDSNGILNVSAVEKGTGKSNKITITNDKGRLSKEDIEKMVAEAEKFKEEDEKESQRIASKNQLESIAYSLKNTISEAGDKLEQADKDTVTKKAEETISWLDSNTTASKEEFDDKLKELQDIANPIMSKLYQAGGAPGGAAGGAPGGFPGGAPPAPEAEGPTVEEVD\n>P61869\nMRLSFFTALSAVASLGYALPGKLQSRDVSTSELDQFEFWVQYAAASYYEADYTAQVGDKLSCSKGNCPEVEATGATVSYDFSDSTITDTAGYIAVDHTNSAVVLAFRGSYSVRNWVADATFVHTNPGLCDGCLAELGFWSSWKLVRDDIIKELKEVVAQNPNYELVVVGHSLGAAVATLAATDLRGKGYPSAKLYAYASPRVGNAALAKYITAQGNNFRFTHTNDPVPKLPLLSMGYVHVSPEYWITSPNNATVSTSDIKVIDGDVSFDGNTGTGLPLLTDFEAHIWYFVQVDAGKGPGLPFKRV\n>P10275\nMEVQLGLGRVYPRPPSKTYRGAFQNLFQSVREVIQNPGPRHPEAASAAPPGASLLLLQQQQQQQQQQQQQQQQQQQQQQQETSPRQQQQQQGEDGSPQAHRRGPTGYLVLDEEQQPSQPQSALECHPERGCVPEPGAAVAASKGLPQQLPAPPDEDDSAAPSTLSLLGPTFPGLSSCSADLKDILSEASTMQLLQQQQQEAVSEGSSSGRAREASGAPTSSKDNYLGGTSTISDNAKELCKAVSVSMGLGVEALEHLSPGEQLRGDCMYAPLLGVPPAVRPTPCAPLAECKGSLLDDSAGKSTEDTAEYSPFKGGYTKGLEGESLGCSGSAAAGSSGTLELPSTLSLYKSGALDEAAAYQSRDYYNFPLALAGPPPPPPPPHPHARIKLENPLDYGSAWAAAAAQCRYGDLASLHGAGAAGPGSGSPSAAASSSWHTLFTAEEGQLYGPCGGGGGGGGGGGGGGGGGGGGGGGEAGAVAPYGYTRPPQGLAGQESDFTAPDVWYPGGMVSRVPYPSPTCVKSEMGPWMDSYSGPYGDMRLETARDHVLPIDYYFPPQKTCLICGDEASGCHYGALTCGSCKVFFKRAAEGKQKYLCASRNDCTIDKFRRKNCPSCRLRKCYEAGMTLGARKLKKLGNLKLQEEGEASSTTSPTEETTQKLTVSHIEGYECQPIFLNVLEAIEPGVVCAGHDNNQPDSFAALLSSLNELGERQLVHVVKWAKALPGFRNLHVDDQMAVIQYSWMGLMVFAMGWRSFTNVNSRMLYFAPDLVFNEYRMHKSRMYSQCVRMRHLSQEFGWLQITPQEFLCMKALLLFSIIPVDGLKNQKFFDELRMNYIKELDRIIACKRKNPTSCSRRFYQLTKLLDSVQPIARELHQFTFDLLIKSHMVSVDFPEMMAEIISVQVPKILSGKVKPIYFHTQ\n>P38398\nMDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLKLLNQKKGPSQCPLCKNDITKRSLQESTRFSQLVEELLKIICAFQLDTGLEYANSYNFAKKENNSPEHLKDEVSIIQSMGYRNRAKRLLQSEPENPSLQETSLSVQLSNLGTVRTLRTKQRIQPQKTSVYIELGSDSSEDTVNKATYCSVGDQELLQITPQGTRDEISLDSAKKAACEFSETDVTNTEHHQPSNNDLNTTEKRAAERHPEKYQGSSVSNLHVEPCGTNTHASSLQHENSSLLLTKDRMNVEKAEFCNKSKQPGLARSQHNRWAGSKETCNDRRTPSTEKKVDLNADPLCERKEWNKQKLPCSENPRDTEDVPWITLNSSIQKVNEWFSRSDELLGSDDSHDGESESNAKVADVLDVLNEVDEYSGSSEKIDLLASDPHEALICKSERVHSKSVESNIEDKIFGKTYRKKASLPNLSHVTENLIIGAFVTEPQIIQERPLTNKLKRKRRPTSGLHPEDFIKKADLAVQKTPEMINQGTNQTEQNGQVMNITNSGHENKTKGDSIQNEKNPNPIESLEKESAFKTKAEPISSSISNMELELNIHNSKAPKKNRLRRKSSTRHIHALELVVSRNLSPPNCTELQIDSCSSSEEIKKKKYNQMPVRHSRNLQLMEGKEPATGAKKSNKPNEQTSKRHDSDTFPELKLTNAPGSFTKCSNTSELKEFVNPSLPREEKEEKLETVKVSNNAEDPKDLMLSGERVLQTERSVESSSISLVPGTDYGTQESISLLEVSTLGKAKTEPNKCVSQCAAFENPKGLIHGCSKDNRNDTEGFKYPLGHEVNHSRETSIEMEESELDAQYLQNTFKVSKRQSFAPFSNPGNAEEECATFSAHSGSLKKQSPKVTFECEQKEENQGKNESNIKPVQTVNITAGFPVVGQKDKPVDNAKCSIKGGSRFCLSSQFRGNETGLITPNKHGLLQNPYRIPPLFPIKSFVKTKCKKNLLEENFEEHSMSPEREMGNENIPSTVSTISRNNIRENVFKEASSSNINEVGSSTNEVGSSINEIGSSDENIQAELGRNRGPKLNAMLRLGVLQPEVYKQSLPGSNCKHPEIKKQEYEEVVQTVNTDFSPYLISDNLEQPMGSSHASQVCSETPDDLLDDGEIKEDTSFAENDIKESSAVFSKSVQKGELSRSPSPFTHTHLAQGYRRGAKKLESSEENLSSEDEELPCFQHLLFGKVNNIPSQSTRHSTVATECLSKNTEENLLSLKNSLNDCSNQVILAKASQEHHLSEETKCSASLFSSQCSELEDLTANTNTQDPFLIGSSKQMRHQSESQGVGLSDKELVSDDEERGTGLEENNQEEQSMDSNLGEAASGCESETSVSEDCSGLSSQSDILTTQQRDTMQHNLIKLQQEMAELEAVLEQHGSQPSNSYPSIISDSSALEDLRNPEQSTSEKAVLTSQKSSEYPISQNPEGLSADKFEVSADSSTSKNKEPGVERSSPSKCPSLDDRWYMHSCSGSLQNRNYPSQEELIKVVDVEEQQLEESGPHDLTETSYLPRQDLEGTPYLESGISLFSDDPESDPSEDRAPESARVGNIPSSTSALKVPQLKVAESAQSPAAAHTTDTAGYNAMEESVSREKPELTASTERVNKRMSMVVSGLTPEEFMLVYKFARKHHITLTNLITEETTHVVMKTDAEFVCERTLKYFLGIAGGKWVVSYFWVTQSIKERKMLNEHDFEVRGDVVNGRNHQGPKRARESQDRKIFRGLEICCYGPFTNMPTDQLEWMVQLCGASVVKELSSFTLGTGVHPIVVVQPDAWTEDNGFHAIGQMCEAPVVTREWVLDSVALYQCQELDTYLIPQIPHSHY\n>P02452\nMFSFVDLRLLLLLAATALLTHGQEEGQVEGQDEDIPPITCVQNGLRYHDRDVWKPEPCRICVCDNGKVLCDDVICDETKNCPGAEVPEGECCPVCPDGSESPTDQETTGVEGPKGDTGPRGPRGPAGPPGRDGIPGQPGLPGPPGPPGPPGPPGLGGNFAPQLSYGYDEKSTGGISVPGPMGPSGPRGLPGPPGAPGPQGFQGPPGEPGEPGASGPMGPRGPPGPPGKNGDDGEAGKPGRPGERGPPGPQGARGLPGTAGLPGMKGHRGFSGLDGAKGDAGPAGPKGEPGSPGENGAPGQMGPRGLPGERGRPGAPGPAGARGNDGATGAAGPPGPTGPAGPPGFPGAVGAKGEAGPQGPRGSEGPQGVRGEPGPPGPAGAAGPAGNPGADGQPGAKGANGAPGIAGAPGFPGARGPSGPQGPGGPPGPKGNSGEPGAPGSKGDTGAKGEPGPVGVQGPPGPAGEEGKRGARGEPGPTGLPGPPGERGGPGSRGFPGADGVAGPKGPAGERGSPGPAGPKGSPGEAGRPGEAGLPGAKGLTGSPGSPGPDGKTGPPGPAGQDGRPGPPGPPGARGQAGVMGFPGPKGAAGEPGKAGERGVPGPPGAVGPAGKDGEAGAQGPPGPAGPAGERGEQGPAGSPGFQGLPGPAGPPGEAGKPGEQGVPGDLGAPGPSGARGERGFPGERGVQGPPGPAGPRGANGAPGNDGAKGDAGAPGAPGSQGAPGLQGMPGERGAAGLPGPKGDRGDAGPKGADGSPGKDGVRGLTGPIGPPGPAGAPGDKGESGPSGPAGPTGARGAPGDRGEPGPPGPAGFAGPPGADGQPGAKGEPGDAGAKGDAGPPGPAGPAGPPGPIGNVGAPGAKGARGSAGPPGATGFPGAAGRVGPPGPSGNAGPPGPPGPAGKEGGKGPRGETGPAGRPGEVGPPGPPGPAGEKGSPGADGPAGAPGTPGPQGIAGQRGVVGLPGQRGERGFPGLPGPSGEPGKQGPSGASGERGPPGPMGPPGLAGPPGESGREGAPGAEGSPGRDGSPGAKGDRGETGPAGPPGAPGAPGAPGPVGPAGKSGDRGETGPAGPAGPVGPVGARGPAGPQGPRGDKGETGEQGDRGIKGHRGFSGLQGPPGPPGSPGEQGPSGASGPAGPRGPPGSAGAPGKDGLNGLPGPIGPPGPRGRTGDAGPVGPPGPPGPPGPPGPPSAGFDFSFLPQPPQEKAHDGGRYYRADDANVVRDRDLEVDTTLKSLSQQIENIRSPEGSRKNPARTCRDLKMCHSDWKSGEYWIDPNQGCNLDAIKVFCNMETGETCVYPTQPSVAQKNWYISKNPKDKRHVWFGESMTDGFQFEYGGQGSDPADVAIQLTFLRLMSTEASQNITYHCKNSVAYMDQQTGNLKKALLLQGSNEIEIRAEGNSRFTYSVTVDGCTSHTGAWGKTVIEYKTTKTSRLPIIDVAPLDVGAPDQEFGFDVGPVCFL\n>P12821\nMGAASGRRGPGLLLPLPLLLLLPPQPALALDPGLQPGNFSADEAGAQLFAQSYNSSAEQVLFQSVAASWAHDTNITAENARRQEEAALLSQEFAEAWGQKAKELYEPIWQNFTDPQLRRIIGAVRTLGSANLPLAKRQQYNALLSNMSRIYSTAKVCLPNKTATCWSLDPDLTNILASSRSYAMLLFAWEGWHNAAGIPLKPLYEDFTALSNEAYKQDGFTDTGAYWRSWYNSPTFEDDLEHLYQQLEPLYLNLHAFVRRALHRRYGDRYINLRGPIPAHLLGDMWAQSWENIYDMVVPFPDKPNLDVTSTMLQQGWNATHMFRVAEEFFTSLELSPMPPEFWEGSMLEKPADGREVVCHASAWDFYNRKDFRIKQCTRVTMDQLSTVHHEMGHIQYYLQYKDLPVSLRRGANPGFHEAIGDVLALSVSTPEHLHKIGLLDRVTNDTESDINYLLKMALEKIAFLPFGYLVDQWRWGVFSGRTPPSRYNFDWWYLRTKYQGICPPVTRNETHFDAGAKFHVPNVTPYIRYFVSFVLQFQFHEALCKEAGYEGPLHQCDIYRSTKAGAKLRKVLQAGSSRPWQEVLKDMVGLDALDAQPLLKYFQPVTQWLQEQNQQNGEVLGWPEYQWHPPLPDNYPEGIDLVTDEAEASKFVEEYDRTSQVVWNEYAEANWNYNTNITTETSKILLQKNMQIANHTLKYGTQARKFDVNQLQNTTIKRIIKKVQDLERAALPAQELEEYNKILLDMETTYSVATVCHPNGSCLQLEPDLTNVMATSRKYEDLLWAWEGWRDKAGRAILQFYPKYVELINQAARLNGYVDAGDSWRSMYETPSLEQDLERLFQELQPLYLNLHAYVRRALHRHYGAQHINLEGPIPAHLLGNMWAQTWSNIYDLVVPFPSAPSMDTTEAMLKQGWTPRRMFKEADDFFTSLGLLPVPPEFWNKSMLEKPTDGREVVCHASAWDFYNGKDFRIKQCTTVNLEDLVVAHHEMGHIQYFMQYKDLPVALREGANPGFHEAIGDVLALSVSTPKHLHSLNLLSSEGGSDEHDINFLMKMALDKIAFIPFSYLVDQWRWRVFDGSITKENYNQEWWSLRLKYQGLCPPVPRTQGDFDPGAKFHIPSSVPYIRYFVSFIIQFQFHEALCQAAGHTGPLHKCDIYQSKEAGQRLATAMKLGFSRPWPEAMQLITGQPNMSASAMLSYFKPLLDWLRTENELHGEKLGWPQYNWTPNSARSEGPLPDSGRVSFLGLDLDAQQARVGQWLLLFLGIALLVATLGLSQRLFSIRHRSLHRHSHGPQFGSEVELRHS\n>P00968\nMPKRTDIKSILILGAGPIVIGQACEFDYSGAQACKALREEGYRVILVNSNPATIMTDPEMADATYIEPIHWEVVRKIIEKERPDAVLPTMGGQTALNCALELERQGVLEEFGVTMIGATADAIDKAEDRRRFDVAMKKIGLETARSGIAHTMEEALAVAADVGFPCIIRPSFTMGGSGGGIAYNREEFEEICARGLDLSPTKELLIDESLIGWKEYEMEVVRDKNDNCIIVCSIENFDAMGIHTGDSITVAPAQTLTDKEYQIMRNASMAVLREIGVETGGSNVQFAVNPKNGRLIVIEMNPRVSRSSALASKATGFPIAKVAAKLAVGYTLDELMNDITGGRTPASFEPSIDYVVTKIPRFNFEKFAGANDRLTTQMKSVGEVMAIGRTQQESLQKALRGLEVGATGFDPKVSLDDPEALTKIRRELKDAGADRIWYIADAFRAGLSVDGVFNLTNIDRWFLVQIEELVRLEEKVAEVGITGLNADFLRQLKRKGFADARLAKLAGVREAEIRKLRDQYDLHPVYKRVDTCAAEFATDTAYMYSTYEEECEANPSTDREKIMVLGGGPNRIGQGIEFDYCCVHASLALREDGYETIMVNCNPETVSTDYDTSDRLYFEPVTLEDVLEIVRIEKPKGVIVQYGGQTPLKLARALEAAGVPVIGTSPDAIDRAEDRERFQHAVERLKLKQPANATVTAIEMAVEKAKEIGYPLVVRPSYVLGGRAMEIVYDEADLRRYFQTAVSVSNDAPVLLDHFLDDAVEVDVDAICDGEMVLIGGIMEHIEQAGVHSGDSACSLPAYTLSQEIQDVMRQQVQKLAFELQVRGLMNVQFAVKNNEVYLIEVNPRAARTVPFVSKATGVPLAKVAARVMAGKSLAEQGVTKEVIPPYYSVKEVVLPFNKFPGVDPLLGPEMRSTGEVMGVGRTFAEAFAKAQLGSNSTMKKHGRALLSVREGDKERVVDLAAKLLKQGFELDATHGTAIVLGEAGINPRLVNKVHEGRPHIQDRIKNGEYTYIINTTSGRRAIEDSRVIRRSALQYKVHYDTTLNGGFATAMALNADATEKVISVQEMHAQIK\n>P42336\nMPPRPSSGELWGIHLMPPRILVECLLPNGMIVTLECLREATLITIKHELFKEARKYPLHQLLQDESSYIFVSVTQEAEREEFFDETRRLCDLRLFQPFLKVIEPVGNREEKILNREIGFAIGMPVCEFDMVKDPEVQDFRRNILNVCKEAVDLRDLNSPHSRAMYVYPPNVESSPELPKHIYNKLDKGQIIVVIWVIVSPNNDKQKYTLKINHDCVPEQVIAEAIRKKTRSMLLSSEQLKLCVLEYQGKYILKVCGCDEYFLEKYPLSQYKYIRSCIMLGRMPNLMLMAKESLYSQLPMDCFTMPSYSRRISTATPYMNGETSTKSLWVINSALRIKILCATYVNVNIRDIDKIYVRTGIYHGGEPLCDNVNTQRVPCSNPRWNEWLNYDIYIPDLPRAARLCLSICSVKGRKGAKEEHCPLAWGNINLFDYTDTLVSGKMALNLWPVPHGLEDLLNPIGVTGSNPNKETPCLELEFDWFSSVVKFPDMSVIEEHANWSVSREAGFSYSHAGLSNRLARDNELRENDKEQLKAISTRDPLSEITEQEKDFLWSHRHYCVTIPEILPKLLLSVKWNSRDEVAQMYCLVKDWPPIKPEQAMELLDCNYPDPMVRGFAVRCLEKYLTDDKLSQYLIQLVQVLKYEQYLDNLLVRFLLKKALTNQRIGHFFFWHLKSEMHNKTVSQRFGLLLESYCRACGMYLKHLNRQVEAMEKLINLTDILKQEKKDETQKVQMKFLVEQMRRPDFMDALQGFLSPLNPAHQLGNLRLEECRIMSSAKRPLWLNWENPDIMSELLFQNNEIIFKNGDDLRQDMLTLQIIRIMENIWQNQGLDLRMLPYGCLSIGDCVGLIEVVRNSHTIMQIQCKGGLKGALQFNSHTLHQWLKDKNKGEIYDAAIDLFTRSCAGYCVATFILGIGDRHNSNIMVKDDGQLFHIDFGHFLDHKKKKFGYKRERVPFVLTQDFLIVISKGAQECTKTREFERFQEMCYKAYLAIRQHANLFINLFSMMLGSGMPELQSFDDIAYIRKTLALDKTEQEALEYFMKQMNDAHHGGWTTKMDWIFHTIKQHALN\n>P42345\nMLGTGPAAATTAATTSSNVSVLQQFASGLKSRNEETRAKAAKELQHYVTMELREMSQEESTRFYDQLNHHIFELVSSSDANERKGGILAIASLIGVEGGNATRIGRFANYLRNLLPSNDPVVMEMASKAIGRLAMAGDTFTAEYVEFEVKRALEWLGADRNEGRRHAAVLVLRELAISVPTFFFQQVQPFFDNIFVAVWDPKQAIREGAVAALRACLILTTQREPKEMQKPQWYRHTFEEAEKGFDETLAKEKGMNRDDRIHGALLILNELVRISSMEGERLREEMEEITQQQLVHDKYCKDLMGFGTKPRHITPFTSFQAVQPQQSNALVGLLGYSSHQGLMGFGTSPSPAKSTLVESRCCRDLMEEKFDQVCQWVLKCRNSKNSLIQMTILNLLPRLAAFRPSAFTDTQYLQDTMNHVLSCVKKEKERTAAFQALGLLSVAVRSEFKVYLPRVLDIIRAALPPKDFAHKRQKAMQVDATVFTCISMLARAMGPGIQQDIKELLEPMLAVGLSPALTAVLYDLSRQIPQLKKDIQDGLLKMLSLVLMHKPLRHPGMPKGLAHQLASPGLTTLPEASDVGSITLALRTLGSFEFEGHSLTQFVRHCADHFLNSEHKEIRMEAARTCSRLLTPSIHLISGHAHVVSQTAVQVVADVLSKLLVVGITDPDPDIRYCVLASLDERFDAHLAQAENLQALFVALNDQVFEIRELAICTVGRLSSMNPAFVMPFLRKMLIQILTELEHSGIGRIKEQSARMLGHLVSNAPRLIRPYMEPILKALILKLKDPDPDPNPGVINNVLATIGELAQVSGLEMRKWVDELFIIIMDMLQDSSLLAKRQVALWTLGQLVASTGYVVEPYRKYPTLLEVLLNFLKTEQNQGTRREAIRVLGLLGALDPYKHKVNIGMIDQSRDASAVSLSESKSSQDSSDYSTSEMLVNMGNLPLDEFYPAVSMVALMRIFRDQSLSHHHTMVVQAITFIFKSLGLKCVQFLPQVMPTFLNVIRVCDGAIREFLFQQLGMLVSFVKSHIRPYMDEIVTLMREFWVMNTSIQSTIILLIEQIVVALGGEFKLYLPQLIPHMLRVFMHDNSPGRIVSIKLLAAIQLFGANLDDYLHLLLPPIVKLFDAPEAPLPSRKAALETVDRLTESLDFTDYASRIIHPIVRTLDQSPELRSTAMDTLSSLVFQLGKKYQIFIPMVNKVLVRHRINHQRYDVLICRIVKGYTLADEEEDPLIYQHRMLRSGQGDALASGPVETGPMKKLHVSTINLQKAWGAARRVSKDDWLEWLRRLSLELLKDSSSPSLRSCWALAQAYNPMARDLFNAAFVSCWSELNEDQQDELIRSIELALTSQDIAEVTQTLLNLAEFMEHSDKGPLPLRDDNGIVLLGERAAKCRAYAKALHYKELEFQKGPTPAILESLISINNKLQQPEAAAGVLEYAMKHFGELEIQATWYEKLHEWEDALVAYDKKMDTNKDDPELMLGRMRCLEALGEWGQLHQQCCEKWTLVNDETQAKMARMAAAAAWGLGQWDSMEEYTCMIPRDTHDGAFYRAVLALHQDLFSLAQQCIDKARDLLDAELTAMAGESYSRAYGAMVSCHMLSELEEVIQYKLVPERREIIRQIWWERLQGCQRIVEDWQKILMVRSLVVSPHEDMRTWLKYASLCGKSGRLALAHKTLVLLLGVDPSRQLDHPLPTVHPQVTYAYMKNMWKSARKIDAFQHMQHFVQTMQQQAQHAIATEDQQHKQELHKLMARCFLKLGEWQLNLQGINESTIPKVLQYYSAATEHDRSWYKAWHAWAVMNFEAVLHYKHQNQARDEKKKLRHASGANITNATTAATTAATATTTASTEGSNSESEAESTENSPTPSPLQKKVTEDLSKTLLMYTVPAVQGFFRSISLSRGNNLQDTLRVLTLWFDYGHWPDVNEALVEGVKAIQIDTWLQVIPQLIARIDTPRPLVGRLIHQLLTDIGRYHPQALIYPLTVASKSTTTARHNAANKILKNMCEHSNTLVQQAMMVSEELIRVAILWHEMWHEGLEEASRLYFGERNVKGMFEVLEPLHAMMERGPQTLKETSFNQAYGRDLMEAQEWCRKYMKSGNVKDLTQAWDLYYHVFRRISKQLPQLTSLELQYVSPKLLMCRDLELAVPGTYDPNQPIIRIQSIAPSLQVITSKQRPRKLTLMGSNGHEFVFLLKGHEDLRQDERVMQLFGLVNTLLANDPTSLRKNLSIQRYAVIPLSTNSGLIGWVPHCDTLHALIRDYREKKKILLNIEHRIMLRMAPDYDHLTLMQKVEVFEHAVNNTAGDDLAKLLWLKSPSSEVWFDRRTNYTRSLAVMSMVGYILGLGDRHPSNLMLDRLSGKILHIDFGDCFEVAMTREKFPEKIPFRLTRMLTNAMEVTGLDGNYRITCHTVMEVLREHKDSVMAVLEAFVYDPLLNWRLMDTNTKGNKRSRTRTDSYSAGQSVEILDGVELGEPAHKKTGTTVPESIHSFIGDGLVKPEALNKKAIQIINRVRDKLTGRDFSHDDTLDVPTQVELLIKQATSHENLCQCYIGWCPFW\n>P0DTC2\nMFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT\n>P13569\nMQRSPLEKASVVSKLFFSWTRPILRKGYRQRLELSDIYQIPSVDSADNLSEKLEREWDRELASKKNPKLINALRRCFFWRFMFYGIFLYLGEVTKAVQPLLLGRIIASYDPDNKEERSIAIYLGIGLCLLFIVRTLLLHPAIFGLHHIGMQMRIAMFSLIYKKTLKLSSRVLDKISIGQLVSLLSNNLNKFDEGLALAHFVWIAPLQVALLMGLIWELLQASAFCGLGFLIVLALFQAGLGRMMMKYRDQRAGKISERLVITSEMIENIQSVKAYCWEEAMEKMIENLRQTELKLTRKAAYVRYFNSSAFFFSGFFVVFLSVLPYALIKGIILRKIFTTISFCIVLRMAVTRQFPWAVQTWYDSLGAINKIQDFLQKQEYKTLEYNLTTTEVVMENVTAFWEEGFGELFEKAKQNNNNRKTSNGDDSLFFSNFSLLGTPVLKDINFKIERGQLLAVAGSTGAGKTSLLMVIMGELEPSEGKIKHSGRISFCSQFSWIMPGTIKENIIFGVSYDEYRYRSVIKACQLEEDISKFAEKDNIVLGEGGITLSGGQRARISLARAVYKDADLYLLDSPFGYLDVLTEKEIFESCVCKLMANKTRILVTSKMEHLKKADKILILHEGSSYFYGTFSELQNLQPDFSSKLMGCDSFDQFSAERRNSILTETLHRFSLEGDAPVSWTETKKQSFKQTGEFGEKRKNSILNPINSIRKFSIVQKTPLQMNGIEEDSDEPLERRLSLVPDSEQGEAILPRISVISTGPTLQARRRQSVLNLMTHSVNQGQNIHRKTTASTRKVSLAPQANLTELDIYSRRLSQETGLEISEEINEEDLKECFFDDMESIPAVTTWNTYLRYITVHKSLIFVLIWCLVIFLAEVAASLVVLWLLGNTPLQDKGNSTHSRNNSYAVIITSTSSYYVFYIYVGVADTLLAMGFFRGLPLVHTLITVSKILHHKMLHSVLQAPMSTLNTLKAGGILNRFSKDIAILDDLLPLTIFDFIQLLLIVIGAIAVVAVLQPYIFVATVPVIVAFIMLRAYFLQTSQQLKQLESEGRSPIFTHLVTSLKGLWTLRAFGRQPYFETLFHKALNLHTANWFLYLSTLRWFQMRIEMIFVIFFIAVTFISILTTGEGEGRVGIILTLAMNIMSTLQWAVNSSIDVDSLMRSVSRVFKFIDMPTEGKPTKSTKPYKNGQLSKVMIIENSHVKKDDIWPSGGQMTVKDLTAKYTEGGNAILENISFSISPGQRVGLLGRTGSGKSTLLSAFLRLLNTEGEIQIDGVSWDSITLQQWRKAFGVIPQKVFIFSGTFRKNLDPYEQWSDQEIWKVADEVGLRSVIEQFPGKLDFVLVDGGCVLSHGHKQLMCLARSVLSKAKILLLDEPSAHLDPVTYQIIRRTLKQAFADCTVILCEHRIEAMLECQQFLVIEENKVRQYDSIQKLLNERSLFRQAISPSDRVKLFPHRNSSKCKSKPQIAALKEETEEEVQDTRL\n>O60885\nMSAESGPGTRLRNLPVMGDGLETSQMSTTQAQAQPQPANAASTNPPPPETSNPNKPKRQTNQLQYLLRVVLKTLWKHQFAWPFQQPVDAVKLNLPDYYKIIKTPMDMGTIKKRLENNYYWNAQECIQDFNTMFTNCYIYNKPGDDIVLMAEALEKLFLQKINELPTEETEIMIVQAKGRGRGRKETGTAKPGVSTVPNTTQASTPPQTQTPQPNPPPVQATPHPFPAVTPDLIVQTPVMTVVPPQPLQTPPPVPPQPQPPPAPAPQPVQSHPPIIAATPQPVKTKKGVKRKADTTTPTTIDPIHEPPSLPPEPKTTKLGQRRESSRPVKPPKKDVPDSQQHPAPEKSSKVSEQLKCCSGILKEMFAKKHAAYAWPFYKPVDVEALGLHDYCDIIKHPMDMSTIKSKLEAREYRDAQEFGADVRLMFSNCYKYNPPDHEVVAMARKLQDVFEMRFAKMPDEPEEPVVAVSSPAVPPPTKVVAPPSSSDSSSDSSSDSDSSTDDSEEERAQRLAELQEQLKAVHEQLAALSQPQQNKPKKKEKDKKEKKKEKHKRKEEVEENKKSKAKEPPPKKTKKNNSSNSNVSKKEPAPMKSKPPPTYESEEEDKCKPMSYEEKRQLSLDINKLPGEKLGRVVHIIQSREPSLKNSNPDEIEIDFETLKPSTLRELERYVTSCLRKKRKPQAEKVDVIAGSSKMKGFSSSESESSSESSSSDSEDSETEMAPKSKKKGHPGREQKKHHHHHHQQMQQAPAPVPQQPPPPPQQPPPPPPPQQQQQPPPPPPPPSMPQQAAPAMKSSPPPFIATQVPVLEPQLPGSVFDPIGHFTQPILHLPQPELPPHLPQPPEHSTPPHLNQHAVVSPPALHNALPQQPSRPSNRAAALPPKPARPPAVSPALTQTPLLPQPPMAQPPQVLLEDEEPPAPPLTSMQMQLYLQQLQKVQPPTPLLPSVKVQSQPPPPLPPPPHPSVQQQLQQQPPPPPPPQPQPPPQQQHQPPPRPVHLQPMQFSTHIQQPPPPQGQQPPHPPPGQQPPPPQPAKPQQVIQHHHSPRHHKSDPYSTGHLREAPSPLMIHSPQMSQFQSLTHQSPPQQNVQPKKQELRAASVVQPQPLVVVKEEKIHSPIIRSEPFSPSLRPEPPKHPESIKAPVHLPQRPEMKPVDVGRPVIRPPEQNAPPPGAPDKDKQKQEPKTPVAPKKDLKIKNMGSWASLVQKHPTTPSSTAKSSSDSFEQFRRAAREKEEREKALKAQAEHAEKEKERLRQERMRSREDEDALEQARRAHEEARRRQEQQQQQRQEQQQQQQQQAAAVAAAATPQAQSSQPQSMLDQQRELARKREQERRRREAMAATIDMNFQSDLLSIFEENLF\n>P06756\nMAFPPRRRLRLGPRGLPLLLSGLLLPLCRAFNLDVDSPAEYSGPEGSYFGFAVDFFVPSASSRMFLLVGAPKANTTQPGIVEGGQVLKCDWSSTRRCQPIEFDATGNRDYAKDDPLEFKSHQWFGASVRSKQDKILACAPLYHWRTEMKQEREPVGTCFLQDGTKTVEYAPCRSQDIDADGQGFCQGGFSIDFTKADRVLLGGPGSFYWQGQLISDQVAEIVSKYDPNVYSIKYNNQLATRTAQAIFDDSYLGYSVAVGDFNGDGIDDFVSGVPRAARTLGMVYIYDGKNMSSLYNFTGEQMAAYFGFSVAATDINGDDYADVFIGAPLFMDRGSDGKLQEVGQVSVSLQRASGDFQTTKLNGFEVFARFGSAIAPLGDLDQDGFNDIAIAAPYGGEDKKGIVYIFNGRSTGLNAVPSQILEGQWAARSMPPSFGYSMKGATDIDKNGYPDLIVGAFGVDRAILYRARPVITVNAGLEVYPSILNQDNKTCSLPGTALKVSCFNVRFCLKADGKGVLPRKLNFQVELLLDKLKQKGAIRRALFLYSRSPSHSKNMTISRGGLMQCEELIAYLRDESEFRDKLTPITIFMEYRLDYRTAADTTGLQPILNQFTPANISRQAHILLDCGEDNVCKPKLEVSVDSDQKKIYIGDDNPLTLIVKAQNQGEGAYEAELIVSIPLQADFIGVVRNNEALARLSCAFKTENQTRQVVCDLGNPMKAGTQLLAGLRFSVHQQSEMDTSVKFDLQIQSSNLFDKVSPVVSHKVDLAVLAAVEIRGVSSPDHVFLPIPNWEHKENPETEEDVGPVVQHIYELRNNGPSSFSKAMLHLQWPYKYNNNTLLYILHYDIDGPMNCTSDMEINPLRIKISSLQTTEKNDTVAGQGERDHLITKRDLALSEGDIHTLGCGVAQCLKIVCQVGRLDRGKSAILYVKSLLWTETFMNKENQNHSYSLKSSASFNVIEFPYKNLPIEDITNSTLVTTNVTWGIQPAPMPVPVWVIILAVLAGLLLLAVLVFVMYRMGFFKRVRPPQEEQEREQLQPHENGEGNSET\n>P04626\nMELAALCRWGLLLALLPPGAASTQVCTGTDMKLRLPASPETHLDMLRHLYQGCQVVQGNLELTYLPTNASLSFLQDIQEVQGYVLIAHNQVRQVPLQRLRIVRGTQLFEDNYALAVLDNGDPLNNTTPVTGASPGGLRELQLRSLTEILKGGVLIQRNPQLCYQDTILWKDIFHKNNQLALTLIDTNRSRACHPCSPMCKGSRCWGESSEDCQSLTRTVCAGGCARCKGPLPTDCCHEQCAAGCTGPKHSDCLACLHFNHSGICELHCPALVTYNTDTFESMPNPEGRYTFGASCVTACPYNYLSTDVGSCTLVCPLHNQEVTAEDGTQRCEKCSKPCARVCYGLGMEHLREVRAVTSANIQEFAGCKKIFGSLAFLPESFDGDPASNTAPLQPEQLQVFETLEEITGYLYISAWPDSLPDLSVFQNLQVIRGRILHNGAYSLTLQGLGISWLGLRSLRELGSGLALIHHNTHLCFVHTVPWDQLFRNPHQALLHTANRPEDECVGEGLACHQLCARGHCWGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYVNARHCLPCHPECQPQNGSVTCFGPEADQCVACAHYKDPPFCVARCPSGVKPDLSYMPIWKFPDEEGACQPCPINCTHSCVDLDDKGCPAEQRASPLTSIISAVVGILLVVVLGVVFGILIKRRQQKIRKYTMRRLLQETELVEPLTPSGAMPNQAQMRILKETELRKVKVLGSGAFGTVYKGIWIPDGENVKIPVAIKVLRENTSPKANKEILDEAYVMAGVGSPYVSRLLGICLTSTVQLVTQLMPYGCLLDHVRENRGRLGSQDLLNWCMQIAKGMSYLEDVRLVHRDLAARNVLVKSPNHVKITDFGLARLLDIDETEYHADGGKVPIKWMALESILRRRFTHQSDVWSYGVTVWELMTFGAKPYDGIPAREIPDLLEKGERLPQPPICTIDVYMIMVKCWMIDSECRPRFRELVSEFSRMARDPQRFVVIQNEDLGPASPLDSTFYRSLLEDDDMGDLVDAEEYLVPQQGFFCPDPAPGAGGMVHHRHRSSSTRSGGGDLTLGLEPSEEEAPRSPLAPSEGAGSDVFDGDLGMGAAKGLQSLPTHDPSPLQRYSEDPTVPLPSETDGYVAPLTCSPQPEYVNQPDVRPQPPSPREGPLPAARPAGATLERPKTLSPGKNGVVKDVFAFGGAVENPEYLTPQGGAAPQPHPPPAFSPAFDNLYYWDQDPPERGAPPSTFKGTPTAENPEYLGLDVPV\n>Q13936\nMVNENTRMYIPEENHQGSNYGSPRPAHANMNANAAAGLAPEHIPTPGAALSWQAAIDAARQAKLMGSAGNATISTVSSTQRKRQQYGKPKKQGSTTATRPPRALLCLTLKNPIRRACISIVEWKPFEIIILLTIFANCVALAIYIPFPEDDSNATNSNLERVEYLFLIIFTVEAFLKVIAYGLLFHPNAYLRNGWNLLDFIIVVVGLFSAILEQATKADGANALGGKGAGFDVKALRAFRVLRPLRLVSGVPSLQVVLNSIIKAMVPLLHIALLVLFVIIIYAIIGLELFMGKMHKTCYNQEGIADVPAEDDPSPCALETGHGRQCQNGTVCKPGWDGPKHGITNFDNFAFAMLTVFQCITMEGWTDVLYWVNDAVGRDWPWIYFVTLIIIGSFFVLNLVLGVLSGEFSKEREKAKARGDFQKLREKQQLEEDLKGYLDWITQAEDIDPENEDEGMDEEKPRNMSMPTSETESVNTENVAGGDIEGENCGARLAHRISKSKFSRYWRRWNRFCRRKCRAAVKSNVFYWLVIFLVFLNTLTIASEHYNQPNWLTEVQDTANKALLALFTAEMLLKMYSLGLQAYFVSLFNRFDCFVVCGGILETILVETKIMSPLGISVLRCVRLLRIFKITRYWNSLSNLVASLLNSVRSIASLLLLLFLFIIIFSLLGMQLFGGKFNFDEMQTRRSTFDNFPQSLLTVFQILTGEDWNSVMYDGIMAYGGPSFPGMLVCIYFIILFICGNYILLNVFLAIAVDNLADAESLTSAQKEEEEEKERKKLARTASPEKKQELVEKPAVGESKEEKIELKSITADGESPPATKINMDDLQPNENEDKSPYPNPETTGEEDEEEPEMPVGPRPRPLSELHLKEKAVPMPEASAFFIFSSNNRFRLQCHRIVNDTIFTNLILFFILLSSISLAAEDPVQHTSFRNHILFYFDIVFTTIFTIEIALKILGNADYVFTSIFTLEIILKMTAYGAFLHKGSFCRNYFNILDLLVVSVSLISFGIQSSAINVVKILRVLRVLRPLRAINRAKGLKHVVQCVFVAIRTIGNIVIVTTLLQFMFACIGVQLFKGKLYTCSDSSKQTEAECKGNYITYKDGEVDHPIIQPRSWENSKFDFDNVLAAMMALFTVSTFEGWPELLYRSIDSHTEDKGPIYNYRVEISIFFIIYIIIIAFFMMNIFVGFVIVTFQEQGEQEYKNCELDKNQRQCVEYALKARPLRRYIPKNQHQYKVWYVVNSTYFEYLMFVLILLNTICLAMQHYGQSCLFKIAMNILNMLFTGLFTVEMILKLIAFKPKGYFSDPWNVFDFLIVIGSIIDVILSETNHYFCDAWNTFDALIVVGSIVDIAITEVNPAEHTQCSPSMNAEENSRISITFFRLFRVMRLVKLLSRGEGIRTLLWTFIKSFQALPYVALLIVMLFFIYAVIGMQVFGKIALNDTTEINRNNNFQTFPQAVLLLFRCATGEAWQDIMLACMPGKKCAPESEPSNSTEGETPCGSSFAVFYFISFYMLCAFLIINLFVAVIMDNFDYLTRDWSILGPHHLDEFKRIWAEYDPEAKGRIKHLDVVTLLRRIQPPLGFGKLCPHRVACKRLVSMNMPLNSDGTVMFNATLFALVRTALRIKTEGNLEQANEELRAIIKKIWKRTSMKLLDQVVPPAGDDEVTVGKFYATFLIQEYFRKFKKRKEQGLVGKPSQRNALSLQAGLRTLHDIGPEIRRAISGDLTAEEELDKAMKEAVSAASEDDIFRRAGGLFGNHVSYYQSDGRSAFPQTFTTQRPLHINKAGSSQGDTESPSHEKLVDSTFTPSSYSSTGSNANINNANNTALGRLPRPAGYPSTVSTVEGHGPPLSPAIRVQEVAWKLSSNRERHVPMCEDLELRRDSGSAGTQAHCLLLRKANPSRCHSRESQAAMAGQEETSQDETYEVKMNHDTEACSEPSLLSTEMLSYQDDENRQLTLPEEDKRDIRQSPKRGFLRSASLGRRASFHLECLKRQKDRGGDISQKTVLPLHLVHHQALAVAGLSPLLQRSHSPASFPRPFATPPATPGSRGWPPQPVPTLRLEGVESSEKLNSSFPSIHCGSWAETTPGGGGSSAARRVRPVSLMVPSQAGAPGRQFHGSASSLVEAVLISEGLGQFAQDPKFIEVTTQELADACDMTIEEMESAADNILSGGAPQSPNGALLPFVNCRDAGQDRAGGEEDAGCVRARGRPSEEELQDSRVYVSSL\n>P02458\nMIRLGAPQTLVLLTLLVAAVLRCQGQDVQEAGSCVQDGQRYNDKDVWKPEPCRICVCDTGTVLCDDIICEDVKDCLSPEIPFGECCPICPTDLATASGQPGPKGQKGEPGDIKDIVGPKGPPGPQGPAGEQGPRGDRGDKGEKGAPGPRGRDGEPGTPGNPGPPGPPGPPGPPGLGGNFAAQMAGGFDEKAGGAQLGVMQGPMGPMGPRGPPGPAGAPGPQGFQGNPGEPGEPGVSGPMGPRGPPGPPGKPGDDGEAGKPGKAGERGPPGPQGARGFPGTPGLPGVKGHRGYPGLDGAKGEAGAPGVKGESGSPGENGSPGPMGPRGLPGERGRTGPAGAAGARGNDGQPGPAGPPGPVGPAGGPGFPGAPGAKGEAGPTGARGPEGAQGPRGEPGTPGSPGPAGASGNPGTDGIPGAKGSAGAPGIAGAPGFPGPRGPPGPQGATGPLGPKGQTGEPGIAGFKGEQGPKGEPGPAGPQGAPGPAGEEGKRGARGEPGGVGPIGPPGERGAPGNRGFPGQDGLAGPKGAPGERGPSGLAGPKGANGDPGRPGEPGLPGARGLTGRPGDAGPQGKVGPSGAPGEDGRPGPPGPQGARGQPGVMGFPGPKGANGEPGKAGEKGLPGAPGLRGLPGKDGETGAAGPPGPAGPAGERGEQGAPGPSGFQGLPGPPGPPGEGGKPGDQGVPGEAGAPGLVGPRGERGFPGERGSPGAQGLQGPRGLPGTPGTDGPKGASGPAGPPGAQGPPGLQGMPGERGAAGIAGPKGDRGDVGEKGPEGAPGKDGGRGLTGPIGPPGPAGANGEKGEVGPPGPAGSAGARGAPGERGETGPPGPAGFAGPPGADGQPGAKGEQGEAGQKGDAGAPGPQGPSGAPGPQGPTGVTGPKGARGAQGPPGATGFPGAAGRVGPPGSNGNPGPPGPPGPSGKDGPKGARGDSGPPGRAGEPGLQGPAGPPGEKGEPGDDGPSGAEGPPGPQGLAGQRGIVGLPGQRGERGFPGLPGPSGEPGKQGAPGASGDRGPPGPVGPPGLTGPAGEPGREGSPGADGPPGRDGAAGVKGDRGETGAVGAPGAPGPPGSPGPAGPTGKQGDRGEAGAQGPMGPSGPAGARGIQGPQGPRGDKGEAGEPGERGLKGHRGFTGLQGLPGPPGPSGDQGASGPAGPSGPRGPPGPVGPSGKDGANGIPGPIGPPGPRGRSGETGPAGPPGNPGPPGPPGPPGPGIDMSAFAGLGPREKGPDPLQYMRADQAAGGLRQHDAEVDATLKSLNNQIESIRSPEGSRKNPARTCRDLKLCHPEWKSGDYWIDPNQGCTLDAMKVFCNMETGETCVYPNPANVPKKNWWSSKSKEKKHIWFGETINGGFHFSYGDDNLAPNTANVQMTFLRLLSTEGSQNITYHCKNSIAYLDEAAGNLKKALLIQGSNDVEIRAEGNSRFTYTALKDGCTKHTGKWGKTVIEYRSQKTSRLPIIDIAPMDIGGPEQEFGVDIGPVCFL\n>Q61410\nMGNGSVKPKHAKHPDGHSGNLSNEALRSKVLELERELRRKDAELQEREYHLKELREQLAKQTVAIAELTEELQSKCIQLNKLQDVIHVQGGSPLQASPDKVPLDVHRKTSGLVSLHSRRGAKAGVSAEPTTRTYDLNKPPEFSFEKARVRKDSSEKKLITDALNKNQFLKRLDPQQIKDMVECMYGEKLSTGSYVIKQGEPGNHIFVLAEGRLEVFQGEKLLSSIPMWTTFGELAILYNCTRTASVKAITNVKTWALDREVFQNIMRRTAQARDEEYRNFLRSVSLLKNLPEDKLTKIIDCLEVEYYDKGDYIIREGEEGSTFFILAKGKVKVTQSTEGHDQPQLIKTLQKGEYFGEKALISDDVRSANIIAEENDVACLVIDRETFNQTVGTFDELQKYLEGYVATLNRDDEKRHAKRSMSSWKLSKALSLEMIQLKEKVARFSSTSPFQNLEIIATLGVGGFGRVELVKVKNENVAFAMKCIRKKHIVDTKQQEHVYSEKRILEELCSPFIVKLYRTFKDNKYVYMLLEACLGGELWSILRDRGSFDEPTSKFCVACVTEAFDYLHLLGIIYRDLKPENLILDADGYLKLVDFGFAKKIGSGQKTWTFCGTPEYVAPEVILNKGHDFSVDFWSLGILVYELLTGNPPFSGIDQMMTYNLILKGIEKMDFPRKITRRPEDLIRRLCRQNPTERLGNLKNGINDIKKHRWLNGFNWEGLKARSLPSPLRRELSGPIDHSYFDKYPPEKGVPPDEMSGWDKDF\n>P13368\nMTMFWQQNVDHQSDEQDKQAKGAAPTKRLNISFNVKIAVNVNTKMTTTHINQQAPGTSSSSSNSQNASPSKIVVRQQSSSFDLRQQLARLGRQLASGQDGHGGISTILIINLLLLILLSICCDVCRSHNYTVHQSPEPVSKDQMRLLRPKLDSDVVEKVAIWHKHAAAAPPSIVEGIAISSRPQSTMAHHPDDRDRDRDPSEEQHGVDERMVLERVTRDCVQRCIVEEDLFLDEFGIQCEKADNGEKCYKTRCTKGCAQWYRALKELESCQEACLSLQFYPYDMPCIGACEMAQRDYWHLQRLAISHLVERTQPQLERAPRADGQSTPLTIRWAMHFPEHYLASRPFNIQYQFVDHHGEELDLEQEDQDASGETGSSAWFNLADYDCDEYYVCEILEALIPYTQYRFRFELPFGENRDEVLYSPATPAYQTPPEGAPISAPVIEHLMGLDDSHLAVHWHPGRFTNGPIEGYRLRLSSSEGNATSEQLVPAGRGSYIFSQLQAGTNYTLALSMINKQGEGPVAKGFVQTHSARNEKPAKDLTESVLLVGRRAVMWQSLEPAGENSMIYQSQEELADIAWSKREQQLWLLNVHGELRSLKFESGQMVSPAQQLKLDLGNISSGRWVPRRLSFDWLHHRLYFAMESPERNQSSFQIISTDLLGESAQKVGESFDLPVEQLEVDALNGWIFWRNEESLWRQDLHGRMIHRLLRIRQPGWFLVQPQHFIIHLMLPQEGKFLEISYDGGFKHPLPLPPPSNGAGNGPASSHWQSFALLGRSLLLPDSGQLILVEQQGQAASPSASWPLKNLPDCWAVILLVPESQPLTSAGGKPHSLKALLGAQAAKISWKEPERNPYQSADAARSWSYELEVLDVASQSAFSIRNIRGPIFGLQRLQPDNLYQLRVRAINVDGEPGEWTEPLAARTWPLGPHRLRWASRQGSVIHTNELGEGLEVQQEQLERLPGPMTMVNESVGYYVTGDGLLHCINLVHSQWGCPISEPLQHVGSVTYDWRGGRVYWTDLARNCVVRMDPWSGSRELLPVFEANFLALDPRQGHLYYATSSQLSRHGSTPDEAVTYYRVNGLEGSIASFVLDTQQDQLFWLVKGSGALRLYRAPLTAGGDSLQMIQQIKGVFQAVPDSLQLLRPLGALLWLERSGRRARLVRLAAPLDVMELPTPDQASPASALQLLDPQPLPPRDEGVIPMTVLPDSVRLDDGHWDDFHVRWQPSTSGGNHSVSYRLLLEFGQRLQTLDLSTPFARLTQLPQAQLQLKISITPRTAWRSGDTTRVQLTTPPVAPSQPRRLRVFVERLATALQEANVSAVLRWDAPEQGQEAPMQALEYHISCWVGSELHEELRLNQSALEARVEHLQPDQTYHFQVEARVAATGAAAGAASHALHVAPEVQAVPRVLYANAEFIGELDLDTRNRRRLVHTASPVEHLVGIEGEQRLLWVNEHVELLTHVPGSAPAKLARMRAEVLALAVDWIQRIVYWAELDATAPQAAIIYRLDLCNFEGKILQGERVWSTPRGRLLKDLVALPQAQSLIWLEYEQGSPRNGSLRGRNLTDGSELEWATVQPLIRLHAGSLEPGSETLNLVDNQGKLCVYDVARQLCTASALRAQLNLLGEDSIAGQLAQDSGYLYAVKNWSIRAYGRRRQQLEYTVELEPEEVRLLQAHNYQAYPPKNCLLLPSSGGSLLKATDCEEQRCLLNLPMITASEDCPLPIPGVRYQLNLTLARGPGSEEHDHGVEPLGQWLLGAGESLNLTDLLPFTRYRVSGILSSFYQKKLALPTLVLAPLELLTASATPSPPRNFSVRVLSPRELEVSWLPPEQLRSESVYYTLHWQQELDGENVQDRREWEAHERRLETAGTHRLTGIKPGSGYSLWVQAHATPTKSNSSERLHVRSFAELPELQLLELGPYSLSLTWAGTPDPLGSLQLECRSSAEQLRRNVAGNHTKMVVEPLQPRTRYQCRLLLGYAATPGAPLYHGTAEVYETLGDAPSQPGKPQLEHIAEEVFRVTWTAARGNGAPIALYNLEALQARSDIRRRRRRRRRNSGGSLEQLPWAEEPVVVEDQWLDFCNTTELSCIVKSLHSSRLLLFRVRARSLEHGWGPYSEESERVAEPFVSPEKRGSLVLAIIAPAAIVSSCVLALVLVRKVQKRRLRAKKLLQQSRPSIWSNLSTLQTQQQLMAVRNRAFSTTLSDADIALLPQINWSQLKLLRFLGSGAFGEVYEGQLKTEDSEEPQRVAIKSLRKGASEFAELLQEAQLMSNFKHENIVCLVGICFDTESISLIMEHMEAGDLLSYLRAARATSTQEPQPTAGLSLSELLAMCIDVANGCSYLEDMHFVHRDLACRNCLVTESTGSTDRRRTVKIGDFGLARDIYKSDYYRKEGEGLLPVRWMSPESLVDGLFTTQSDVWAFGVLCWEILTLGQQPYAARNNFEVLAHVKEGGRLQQPPMCTEKLYSLLLLCWRTDPWERPSFRRCYNTLHAISTDLRRTQMASATADTVVSCSRPEFKVRFDGQPLEEHREHNERPEDENLTLREVPLKDKQLYANEGVSRL\n>P04808\nMPRLFLFHLLEFCLLLNQFSRAVAAKWKDDVIKLCGRELVRAQIAICGMSTWSKRSLSQEDAPQTPRPVAEIVPSFINKDTETIIIMLEFIANLPPELKAALSERQPSLPELQQYVPALKDSNLSFEEFKKLIRNRQSEAADSNPSELKYLGLDTHSQKKRRPYVALFEKCCLIGCTKRSLAKYC\n>Q04721\nMPALRPALLWALLALWLCCAAPAHALQCRDGYEPCVNEGMCVTYHNGTGYCKCPEGFLGEYCQHRDPCEKNRCQNGGTCVAQAMLGKATCRCASGFTGEDCQYSTSHPCFVSRPCLNGGTCHMLSRDTYECTCQVGFTGKECQWTDACLSHPCANGSTCTTVANQFSCKCLTGFTGQKCETDVNECDIPGHCQHGGTCLNLPGSYQCQCPQGFTGQYCDSLYVPCAPSPCVNGGTCRQTGDFTFECNCLPGFEGSTCERNIDDCPNHRCQNGGVCVDGVNTYNCRCPPQWTGQFCTEDVDECLLQPNACQNGGTCANRNGGYGCVCVNGWSGDDCSENIDDCAFASCTPGSTCIDRVASFSCMCPEGKAGLLCHLDDACISNPCHKGALCDTNPLNGQYICTCPQGYKGADCTEDVDECAMANSNPCEHAGKCVNTDGAFHCECLKGYAGPRCEMDINECHSDPCQNDATCLDKIGGFTCLCMPGFKGVHCELEINECQSNPCVNNGQCVDKVNRFQCLCPPGFTGPVCQIDIDDCSSTPCLNGAKCIDHPNGYECQCATGFTGVLCEENIDNCDPDPCHHGQCQDGIDSYTCICNPGYMGAICSDQIDECYSSPCLNDGRCIDLVNGYQCNCQPGTSGVNCEINFDDCASNPCIHGICMDGINRYSCVCSPGFTGQRCNIDIDECASNPCRKGATCINGVNGFRCICPEGPHHPSCYSQVNECLSNPCIHGNCTGGLSGYKCLCDAGWVGINCEVDKNECLSNPCQNGGTCDNLVNGYRCTCKKGFKGYNCQVNIDECASNPCLNQGTCFDDISGYTCHCVLPYTGKNCQTVLAPCSPNPCENAAVCKESPNFESYTCLCAPGWQGQRCTIDIDECISKPCMNHGLCHNTQGSYMCECPPGFSGMDCEEDIDDCLANPCQNGGSCMDGVNTFSCLCLPGFTGDKCQTDMNECLSEPCKNGGTCSDYVNSYTCKCQAGFDGVHCENNINECTESSCFNGGTCVDGINSFSCLCPVGFTGSFCLHEINECSSHPCLNEGTCVDGLGTYRCSCPLGYTGKNCQTLVNLCSRSPCKNKGTCVQKKAESQCLCPSGWAGAYCDVPNVSCDIAASRRGVLVEHLCQHSGVCINAGNTHYCQCPLGYTGSYCEEQLDECASNPCQHGATCSDFIGGYRCECVPGYQGVNCEYEVDECQNQPCQNGGTCIDLVNHFKCSCPPGTRGLLCEENIDDCARGPHCLNGGQCMDRIGGYSCRCLPGFAGERCEGDINECLSNPCSSEGSLDCIQLTNDYLCVCRSAFTGRHCETFVDVCPQMPCLNGGTCAVASNMPDGFICRCPPGFSGARCQSSCGQVKCRKGEQCVHTASGPRCFCPSPRDCESGCASSPCQHGGSCHPQRQPPYYSCQCAPPFSGSRCELYTAPPSTPPATCLSQYCADKARDGVCDEACNSHACQWDGGDCSLTMENPWANCSSPLPCWDYINNQCDELCNTVECLFDNFECQGNSKTCKYDKYCADHFKDNHCDQGCNSEECGWDGLDCAADQPENLAEGTLVIVVLMPPEQLLQDARSFLRALGTLLHTNLRIKRDSQGELMVYPYYGEKSAAMKKQRMTRRSLPGEQEQEVAGSKVFLEIDNRQCVQDSDHCFKNTDAAAALLASHAIQGTLSYPLVSVVSESLTPERTQLLYLLAVAVVIILFIILLGVIMAKRKRKHGSLWLPEGFTLRRDASNHKRREPVGQDAVGLKNLSVQVSEANLIGTGTSEHWVDDEGPQPKKVKAEDEALLSEEDDPIDRRPWTQQHLEAADIRRTPSLALTPPQAEQEVDVLDVNVRGPDGCTPLMLASLRGGSSDLSDEDEDAEDSSANIITDLVYQGASLQAQTDRTGEMALHLAARYSRADAAKRLLDAGADANAQDNMGRCPLHAAVAADAQGVFQILIRNRVTDLDARMNDGTTPLILAARLAVEGMVAELINCQADVNAVDDHGKSALHWAAAVNNVEATLLLLKNGANRDMQDNKEETPLFLAAREGSYEAAKILLDHFANRDITDHMDRLPRDVARDRMHHDIVRLLDEYNVTPSPPGTVLTSALSPVICGPNRSFLSLKHTPMGKKSRRPSAKSTMPTSLPNLAKEAKDAKGSRRKKSLSEKVQLSESSVTLSPVDSLESPHTYVSDTTSSPMITSPGILQASPNPMLATAAPPAPVHAQHALSFSNLHEMQPLAHGASTVLPSVSQLLSHHHIVSPGSGSAGSLSRLHPVPVPADWMNRMEVNETQYNEMFGMVLAPAEGTHPGIAPQSRPPEGKHITTPREPLPPIVTFQLIPKGSIAQPAGAPQPQSTCPPAVAGPLPTMYQIPEMARLPSVAFPTAMMPQQDGQVAQTILPAYHPFPASVGKYPTPPSQHSYASSNAAERTPSHSGHLQGEHPYLTPSPESPDQWSSSSPHSASDWSDVTTSPTPGGAGGGQRGPGTHMSEPPHNNMQVYA\n>P07949\nMAKATSGAAGLRLLLLLLLPLLGKVALGLYFSRDAYWEKLYVDQAAGTPLLYVHALRDAPEEVPSFRLGQHLYGTYRTRLHENNWICIQEDTGLLYLNRSLDHSSWEKLSVRNRGFPLLTVYLKVFLSPTSLREGECQWPGCARVYFSFFNTSFPACSSLKPRELCFPETRPSFRIRENRPPGTFHQFRLLPVQFLCPNISVAYRLLEGEGLPFRCAPDSLEVSTRWALDREQREKYELVAVCTVHAGAREEVVMVPFPVTVYDEDDSAPTFPAGVDTASAVVEFKRKEDTVVATLRVFDADVVPASGELVRRYTSTLLPGDTWAQQTFRVEHWPNETSVQANGSFVRATVHDYRLVLNRNLSISENRTMQLAVLVNDSDFQGPGAGVLLLHFNVSVLPVSLHLPSTYSLSVSRRARRFAQIGKVCVENCQAFSGINVQYKLHSSGANCSTLGVVTSAEDTSGILFVNDTKALRRPKCAELHYMVVATDQQTSRQAQAQLLVTVEGSYVAEEAGCPLSCAVSKRRLECEECGGLGSPTGRCEWRQGDGKGITRNFSTCSPSTKTCPDGHCDVVETQDINICPQDCLRGSIVGGHEPGEPRGIKAGYGTCNCFPEEEKCFCEPEDIQDPLCDELCRTVIAAAVLFSFIVSVLLSAFCIHCYHKFAHKPPISSAEMTFRRPAQAFPVSYSSSGARRPSLDSMENQVSVDAFKILEDPKWEFPRKNLVLGKTLGEGEFGKVVKATAFHLKGRAGYTTVAVKMLKENASPSELRDLLSEFNVLKQVNHPHVIKLYGACSQDGPLLLIVEYAKYGSLRGFLRESRKVGPGYLGSGGSRNSSSLDHPDERALTMGDLISFAWQISQGMQYLAEMKLVHRDLAARNILVAEGRKMKISDFGLSRDVYEEDSYVKRSQGRIPVKWMAIESLFDHIYTTQSDVWSFGVLLWEIVTLGGNPYPGIPPERLFNLLKTGHRMERPDNCSEEMYRLMLQCWKQEPDKRPVFADISKDLEKMMVKRRDYLDLAASTPSDSLIYDDGLSEEETPLVDCNNAPLPRALPSTWIENKLYGMSDPNWPGESPVPLTRADGTNTGFPRYPNDSVYANWMLSPSAAKLMDTFDS\n>P35498\nMEQTVLVPPGPDSFNFFTRESLAAIERRIAEEKAKNPKPDKKDDDENGPKPNSDLEAGKNLPFIYGDIPPEMVSEPLEDLDPYYINKKTFIVLNKGKAIFRFSATSALYILTPFNPLRKIAIKILVHSLFSMLIMCTILTNCVFMTMSNPPDWTKNVEYTFTGIYTFESLIKIIARGFCLEDFTFLRDPWNWLDFTVITFAYVTEFVDLGNVSALRTFRVLRALKTISVIPGLKTIVGALIQSVKKLSDVMILTVFCLSVFALIGLQLFMGNLRNKCIQWPPTNASLEEHSIEKNITVNYNGTLINETVFEFDWKSYIQDSRYHYFLEGFLDALLCGNSSDAGQCPEGYMCVKAGRNPNYGYTSFDTFSWAFLSLFRLMTQDFWENLYQLTLRAAGKTYMIFFVLVIFLGSFYLINLILAVVAMAYEEQNQATLEEAEQKEAEFQQMIEQLKKQQEAAQQAATATASEHSREPSAAGRLSDSSSEASKLSSKSAKERRNRRKKRKQKEQSGGEEKDEDEFQKSESEDSIRRKGFRFSIEGNRLTYEKRYSSPHQSLLSIRGSLFSPRRNSRTSLFSFRGRAKDVGSENDFADDEHSTFEDNESRRDSLFVPRRHGERRNSNLSQTSRSSRMLAVFPANGKMHSTVDCNGVVSLVGGPSVPTSPVGQLLPEVIIDKPATDDNGTTTETEMRKRRSSSFHVSMDFLEDPSQRQRAMSIASILTNTVEELEESRQKCPPCWYKFSNIFLIWDCSPYWLKVKHVVNLVVMDPFVDLAITICIVLNTLFMAMEHYPMTDHFNNVLTVGNLVFTGIFTAEMFLKIIAMDPYYYFQEGWNIFDGFIVTLSLVELGLANVEGLSVLRSFRLLRVFKLAKSWPTLNMLIKIIGNSVGALGNLTLVLAIIVFIFAVVGMQLFGKSYKDCVCKIASDCQLPRWHMNDFFHSFLIVFRVLCGEWIETMWDCMEVAGQAMCLTVFMMVMVIGNLVVLNLFLALLLSSFSADNLAATDDDNEMNNLQIAVDRMHKGVAYVKRKIYEFIQQSFIRKQKILDEIKPLDDLNNKKDSCMSNHTAEIGKDLDYLKDVNGTTSGIGTGSSVEKYIIDESDYMSFINNPSLTVTVPIAVGESDFENLNTEDFSSESDLEESKEKLNESSSSSEGSTVDIGAPVEEQPVVEPEETLEPEACFTEGCVQRFKCCQINVEEGRGKQWWNLRRTCFRIVEHNWFETFIVFMILLSSGALAFEDIYIDQRKTIKTMLEYADKVFTYIFILEMLLKWVAYGYQTYFTNAWCWLDFLIVDVSLVSLTANALGYSELGAIKSLRTLRALRPLRALSRFEGMRVVVNALLGAIPSIMNVLLVCLIFWLIFSIMGVNLFAGKFYHCINTTTGDRFDIEDVNNHTDCLKLIERNETARWKNVKVNFDNVGFGYLSLLQVATFKGWMDIMYAAVDSRNVELQPKYEESLYMYLYFVIFIIFGSFFTLNLFIGVIIDNFNQQKKKFGGQDIFMTEEQKKYYNAMKKLGSKKPQKPIPRPGNKFQGMVFDFVTRQVFDISIMILICLNMVTMMVETDDQSEYVTTILSRINLVFIVLFTGECVLKLISLRHYYFTIGWNIFDFVVVILSIVGMFLAELIEKYFVSPTLFRVIRLARIGRILRLIKGAKGIRTLLFALMMSLPALFNIGLLLFLVMFIYAIFGMSNFAYVKREVGIDDMFNFETFGNSMICLFQITTSAGWDGLLAPILNSKPPDCDPNKVNPGSSVKGDCGNPSVGIFFFVSYIIISFLVVVNMYIAVILENFSVATEESAEPLSEDDFEMFYEVWEKFDPDATQFMEFEKLSQFAAALEPPLNLPQPNKLQLIAMDLPMVSGDRIHCLDILFAFTKRVLGESGEMDALRIQMEERFMASNPSKVSYQPITTTLKRKQEEVSAVIIQRAYRRHLLKRTVKQASFTYNKNKIKGGANLLIKEDMIIDRINENSITEKTDLTMSTAACPPSYDRVTKPIVEKHEQEGKDEKAKGK\n>P00533\nMRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFEDHFLSLQRMFNNCEVVLGNLEITYVQRNYDLSFLKTIQEVAGYVLIALNTVERIPLENLQIIRGNMYYENSYALAVLSNYDANKTGLKELPMRNLQEILHGAVRFSNNPALCNVESIQWRDIVSSDFLSNMSMDFQNHLGSCQKCDPSCPNGSCWGAGEENCQKLTKIICAQQCSGRCRGKSPSDCCHNQCAAGCTGPRESDCLVCRKFRDEATCKDTCPPLMLYNPTTYQMDVNPEGKYSFGATCVKKCPRNYVVTDHGSCVRACGADSYEMEEDGVRKCKKCEGPCRKVCNGIGIGEFKDSLSINATNIKHFKNCTSISGDLHILPVAFRGDSFTHTPPLDPQELDILKTVKEITGFLLIQAWPENRTDLHAFENLEIIRGRTKQHGQFSLAVVSLNITSLGLRSLKEISDGDVIISGNKNLCYANTINWKKLFGTSGQKTKIISNRGENSCKATGQVCHALCSPEGCWGPEPRDCVSCRNVSRGRECVDKCNLLEGEPREFVENSECIQCHPECLPQAMNITCTGRGPDNCIQCAHYIDGPHCVKTCPAGVMGENNTLVWKYADAGHVCHLCHPNCTYGCTGPGLEGCPTNGPKIPSIATGMVGALLLLLVVALGIGLFMRRRHIVRKRTLRRLLQERELVEPLTPSGEAPNQALLRILKETEFKKIKVLGSGAFGTVYKGLWIPEGEKVKIPVAIKELREATSPKANKEILDEAYVMASVDNPHVCRLLGICLTSTVQLITQLMPFGCLLDYVREHKDNIGSQYLLNWCVQIAKGMNYLEDRRLVHRDLAARNVLVKTPQHVKITDFGLAKLLGAEEKEYHAEGGKVPIKWMALESILHRIYTHQSDVWSYGVTVWELMTFGSKPYDGIPASEISSILEKGERLPQPPICTIDVYMIMVKCWMIDADSRPKFRELIIEFSKMARDPQRYLVIQGDERMHLPSPTDSNFYRALMDEEDMDDVVDADEYLIPQQGFFSSPSTSRTPLLSSLSATSNNSTVACIDRNGLQSCPIKEDSFLQRYSSDPTGALTEDSIDDTFLPVPEYINQSVPKRPAGSVQNPVYHNQPLNPAPSRDPHYQDPHSTAVGNPEYLNTVQPTCVNSTFDSPAHWAQKGSHQISLDNPDYQQDFFPKEAKPNGIFKGSTAENAEYLRVAPQSSEFIGA\n>P00441\nMATKAVCVLKGDGPVQGIINFEQKESNGPVKVWGSIKGLTEGLHGFHVHEFGDNTAGCTSAGPHFNPLSRKHGGPKDEERHVGDLGNVTADKDGVADVSIEDSVISLSGDHCIIGRTLVVHEKADDLGKGGNEESTKTGNAGSRLACGVIGIAQ\n>P01375\nMSTESMIRDVELAEEALPKKTGGPQGSRRCLFLSLFSFLIVAGATTLFCLLHFGVIGPQREEFPRDLSLISPLAQAVRSSSRTPSDKPVAHVVANPQAEGQLQWLNRRANALLANGVELRDNQLVVPSEGLYLIYSQVLFKGQGCPSTHVLLTHTISRIAVSYQTKVNLLSAIKSPCQRETPEGAEAKPWYEPIYLGGVFQLEKGDRLSAEINRPDYLDFAESGQVYFGIIAL\n>P0A7L0\nMAKLTKRMRVIREKVDATKQYDINEAIALLKELATAKFVESVDVAVNLGIDARKSDQNVRGATVLPHGTGRSVRVAVFTQGANAEAAKAAGAELVGMEDLADQIKKGEMNFDVVIASPDAMRVVGQLGQVLGPRGLMPNPKVGTVTPNVAEAVKNAKAGQVRYRNDKNGIIHTTIGKVDFDADKLKENLEALLVALKKAKPTQAKGVYIKKVSISTTMGAGVAVDQAGLSASVN\n'
EMBEDDED_BENCHMARK_3DI_FASTA = '>P01308\ndvpvvvvvvvvvcccvvpdpppppddddddpddppvvvvvcvlvppddppdpdddddddddddddddddddddddddddddpppppppddpppvpsvdpddpvvvvvpsd\n>P02798\ndppqfpadpplpdqclqvtnglddprpsrqddnaladdrpqvvcpvhdpdpdrdnhdpvrd\n>P62944\nddlvvllpddddpllvvllvqlpdpdlvsvlvslssvvscvvnvhdnvvclvsllvqcpdpdpssvvsslvvllvccvvpvvsvvvslvsllvqcvdpdlssvlssllsllssldpvslvsslvslvcqcvdpdlssnlssllsllsscvnpvpvsvvscvlvslvvqlvdpalsnvlsslvsqvssclvpvvppsppddpvslvsllvscvvddlssvlssllvllpdddpalvrllvslvsllvqlpppdllsnqssllslvlslqrhdcpdpssvvslqsnqvsllvqcvdplvsnllslllvllvclvpvvsclpvlvslqddpprdpssvvsslsvnlssddpvcvvvllvsllvqcpdpdlvsnlsslvsllsscqvyvvcvvvslvsllvqvvvvdlssllssllsvllvclqpppppvvclvvnvvnpvsdddlssllsnllvlllpvvpdvcsvvvlvvvlvclvvhdpsnlqssllsllsnclnvvvprpvvnvsslccqqpvdpdpvsnvssvvsvccvvpdsvvsccvsnprddrdhspppdddpflsqqlssvgshvcssvshhcvvpdppspnpdrdgrdhhpdpddpdddddddddddddddddddpdppppvvvvvppppdddddddddddddddddppppddppvppdddddddddddddddddddddddddppppppdddpppddpaadwddkdwfddcvqpvqwtkiwtwgadpnwiktkikifgqhqdwkadkfkfkdaaaqqktfpdtfpppdthhhgrmdititiidrnhhhdqdvvrqkiwmwmdippgidigidghqqlnqwglpfddapvvlvvvvvvfdcvlkdkdkqaqfdddpvvlqvscvssqkhfndwdqdpqkiwtwiwiagnvgwikiwiwihhpphrmimiiigtppsscvvsvvvnssssrvd\n>P01542\ndkfaqdpvlvvqlvvcppvvpdsvvscvvsvidddpdddddpsrhd\n>P43408\ndqaaeeeeaeppqlcrvvlvvvlqvvvvvvvahededellvllvvqcvvvvndddsvslvvddpvsslvsllvslqvqlvvsrphhyyyywhcwdqdpvgthggqdlsscvnnlhqayeyedeqlvssqvsvvvdpvdddddddsvnnvvssvvsvvsrvvscvsrvhyyhyaypdppcsvvssvvvnvvnd\n>P0AFL6\ndpppdppdpfakakewepfqfkikiwiwgddpndidtddidigtlnqnvqqdpvragdpsslvsvlvvllvvqvvcpphdllryaaeyedslvsrpvvvvsqvssvvrhvhhyfyfdqqqqlllfvlqqvvfdpdawweweweagaqwiwtfidhpsdtddtdifrqhlnvllvvqvpvqfdaprsllvllvvllvscqvpqvvdqvvdipayeyeddllvqllqllcqvpvvqskdflvslvvvsvqqnvdgglcrddthrhdnscssrnsssssnvnsvcvsnvdgiyhygptgnrssvncvsvcpvvpvpqlvvllvvlcvvlvppqlllvqllvqlvvqvvlvcvvcvvlddvlvnvllnslsnqlqsqvvvddvcrlvsslvclqpddrhpdgnvssnlssvlsnllddaddpvprdddpvhdpqnsvvssvssnvsslqcvvpvprddfppfhwdddrqaietegepcpcvvvvvsvvsvvvvqvnqvvdpphgyhyyyddddpppd\n>P0ACJ8\nddppdpdddvlvvvvlvqwdkdkdaafdwpfdfqawqwkkkawqaakkwwwdadpvrqiatldmdghgdmpscpclvdppdtrhtiiggnhitimtmdtsvvvvvvcvvpvvvvvsnvvvvvvsvvsnvvlvvclvppdllsllvvvvlvqcpdpqwdddpfatkdfaalsnscshnvhdsvvsvvsqvvcvvvvqwdddhriigggpyd\n>P0A988\ndkakdwlvlvqvqlclqqvllpddqpdpllqkwkwwdddqktwtwhdnvqkikifigggptdddtdiwifrsvvvnvvsvpddhrwmwiwdddpqwiwidtdpdiditgiggpvprddddddafpwkdkffllvlllqllqalvaqddpdpvqlsqwwkwwaaaqktkiwhhpvqkikifidggpdgidtdmftfgsslsvssnvvrpngrqmwmwgdddfkiwidgdrmimmgttdpdddppcvvqddddqpwkkkdflvvvlvllvqqqvaaappqqkwkwwddqqkiwikhhypvgdididmdgihtdddiamfmfnsvlvnsnsvsdpegmkmwrddhrqgwiwidhppdprmimiggtddd\n>P0A7Y4\ndffekekekakddpdqfakwkiwmwidgppdididmaiagtdgrlvrrlvrllvrlvvdpaehayeyedqdpsvvccvppnvvvcvvvvqadpvrdhddpnvssvsnvvspvrynydydhdnacpppvnnvvrvvrtvvnsvphdhhppsddppd\n>Q57733\ndddpdpvvvvvvvvvcvvvpddqpdwdfdddplqktktkdddkdwdwdddqfkikikifapqfdpvqwdwddffqktkikgahdaddddppdddpddpddrdgiimmimrhsatfdrvpkdwdddnrmimiittgdpvrhddddddd\n>P22579\ndddddddddddddddddddddddddddddddddddddddddddddddpdddppddpddddddddddddddddddddddddddddddddddddddddddddddddddddyddddddddddddddddddddddpddddpdpppdddddddddddddddddddddddddddddddddddddddddddddddddddyddddddddddddddddddddddlddddpvlvvvllvvvcvvcvvpcvlsvqlvvlvvccvvvvddpvvslvsqcvscqvpqvssqscqnvddpqwhwdddpdsqawiwtghpvgidtrhpddddddddddddddddddddddddddddddddddddddddddddddddddddddddddpdddddddddddddddddddddddddddddddddddpvlvvvllvvvcvvcvvpvvlnvvlvvlvvccvvpvpdvvvslvsvcvscvvpvvssvsvvsvdddpppvvvvcvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvpdddddddddddddddddddddddddddddddpdddpddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddpppppvppdppddpdddddddddpcvvvvvllvvlcvqlvdpvvnvvlvvlvvcvvvvnadpvrslvvccvrpvvpvvssvvvcvvsvdddddddpppppppdpdddqvvfdddpqwkgfddpvqlpdddppddpvnvvpddsgidtdhppddvvvpppdpdfdplrvllvvlvvvvvslvlllvllvvllvllvvvqvvlvpddpvclqvdfadqpspdpdnvsqlvnlcsqvndvvsvvlvvccrnrvspssvvvsvssvvvsvvsvvvsvvsvvvslvscqvrvlvrlappqvvvvvvvvvcldlvnlvvqqvvlvvvlvvqvvdvvrdnddfsdkdfdpdplvlllllvllllllvqdppddplvsvllsvlslqvcclvlvddsvvsvvsnvvvvvvvpppdddddddddpdddddppdddhslcslppvsvvvvvvvvpddddddpdddddpddpvlvvvlvvlvvllvpfpddddllvvlfppfwdfqdqkakqwdwslvsslvvlvslssvllvqvqvcqvvllvvvvpddddvvcvvvvshdcvcvvvvladdsdrvsvslsvlsscvsnvvddlvsnvvscsvrvpnrnsscscvnvssvsssvsssvcsvdplssvsvvlsvvrspdsidgllvlvssvvsnpvsddppiwikiwmagnvrrmtgihtddpqarsddqdpdlvnvvssqsnqnsrpddghnddpvpdddpddpvvcvvpppddddddppvpdqawdwdffwdwdadspprdtdtdppgdidtggpdpddppppcpvvvvvvvvvvvvvvvccqqpppphpvvpddpvvvvvvvvvvvvvvvvvvvvpddddddddydddddddddddddddddddddddddddddddddddddddddddddddddddd\n>P15378\ndlvvllqvllllllllllllvavvvvhdsppdadadppprdhddpccsrapvncvvlvqadppprhghdpvnnvssnvssvqlsllcvqqppdllsqlsnlllsllssqlsncqvpvaadvvsvvvsvvsnvvscvsvpdpdncqlvqqlvcllvvvvvvcvvvvrpddpnssnvssnlcnnqgnqlsvqlvvqlcvqqvvvvvvcvvvvvdddpdddrssnsssvssscsvrcvvvvvvvvvvvvvd\n>P34690\ndaaeaeeqkdqqslllvvllvlqlcvlvcqallqarpdpppqpdplccnqwddpvprgtagsyayefqacpslvcccpdrnvnrydplsdyyhhdhllqfllclaqnslvvcqvsnvvsnvvsvvvdpdyqayeyeaelrgngrqnsvlnnlvvccvvpvrhayeyeyayddppdhpdlcsllsnllsllsclqrhlayeyfypvllqlvccvalvpvdddvnqssnvssllvcllcllsradwpqradpvnlsvqappdsnlrywwkfkpdfhfqvcvvpdddfplvqlvvrqdcsrirdpfnllvkawaakekrkeapddpvsnvvsvvvvvpdpshhhapvcpprydyiygrhfrdddvphgghrdrmmimmitrilrclvslvsslvslcvcvvvvncvcvsvvsvddpvssvvssvvsvvsnvvsvvsrdhpvvvppppdddd\n>P0A7U3\ndaddpvvdfdddpvqvvvlvvclvvvdqdaaedqqqrhfdapscaqhwyqydlppggdtdhdhpvrggpgnnvvrhndddpddpppppdddd\n>P63165\nddddddddddddpddddppfwaweweaepvrdtdididgqqdfclvvlvvvcvvvvhdsvqkfkdfpndtgdrrdgcnvvvhhhhgyiyidgppppdpppd\n>P0A7N4\nddddpddddpvrvcvvcppvddddlpqwdaapqprdidgvpdadpcqddnndrnddd\n>P24311\ndvvvvvvvvvvvvvvvvvvvvvvvvvpppdqdpcsvcvvvcvvvvvvvvvvvvvcvvppppddpvddcvvppdddppddd\n>P56252\ndwddkawdwdfflvrftaikmwtddplgtfidtafdfqqaqpqfqfadfqqpcvprrrrhdvqlrcllhppltvvlvvvvdallplvvslvsvcvsqpdprspsnglrslanvqlsslrvscsvvvhdslqsllvllvhpawaaafewafaeadpsqapaqfqfggktktqlldqalvvslvlqvqlsvlllvllcvvqnnvqqdatlqrhtyhrdldpvvvqvssvvscvvsvnvvrmfmatersqqsqddpdqwgasrvvddpdpcpridglvvllvvvvvcvvvgryqyyecsdhlpplvsllvslvvdpgfyeypsvcllppvslvvclvsvsgqeyeqdcssnrhsssssvslvssvvsngayedeghsgadldqsslsncrssnrrhyyqyhvddnssvsnsvvvvvvcvvcdpvhhhqrpvngpsd\n>P18253\ndfwkkffkdwqnhtlgmwikdwpcvqqvqvrvqqqcqqacvvvhhqafwfqafaaaqfktktfqpppnhppaaatpvgqwafdrdqpdalaaffwkkfdapwgrggggmiigthgrdrvcgptidttigtpddvvsrvvqnvqaynvrggvtgmgrnhmhtd\n>A0A6G0XC32\nddfdaedepacccpdqedadagarhqyyhhanvqnhaadaynyqnhqyyepenhanhqlvrhlvnhdhqnhayyehalhhhpdddplvsllvnlqshqnhqhyehalpqaelssvvsclpsnpnhqeyeyedapprpphhhhddpvsvvvscvvvvshdydyd\n>P42212\ndaplvvlqqakfkeweweweaavnptwikikidiahlqqqkdwmkiftppfddqwapqvcpcvtcvqvsvsahedplfplapqqsnlpqqgkwkwkwkdwvpakiktkiwtwhdddrytyiymyiyidrgdccdctsvvffdqwfdkdkkkwafdvvqlwiwiwdwgwtqgnvrhttiitmtmimhrsdpdhgrghhiwiktkdkgwdddppdpgdmiiimmgiyiddddsvnsvvvd\n>P61626\ndddddpppppppppdadwdqddllrvlvvcvvlpvacqvhdhsllvlqqlcqqpvstqqdkdadpvqrwmqggsnrhiqqedapcpgrpngnnvlvyhssvsshnpnnsvssslsvvlvdpchvvvrprsvvgpppdpsvvscppspd\n>P68871\ndqdddpvllclqvvlvvvdpllllllqlqlvlcvvpvvvcvlvvvqddppdsccssvrpvssvvrsvvsvllvvcsvvvvpllvscvvllcccappsvddlvsllvslvsslvsscvvcpvscdpsnsvssnsssvsnscssnvphd\n>P37840\ndvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvcvvvvvvvvvvvvvvvvvvvvvvvvvcvvvvpppvpdpvppppdpdppppppdpppppppppppppppppppppdppd\n>P02686\nddddddddddddddddddddddddddddddddddddddddddddppddppdddppdddddddddddddddddpppdpddpdppdppddpppvlvpdppnppcpvppvpppdppppppdppppppddddddddddddddddddddddpddddpddddpddddddpppdpqalvnlvccvvvppcpdpppdddddddddddddddddddddddddpppddsvrvnvccvvpvddpdpppdddddddddddddddddddpddddddddddddddddddddddppdpvcvvpvpddddpddddddddd\n>P61769\ndppvvvvvvvvvvvvvvppddfwdwdwdwewpdqadaqdktkikikiwggpdpdkdkfkdkpndgdppkdkdpwdhdpvriiimmimdmdgddppikiwmwmddpvdpdididtydsvd\n>P0ABH7\ndapdwdwdddpnddtdtwhwdddpdddidtdppcvvvvvdddddvpcprddpdddqawddqqavqfidglnhtlvcclpphwlqqslccrqpsdgddpvrsvvslvlllvqqdddvvlvvvlvpddlpdqllvqvlvslqcvcvvpvvlffpldpvslvvllsnllrhllqsllssqcsnvvadfdgfdspddpqqrslcrnddhppddddrdpllssllrsvlsllqadppdqllvqllqcllvsggnsnslssssvrcppcvhpllllvqvvvlvvcvapvsvvvlvvlcpdpphprhnrqwddprhqaanssvvslvvslvslcvvlvdpdrsvvvlvvvqvccvpppvnnvvrghghnsssqssscvssvhdsssrssssssshssvssssssvsvvvvphndddddddpdddddddddpddd\n>P0A9Q7\ndafddlvvvvvllvllqvllvvlqpddlvllllllllllvlllvcqqvllvllcvqqvffdsvlssvlsnllrpqqcvvpspddaaawpdddvpqfktkgktfleeeeeeaeslnqqsslsnvlslcssrsyayeyqydqsrlvsnvvsqvssqvssvvspgrnsryhyrnhhdpvsslsvlqpprhaeyeadyapvrqvsnvvnphhyfaafaaqaeeeeapafplllvlvlllvqclrrqqlqrqrhleyeyepvcqvvnvvslvvpqedeaddlllvllqpqcddpldgnsvqardaaqvssvssphgddssrrayeyadadcasvgsslerrngryyyyyyahhlvrslvsslssclhgraqaeyeyghpcvvpvvsvvvsvvrhrylyyyyshnrsqvrqddppdnpqrhapgagrpvsrpgpdsggdgnvvrmdmdmdgdfdaddadqdwaqaeaaeapcqlvvcvcclvvpwaeeeefeapvcvvvcvvvvnvvscvvsnrhydydhhfdpqgapvslvvllvvcqvrqgqayeqeeaqrsllsslssqlcnvcvpddlvqqldadldlppgpddrdqgnprhayeyeylalarlsqiaqwhwhahpvprdirigghsssggnyyylhlvshqppdllrlllrllnllqlllflqqwlqndpvlvvlslqlnllslvqslvcnpcrnvcsvssrsssvsnnssssscsrlngalqqlllvlccsqpvddsslssllqvqqllqqlldqdfpafqdhsslpgrcslvsqlvslvsnvqddppddsvssnvsvsvssvvscvssprdlavvsvpddpvscvvcllvslvsslvdnrhsgrrhnddslsssqsnvcrhvshhddpppspppddpddddddpdpdddd\n>P0A953\ndfwkffqfkfkfflqdrglvsnlvclqvvafskdfapqcvvvvfldriasegpddlppvddpvlclqadpfqsqaqrgvvrrcvrqvpdlvrfapdqqeeeeewfqqqdlplvvqlvvllvdpvhpvsnddrsclrrdrlvnqvsrcvvsrgnhytgfahdwqcgqlvqvvvqrvclvvvshfkyktftwfddhsspvvvcsvvqqafrpcsvprllqfffqappqrhfhyytgimitimgrvvvcvvvvgdtlwifqfkfkffpqpaqfaaplvrqlrrvvrrcvpppddaqeeqeqrrshprhslsplvsqcvnqplnaheyahccsrgtrntnrnnrvsvssvsscqvqqkggasenrpdgdpscpshnhhhhigrhhgakykywtargsriimmiiigdddd\n>P00698\nddddppppppppppqqdkdqddllrvlvllvvlppacqvhdhslllsqqlcqqpvstqqdkdqdplrwmfggsnrhiqledapcpgrpngnnplvyhssvsshnnnnsvssslsvvcvppchvvvrpssvvppppdpsvvsnpphpd\n>P0DP27\ndvvvddpvrlvvlvvvvcqqvvvppqkhalvsvqvvcvvvvhhddsvrsqvvqvvqvpvppriagsvssvvvvvvvvlcvvvvvvlvvvqcqqcvvppqwhalvsvqvvcvvvvhhddsvvsqvvqvvqcpvpprihgsvssscvvnvd\n>P62937\ndawwkkkffkdwqnhtlgmwiktfpcvqqvqvsvqqqcqqacvvvhhqafwfqffaaaqfktktfqpppnhppaaatpvgrwafqrdqpdaqaffffkwfdapwgrggggmiitthtrdrvcrptirttigtpddsvsrvvqnvqahnrgggpttmgrnhmhtdd\n>P10599\ndeaedqepvsvvvvlqvlaqakekewaaappdpqqvvqvvvlvvvcvvcvsyyhyyydcvryvvvcvvvvpddpgkmfmdgssdtpdidhtndsvvvvvvvvvsd\n>P56201\nddddddddddpddppppppdqdqddqdqweeeeeeeccldpcnlppvlqlvllvcvvcppvsydyayeyeyqddqvvqlvsnvvslvphdddppddpvvvvvssvvnnvsyhyaydddlvslqvvqvvvvcvcvvvvidaqeyeyeppddlvcclvvlvsclvhrgddpnhaaayededdqaqflvsnvvscvsnvvrddllryfyddllllqpllvcllvqcvqcvvqpvvqlfpvwfqakewefeaqadcapvlvvclsrfpcsvdvlgnsllsvlssqfdrfpdpqrlvrlvvrslvqlvqkdadaqqqkafadapcsqvsccvvvvddppdgrqggqkmwgwiagrdrgtvphtyiyiygyleqfgwtkmktwtdqpdddpddpvpddllpdspdifmwmqtsagdpvratkikgfnvrdqtdddcqfkdfddfdppdddpnhgsvrirmigtpdddrssssvvvcssssnrnsgdgssnssssssrcnnncvnsvpprhqyanghcsspqqpdwdddfsgidrpddrdpppdpdddddddpvndawdwddflnaieiehallslllvvlvvvqvlqvvccvppnaaeeeeeaddsclsnqqcnlrghpvrplqryayhyqkfqqdpccdprtrnvsncvrhvvrddhdpvryyyqpqqdpnggpdvvrvsqvvsvvvcpvppvvlahqeyeweadlqqdihpqhaqdppladddrqkdwaadpdpvgimiggglvsllsyqeyeyeyedlslqvvsvvlspdaqdcnrrvnssndrpnhhyyyryycnnhvd\n>P16949\ndddqpwdwawddddpvgtdididsddddppdpppdppddppdppddpvnvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvddpppppddd\n>P68082\ndaddpvllvlqvvllvvcvvpqlvlllqlvlqlcvvpvvvqvldpprnvpddsvssvvdpvssvvrsvvsvlvvvqsvcrlvcpvslqvvlvccqppsvpalvslvsslvsslvsscvppvpscdpsssvssvvssvsvsvsscvssvvvvgpd\n>P63101\nddlvvlvvqlvvcvvvvvlvsnlvsllvnlvvqaaddpsslvsnvvslcsvlvvlvvvlvvlvvqlvvcpvvvvsnvvsvvvsvvslvvlvvslvsllvslvphqcvndpdllsnlssllvqlvsllsnllpdddpvnvvslvrsvvslvvsqvscvvrhqllalsnlvsllsvlvscvprvvnlvvslvslvvslvrcvvcvvvddpvsnvrsvvssvssvvsnvvsvvvvvvvvvvvvppddd\n>Q9R0Q7\ndeaafwawadaqfkikikrfaaqwdpwdwqddqqkiwtwtfgdpvrhihidiarflgtfdrvpkdwdddrgiimtitggpdgrddppdghndpdddprydyppvpydpnnpddpvvvvvvvvvvvvvvvvpdddddddpppddddddddddddddddddd\n>P04637\ndddddddddddddddpvvvvvvvvpddppdpdddddddddppddddpdpppppddddddddddddddddddddddddddddpddpddpdpddfffdfddqdffcfpfrkdkdadcdaldqpdqwhadpvqleiehaafdkgkikiatnddddpqkkkkkfkafpdpvgrrgaawaapvqlpdppdpvqdrstdqkdwppdpqkdwdqdprsrtimimdgddqddvvdridiiimggnhaccdprgnnvgkmkmkmfiahnvrhgrtihigiyghdsgrsvvvvlvvllvvvlpdqddddppdddddddddddddddddppdpgnndddddddpddvrvvvvvvvvvvvsvvvsvvvpddpddddpppdpdddddddddddddddddddddddd\n>P0DTC9\ndddddddddddddddddddpdpdddddddddddddddpppdpdpdqpqaaalafwkwfpdpddwdwdaqwdqfddppddlqqqwwkwakdwdwdqdpvrdidtdtitigtdtalehppnvdfapdddpridihghpphhrdhppvdtgdrvvvhdhdfgdipppidtppridtpppppddddddppddddddddddddddddddddddpdddddvvvvvvvvvvvvvvvvvvvvpddddddpdppcdpvnlvvllpddlvphfaalscfpcnrqndadddpshdqfdapqcrngtcndpcnvvvcvppddpvcqvvfwdwdwdqdpvgidididgdddddppppcsvvsvvvrvctgnnvvpdddppppppdddddddddddddddddddppdddpppcvvvvvvvvvvvvvpppdddd\n>P0AEX9\ndddydddddddddddpddppppppaaffdpqefeeedaplflqvllvvlqvvlcvvpvhhydydhdppclvvllvqllvlghgqkykdflqsqqlcvvvvffdfddddpvlvvqfdpvqqvsqddpntrfwafwfkkwkwkkfqcvlpvdqaqapvcqvvvqvvqvvvqaaaeaeallfvlqlqlqlpqqpadqfdddpndtdlldhrcldpsnlvslvsvlvccvsvsdplpghhvnrlvcvlcvrygiymdiqssvvssvvsprrmdtafdhhypnggragamtgttmtgtsshpcnvvvvcsvrprcldlsnqvsrcvsgnraqtrrpvnnvvvvpdnnsvrrvvnnvsyyhrdnhslvvqlrvlssqlssccnvvvdpnnvssvsssvssnd\n>P00918\ndpqdaacdpprrlvclcvvqvcsqpqfaddaeaevvqaaadqpfwdkdwqqpqwwwwkwaaqllwikiatdqpdlntwidggpddaiwhfgiktkfaapdqfatdqyhyvpgagrmkmwgktahcvqvdlvsllvaqqgieieiagehedaaqplcvlvlvclvqrqarpgmdtdtdrrcvsqddpdqwwkwwqhfpqdpnggsryiytygpdhdyrhpvssvsrqshashhppddgdgnhrsyhhhddnhpmhiytndd\n>P0A855\nddddddddddddddpddppdwwwawdwdfqqvpaqeeeqafaaepdpdddldrlsvllcvqssqlsrynyddpvpqqdrdndlvpddlvsqvvvvhqkykywhwyadpvqwiktkifiatspprhsdgqdididtghsvcssvssqvvnqvvccsvpvfggqqqwkkwwwfqfpddqfriftwmagprthpidtldthsfdkaaweaqlvrqktwiwdcrvvatwiwiagrvprdidtqddaphdwdqwyaqnvrqkiwfwdcnppaifiwiagpvvsdidtqdddpwhwgnweddlvsfktwtwtcpvpaifiwmdgsvdddthtladpadhwgnwyaannrqkiwtwgdhpnfiaiwiagrpprdidgqddagqwdnwearnsqqkiwtwhdddpaiwikmttnvnsiiittdipgghmhnmymhhrd\n>P00784\ndddddppvvvvvvvvvvvvvvpppvpppplqqddpvlvpdpvslvvslvvlcvvlvdddpdvvlvvllsvqlvvlvvvqvvvvvdpdqfgfhsfpcssddlvrccqqawqqadqdqddppadadpdddlvpddfdqkfflvvvvlfdaaaalgrfsllllvllqsqvqnvccvppvdrfawdsllllqpqpqadrsnhghnqsnlaclqvartagcvlppddsghdhncpvvsphhphgfnhkhwfdfldpssvqsvlsvtkkkwwffqsdscssrtregadeddadqsttgiwifrmahnfwtktaarralrdyprrihtygdpppgslgrrnrrhimmhthhd\n>P69441\ndaeeeaeaqllcvlvvqvvccvvpvaeedelvlvlvvclvvvhpsvvqcpvcvlllhdddlvsslvvvvvvcpdprnpvyhyydpppqavvrvvscvvvvnqhaeyeyrdddlvvslqsllqwwapsvvsdiagvpqraapdhqadvpprdgtdhdscsdsnssvsnvvsnvvgrvvvvvvvvvcvvvvryhydyfhsvdrpvvsnvvvvvvvd\n>P02572\ndppppfwakewedfqfwifidtfpdqgtqdifgqkkwafpdpppcvpppddridgtvrrvvvvvgtdmdgqdalqagpdpvsvlvvvvcvacprvnhqqlrheyeyegaapydlvrllvvvccscpvsvhqwyayaylqlllcvlvvdqfawefaaaqqwtkiwggdrsdtdnqlididrdhlvlllvvlqvvvvvvvdhqddpvssvvssvcqlpqaaadldlvvqvvclvvdqpskdwdqdpvrdididtssrhvslcclqvvcsvvdpdhhslvsnvsslvsddpvrslvhqchyeyaydrvqhhcpqvnsqvsncvvddvphdgdydydnvrncssvssvsplspdpvnpvlidgsvncvvppssvnvvrhd\n>P63244\ndakdkdfqdwafddpfffqekeaaplqlqwiwtfgqvqwifiwgqpsdshhridgpdiqhddpggwqykyaannrqwmwtwfqvqkifiagpvvsytpdiahdgpggwrekeaanvrqwiwtwfqsqwifidgpvrdtpdippdptdpggfqykywqndpvfiwiwtffqsqwiwiagpvvrdtpaiddddpggwndkdaapnrqkiwtffqvqwiwiagpvvrdtddiahrvagwqewdaqlpacwiwtqgfqkiwianpvvrdtsdidhdpppdpdppddhwgfrywdarnvsfwiwtitrvrmiiimgiddddd\n>P00439\nddddddddddldadqdvvrfgdpahqadadpfwwkkkkkkafadvclvvvlvvlcvvqvwdfpdwdwdqppfdngmiiiittttpvcvvsvvvsvvcccpvrvmdmaiatcvpdprhfddadfflqcllvfllvardqfldddppapcnpppvqsvvlnvlnvqqnpdgppdqrdfddddqlllqlqlvllvllvvccvpfadpllvvqpvvcvvqqvsdsrdqggqrslqvllcvqanagegegrfdghllsqllclllnyhhahaaggdsvcsqddlgcysllcvqqqssqssdhlssllsnllsqlshrfdslvsllsslvcclpqafaweddppdihdrhsvlrshsvlsvvvrdpqaaedadqsvvsspdddhspdrtsyhhyyhdsnsssvnsvvvsvpgddqfhwdadnvrsgidthndpsvvvvvvvvvvvvvvvvvvvvvvvd\n>P62593\nddddddddddddppdpppdppqqdcqlvvlqvvlcvlqvwdkwkwkaflvprdtrdtdqqpdkdacflvcllllllvqllcvqvvnfdqadfdadapvlddpqavpqvvcnppgdgllvlslccfarvgqsssvvscvsqphqvssqvvlvvvplpqagaddgppvsqagdppdshnihgqssvqsslscsqpnpsgppvsnvssqvsqqnhdplclalvvqddpqkgkrkgwghgpqgkiwmwifidhnrdggmimimiigngpddpnsssvsrnvssnsvvvsd\n>P00766\ndqaaldawddddlpqpqlfdftdqlnqqqkkfkaalvrdgqeiwgaafqfkikfflvsvddqriwiktsaffqpdpptpidtwtfpdkardppqdpvlrapgmimtttpgthdddsrhggahadalpddddfqakwkwkfqagswlpdpgtgrgmgmdiwtwddlvvlcvqvppsddpqktwtalsnaadhgshggtftwgddpngtygqwgqgawdsgrdrnhitmtggrsncvvvvvvvvvvd\n>P0A6P9\ndqawadkawdwdafpvrftaikmwtagpvrfififfafdfpdafpqfqfadwqcpcvprrrrhqvqlrccrrpvlgvqrggpgllplvvslvsqcvscvdprshnntlrssanvsrrslrrvcvvvvhdslqsllvllvrhlawaqawefafaeadppqapeqaqfgtktktqqqdfalvvslvlqvqlsvllvvlcvvvvwdqdgtllrhgythdlypvvvlvsscvssvvsvhdflnrmametenqqqsqddpqwrfrvspvrdtdhllrllvvvvvscvvgvhaeyecsdhlppqvsqlvscvppvlrhqyeycsvcllapvsvvvclvsvggqeyeqdssshshsssnsvscvvsvvsngayeyedhsgfdldqvslsnctsnvsrhyyqyhpddnrsvsnsvvsvvvcvvcppshtnnrqvsgrnrd\n>P04406\ndqaaeeeeedcpllslllvvlcvvvvrhhyaeyealvaaqvrsqvpnqqdpqqggdpwdwdddpcfiatnnhthhyhndpqllpaqcvvsnhqeyeydpvpqqecvsqvsnvnsvhqayeyledhppaqedaplqalvvdalvhryhypyallllavlnlvvlqcvpfnfpakekekafeddpladcaqddndvqnqsnhrlqpdfgkaadcnqccscsrvvschvrygyiytyhndhwkmkmkmktftnhfadlvrslvslcvccvpscpqaeaedadpddrsvcrlrqyswywhsppkddpgrgiiitmimdgrrrnsssrvsvsssssvvnd\n>P24941\ndvqkdfdawddaddqgtwtwiagnvpraiwikgkgdddpvdqaadpqlvvqvvvqqvlddlqawhfpdwdddrhiiitithdfddfqvvvlvvcvpvaddllllllqllsnlvslvscqvvqwaqlqddrrqwgahqlgrihghntpscvsvvppsppvplpqrfcllaalcvllappddgnlsvlssslqvslcsqvvdrqqrasdsvsslvslcqaawdddcvqencscvrpnddpppdthdhddvcvsrvrddpqssvlsclssrnhsvsrdhsvrssvgpscpprdnhrpnppd\n>P02994\ndppdfaefeeeeafapplcrqllvlllcvvlvqddpvqlvvlcvvcvvvvrnlcssscvfppdpvcsvvvwdqawgwgwgdaprytyiytyatndlvrvlrvlqslllgleyeyraeldvprnccqpdpnhcslvsllssvllpnqhyeyehepcvnvqvdpvslvvslvvvcvrnvvsphnsqqywyaydysrvgdlapdhdpshpvdpftwhqdpvgidtdhhpsvsvvprddddfaqvawwkwafqfwdqdppqgiktktftahhkdaqlfwkakpalqaiwgwhfkdappdtdrmdtgsgtiiigtppdhsvsdggqiimtgpvhphffwfqkfkkkkafaddpdwaafqdwfwkdanrdtftktqhffafkaqppprdtddggdrtddhgiititmigtpdthtaaacvrrvsrfwiftddpsdggmiigthhtdgdpdnhdydpnnvvsvvd\n>P61583\nddpvvvppdddqpwfkfwdaddddppddpddpdpdpppddgdrdrddpddddpdhrifiwtwgcdpvndtdtdgd\n>P05089\ndpppaaeeeeeeqlddqldddpqlscqvvllvvvvlvvlsvvlrhnydypyyddqdddppqdadvlatsqqsllrslcvllvvllvcvvvvhfyeyryraqlnvlnnvnslcvnflleaeeeqaqedlcdacvrdprnrnssrnlclqepvnvvvhdpthsnppgdnrdhlqryeyeqhddydprrvvvcvvrvrhyqhlvncvvqplvvvlvvvccsnpvvhdgayayedellqaacvqqvqfddhdhphdhlvsllvnllsllvvvrhryyyyygrrlvrqpdpvnsvrnsvsssssvvsnvpqdpvgdddpappvdgdd\n>P07688\ndddddpppdpddpppppvvadddpfqdpvlqvvvvpdpdqfhfdfqadpddlvqvlllqfadppddafaedppllppddfdqwdflcvvqvlaclsvdfaapasfnllllvlvqsfvqsllcvvlvspdrfrfdslqlqavlpplqhqrrnhghnqssqvcqappftagedddpppgtfhhdphdgasrtdpddhhhrphhdhrdhndqagpppdddtsnlrgwhfpdkhfydlaqssqqsccvprnkwkwkfwdfsrlsrtaagadadddghtshmgiwifgmwhadpnftwtkiaannalsphprriytygpdpcgrvrsvgimdthipnvsd\n>P06280\ndddddddddddddddddpppppppppvplqfpplvpclffaaaaeapqplnqacpcpvcvqrhqalvvllvvlvlcqvvcvvvlphaedeyepnqwdqaadpvretwghcvryvvtllvsqvssvvsvgayhyeaelaqappvggghcvvpllvhllvcvvsrhaeyayhyphdddlvsllvslssnlvsnvvnvdhyayeyqsqvvcvvpdhddlvsvlrrhqeyehydedhlapvlvvvlqvvcqvcvvpaqvswhrsyayasdeqaqqddhddlqlsqvvlqqcllqlhrhhhhyrsvrgdplsscsssqsvsscssrasrrghkhfldddpqktwikhqgpllkmkikifrqdqddfwdkdkdqccsppvsslqvqkwfkwwppphtdgpgidgnrdmdmdtaghrhmtmmmihyvvsvvvvvvd\n>P19367\ndvvvvvvvvvvvvvlvvllvvlcvvvvlldddlvnllvllvlllvlclllldpvrqvprlqnqafqpaqaafplpfadwawewapdfpwtkikiwgrddpppddidmdmdtddddlcllqeallsvllvvlvvvlvvcvvvvnlpvqhayfyadaadfhdphflwtqgcdddqsgdhpprhrdtvsvsnqvsnvvvvngdyhhlgydhlqllllnlvcvvpvqrfkrweaaswtwmkgwdqpcsrpvdfagdgttittrlclqpclvvpcvssddpllqvllvvdphrsgsslrclahqnrlqqsllvvvlvclcvcnflnndddplsvdgplggsvlllclqdppcnlvslcvsvvvsvgrgdssnssssnsssvssllsnlssvlsnvlsvqvnvcvsvvhqaeegeyrydydscprrpcrvvssvvnncvsnvrydyhydynppgsrnssssvssvsvvvvvlvvvlcvqlvlldddlvnllvllvlllvlclllqdpvrqvvrlqnqaflqpqdffpqpfadwewewadddqwtkikiwgfdgdpdtdidmdmdtdgddlcllaeallsvllvvvvvvvvvcvvvvnaddaheyfyadaadfrdphqqwtagcdddqrghhpprhrhtvsvsnqvsnvvvprghyhhlgyhhlqllllnlvcsvpvqrfkgweaaswtwmkgwdqqcsnnvdddrdgttitgrvclqpclvpscvssddpllqvllvpdphrsgsslrllahqqrlqssllvvvlvclcvcnflnsdddplsvdhplcglqllqvlqdppdrlvvlcvscvvsvrpggssnsvsssvsscssllsnllsllssvlsvqvvvcvsvvhqaeeheyrydypncvrhpcrvvssvvnscvsnvrydyhydydpprnrnssssvssvsvvvvvvvvd\n>P0A6F5\ndfakdkdffpvllvllllllcvllvqllqlfallrdwdwaddpdddifianqslvslvrdadpppssrvssvllnvllvllcvqqvdlssvlslllslllvlllvlvvvvfdlqlllvlqvvllvlllvlllvlfdapddlllqllllcvllsnpsvlsnllsvqcvqlhlpfaedeaafpaqdkdkdwafkfkdfkffpdllldppvvqskhkfaweweaaeaaeaadcllcvqvqvqvlvvvgayeyeyqyydhprvvvvsvccnvvsgnytyiyqhdddvlsvlssvlvcqqfvaayqyvvvpggsspddsnrigtarmwmghrtmimgahhphdpvslvvslvvlvvvlvvdpdpvsnvssvvsssrsvrtyiyiharnpdpsvsvnssssssssssqsslcnvsgkaqalllsllqsllvcvppddpdvssvssnvsssvsslrsnlsncvslvhhsvvqsvvsnpddrqwharsvvrdtdrsvvvsrigrsssvssssvsssvsssvsspdgmdmdrddddpppppvvppdddddddddddd\n>P02768\ndvvvvvvvvvcvvpvvpppdddppcddlapllvlcvqcdlllqlllqlllclllvvpddlvlsnvlsvvssvvnvvcnvpvpppprnddpcvsvlvslqvppcccvpqpclnvlnvddppsslvsslvsrdqappdddddddplvvllvvcvvdvsvslsvqssnlssnqslqqsllsvvlsvllnvqsvplspdppscvsrvvsvvvsvvvsvvrsllllllllccvqvnllsqlllqllvqlqafvqadlvrssvlsvlvsvlsvcssssvnsvsnssllvslvvclvcvvsgapqlvvlspddrssssvcvvptdhhdhdppdddpcvqqaqdpcnlvccvvpvsvslsvslssvsnvqnlaasllsvvlsvlsvvqsvvlspdphsnvsrncsvvvsvvsvvvllvvlvvllvvcvvqpllvsllvqlllvcqlcvqddlvlssvlssqlsvlsvvqspddssrnnnsnvssnlsslsvvlsvcvvppgdvlsvclsnvdsscnvvssvprggpppqdadpddlvllqdalvllpddpvvvssslssslsnlcsnpvpddpvlsvvlsvlsvvqqvvlspdpnnrvssvpvsvvssvvncvsrvd\n>P08684\ndcpddpddvvvvvvvvvvvvvqqcvqcvqqcvcvvqvfaedrqdrlqkcvvvlpvfplvvvqvgcvvrnawhwtalrnfiaiegddlqllccqfavcclpffvafddqfaaallclalnndthpsvvqvlvlqlvqpdpvnlqvlqvvllvllvlllvllvvcqvvqhwdqllqsllqsllqslccqalvdgqnsnppsvrplsvllvqqqpffcpppvssccssvvvcrvvcnvvvnrprdpvslvvllvvlvvslvvvvpdpddddphnnnslvvqqvddddpsrdradssssssnssvssncsrpplsqllqlllllcqvvvvllvvqlvllcvlapplnadhpvslvvrqslvlssllsllqflfqqkdktfgqawdddpnrtdhhshiymhrqnslsqpcvldpvsndrdsccshpvnvvsdscsssasqhdhssgnscvsvsssssssssssqsnwwdkhddppadvpfdfdsgntthtpdrgiigihtppvpppdd\n>P35354\ndvvvvvvvvvvvvppqqfdcclvvqafqpwawddddrndiatdcppsqaddrrsphhdpvrvvcvvpdddllvvlcvqqddlvvlvvqqvpvvsllvvllvllvvlqvladpqafaflvdqgrdpclrpdlfffgashgfndpracdlqrpdwdndffalllllvqwfaapdfafqllffflllvlvlvlqcllqpaappvvalrghlnsqnflqplqaqasdpllnvlqppqallagdwdddpnatfadfcvsrvraadddppqdrrlrgdgsrrdllfflssrlvsrlvnlqlvvqsvvlcvlcvpdgnvlsnqlssllvsllslqqcfqanvcsqssgssggdddlvspqpdpaalahghhsvlvllspqqlqddqwqdfpndthgsvcgtsdcvvcvvrdsqsvqqsrqagttgqqaqtnrrypvlsvqssvsvvscsvsndhaqqvlcsslvhhqdpalcqfqvddsvrvvvcvrtvgsrrhhnnssfrrghaddshsghssscsvnssnvscsscshpcsdpvnvdqvnnshpvsnvssnprhpqvscqsrpppshdhhsrhpdsvvsvvvnvvvppppdddddddddddddppsnsd\n>P04150\ndddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddpddpddddvvvlvvcllppvldddddddddddddddddddddddddvpvvvvvvvvvvvvvvvpdddddddddddddddddddddddddddddyddddddddydddddddddddddddddddpdppddddddddddddddddpldddppdpppppdddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddpdddfqaaqlarpgfpqddqnarhdpvlnvllvccsvpsdpfddpdpldddhhppccvvgssvsnlscvvrpgdpvvvvppppppppdddddddddddddddddddddddddpppddpllrllvvldddfddlpddpvdddapqvllqslqvlllvvlvsvlssqvsrplsvvfdpvqsvqlslvqslvlqllvqlvvvlvvvdlqwgpngpndidhpvncphppnvvlsvlssvsnvlcnvlvddpslssllsvlsslfkdfpvagpprvssvvsnsvslssnlvvlvvvdspsvcsvvssvsslvsslvvvvssvvvlsvvlvllpdpvrshdhdpssnvsnvvvvvcvvvvrmdgrgpddd\n>P0AG67\ndppdvvnvvvvvvvvpppdqfdwakwafqdddpfwtftgsvdpdtatdgqvlqqdpvgdrpddhgdidtwgfrdcdppprhtdtdvvsvlvvvqvvvvvvlqvvqdkfwwafadddpfftftatsnftetegpvqadpdpdvdppvrhgdtdiwgfpdddpvpsytytgpnvvcvvvlvvvvvvvvvvddqfdwakwffqdadqqftfthrsnatetegcvqaflandrgpcvpddhgdidtwgfndadpvrsytytgpvvvddpvlvvcvvvpdafdkfkfffadfdqqwtwthgdgnatetegccqqddfdspdrpcvvdphgdiaiwgfhdadsvvrytytgsnvvdddllvvcpvvddqfdkwkfffqdddpqftwgqtpsratetegpvqqdddddppvsvvvddhgdidiwgfndddppvsytythpnvvpppplvvvpvvddqfdwakfffqdfdlqwtwthrdvvqiateglcqqdpdrdsgscvpddhgdidtwtfhdadpvvsytytgpvsvvvvvvvvvvvvvvvvvvvvvcppdpvvvvvvvvpd\n>P15056\nddddddddddddddddddddddddddddddddddddddddpclvvvlvvlvvvlvqlvvvlvvlcvppvpppdddvvsvvvnvvsvvvnvvsvvvnvvsvcvvvvpdddddddddddddddddyddddddddddddddddyddddddddddddffweweqepprdidifgqdfpdflqrrcvvvqqlvlhhplqkwkfwddpndthtddrgdrsnvrtptymyidgdppfpwaafpkdkdadpdfdaqppprdtprtdiagpqqghthhpvcvvvdrgtdgnvvvsvvvvvvvvcvpdddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddadfddddddddddddpddpdddddddddddddddddddddddddddddddddddddddddddddddddpdpppppppdpddclvvfaddppqkafddwradhpfatwtfidhvhtkikgfgpdqdadpqlvvlvsvlvvvlssdddqqafhfrgwyppphtitithdadfafqccclppvnppddpllllvllllvlvvllscvsvvffqqdqasrqwtqgdvrrtyghngpclssvcrrpppdddspppddpllaalqslvvpdppsgdllsvlssslqslvcsqqsdrfppvdpdpvvssvcrnvvvdfgdlvsgdpphdplssvlsvqsndndsvshdgsvvssvssvvvsvpppddpddppdpppppddpddddpddddddddddddddddddddddd\n>P05067\ndddddddddddddddddddddpdqppqfpdfwwkwadaqffiwiqdrnpraidgdpvladhddddlvvvlvsscvvcvvfaqdakawdpdwdwdaqthgrpdpgrhddidttgiigrdgddddkddddadpqkdkdkdfdpvdkaapvvqvvvqqvvlvvvqwgwddkiaadhpdhriggmmitithhdpddppddddddddddddpddpqdddppldddpdppppppdddddddddddddddddddddddddddddddddddddddddddddddddddpvvllvvlvvllpddwdqddaqdfqwwwfhdpvvlaididghrpddtgnrthndrvvncvsrnvsvvvvvvvpdddddddddddddpdddddqqllrvlvpdpldsclsvslvssvvslvvvlvvvvvvlvvvlvvvcvvcpppdpvvnvvsvvvsvvvvvvsvvvsvvsvvssvvvsvvsvvvvlvvqlvvllvqlvvllpdvvrdlvsnlvsllsnvvsllvqlvvllvvlvvccvppvpvsvvcvvvsvvvnvvsvvvnvvsvvsvvvvvvscvvcvvvvqvsvvvsvvvpvvsvvvvvddpppppdpppddddddddddddddddddddpdppppddddddddddddddddddddddyddddddddyyddddddddddddddddddddddyddddddddddpdddpdddddddddpppvvvvvvvvvvvvvvvvvvvvvvvvvpddddddddddddppqppvnvvvvvcvvvdppdvvvvvvvvvvd\n>P0ABB0\ndpddvvvvvvvvvvvvvpdddddddwwkwwfadddqqktktftppvddqqfwkafpprwtwgfhdddpgititgtqfdcppddgggmittpvdhqkafaalqlqlfeaelrnatppppdgtdgpdidhlqdfaddpvqfdafqafqallfllccqqptftflfeaeeeeadplcsvlsvlssllvclpvqaaeeeqaaqaapvvvvvssvvcvvsnssrryryyddhvptalvrslsrlssslrnqvvqqlqqatyeyednavvnnqvsqlvscvvvvadadpvrghpcslvsvlssssnfgkgaqvsscvsvvnpdgpgiygyyyyhydhddpgdcvddvnvsvcvrgqwyfyrypvqvvvvqpsrtdsvrtygnspcsnhqplssvlspvlsvlqvllvvlvvvvvpdpdddpvsvqsnvlnllssllsgddrsnrhapllsslsslcssvcvcvppdsvcsvvvsvllvvqcvpppvvlsvvcrvrsdddpvnvvvsvvssvvsvvvvpd\n>P23246\nddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddpdddpdddppddddppddpddddddddddddddddddddddddddddddddddddddddddddddddddydddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddppplvvlvvvlvvpadvpaafqallqkkkkapaalpddpvnvqvllvslphwddwaddsvrrmimiggthnvssvvsqvvqaqdqdpntgmhidrddqllkkkkaqadqpdaqvlqlvqlcvlhawngkhfdadpvrtgprmimtrhrdnnssvsvqvdlvvhfrdsdppsggidmdhdddddpdppqdpvnddpdpvvvvvvpdddddddppdpvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvsvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvppddppppppppddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddd\n>P21802\ndddydddddddddddddddddddddddddddddddddddppdpdddaaededeqqakdkdfapdppfpakwkdfpphtddddpqwdddgrmiiggrddqvpfgkmkiwgddppdidmhihgyghdpdppdddpddppddppppppddppdfwkkdapdcvlvpqqedeeefqakdkrarhmagvvgwdkfkdfqnhgddpvvdqadwdqdrvrriimgggddqvsftkmkmwitdpvdididihgyhydyddfawkdwppcfqpadedeqqakdkgfididgpddkkkwkkfqdddpndqadpvrdgdidtddtpdladdprrsrmdmdgnddprnwgkmkmwigdpngidmdiytydydydddppdpvppdpvvvvvvvvvvvvvvvvvvvvvvvvvvvvpddddddddddddddddddddddddddddddddddddddddppppfdddpddddddpddpdddddaddddvvfaadpvqkawdawphddplwtktwiwgaclpvvhngdtdiwiktfgdpphdpvsvvlvvvqlvvlvlldddqaawhfshwhrpddttitithdfvqafqlnslllqdaapppsppdnpdhrpdhddlllllvllllvlvqllscvvsqkdqqdqarsqwgahpvrrihghdssvmdhqvphqwdaddddpdhplllfalccspvvtddslrvlssslvrsvcslqsnddhpppddsscvnvcvvvvdddddrpsddpvsvvlsvlsrdnpsvsrdgsvvsnvvsvvvsvvvvcvvvddppdddddddddddddddddddddddddddddddddddddddddddddddd\n>P11413\nddppdfddllrvlvvvvcvlcvppppplvaaaeeeeevcvdpcnqqpvvllvlvcqlsvldrpryayeyeeqdpdwlvvscvrrvvnnpddpvcvvvvvvrsvrygyahddlqdlvslqvvqvvqvpgdvslqyayeyeppddpvsllsnlvsclvhhdrphhhaayededdcaqflvsnvvslvsncvrdpllryfyddllllfpllvcllcvapvdpvnpvqlfpvwfqakewefeaqadcplvllvclvrfdcsvdvlgprlssvlsspfddfpdpaqvslvvssllqlvqkaafdpvqkafaweaadppdddrnrhtscnnpsddhprqggqwmwgwifrrdrgtvlhtyiytygylaqdgwtkmktwtdddppdpqvplddtkmwmqtrddqgktkiwdwdqdpdpgghtatdmdidgccvvvvpddgrrsvsrsvscvsssrnnsgqgssnsssssnrhnvvvvvcvvvvdrhhydrrnhrtdvvsvvscvvspddddvrdddddpvvd\n>P07901\ndddddddddddppppddddddddddllvvlvclqpplapdllclvllvvlvlllqalvlvvvcvvpvvqcvldvafawewefdppqqkikiwtqgqwdapccccpqvvdaqhdpscvvvvvppdddpdfhdpnrsssnssllnfaqkkwkwfdhpphaikikidrsnndididgdpdddrsggmmimgngdpvrsvcsdpvsvlvsclqwplarphwywywdkdkdwdwdfpvvvvvvvvvvvvvvvvvvvvdddddddddddddddddddddddrdtdidididididgrnplydplvddpvpddqvnqqvnvcsqpvdpggfldkdwdwdddpwtktkikthhlaadpclpvpvdqlqqefeaepnttqdrrcclqepslvsrihmymythdfcadpnsndgpdsvvsnvnnvvvvvrvlvrlvvcvvvvvslvsscvrrvlsllvcclppppcvqsslqsdwfdkpprppgtdhpvvlvvpddpldqaaeeeedadpvlvnpaqlcpvcvvvvhmytyhhdlsvqsslvrcqdhpnggydysqeppdddpddpvvvvvqvvlqvlcvlvfvllcvlcvvpapgegadprgdaqfkakgfypaadalsnvvsqvpdppddpvcrvvrptgiymhghspdpvsvvlsvvcvvpspppvsslvsvlssvvncvssnhddpdpvvnvvsvvvvvcvvvvndpppppdpppdddppppddppddppddddddddd\n>Q02880\ndddddddddydddddddddddddddddddddddddddddddpdddpdddppcrvdddddplvcclvpvcqaqralewdwdwdffqdpvqgtdtdiftdgrsvlqllllvvlvqlllcvvfvpwaeweweadfvqqktkiktfdakqdqdqdppprhgpvcdlqldaqddpadpqvpwglgpdhrscnnlsnlqqfqkkkkwwdgqvnqkiwikmaghrspdidpidmdghhdgtmimimtrgncvssvddtddprsvrrsnvllllclaqranyfyhynnhtdprrynvssvcsvqppdddpvrhgwdkdwddqdpfktkiktwdpqfaaefedrnsqtlqveeqvvclvlvvllvlllvvlcvvcvffdrddsrllntgmhmyihggftpwdapgnsrnytydhpvrrvddrdddpvrsvvsnvrcssvvsvvvrvvvllvvcqvqqadfldqddpddvqkqaanqqshqcqllaeeeeeaavllvvlvvllcvqvprhrymygydpaaddqplpdgsvccsppprnsvvnhrqnngfpdadqdpvscsrtrhsayeygyfpdqsslnnvlsvvsvccrghvnslavlrykyfaafqkwkdadpdidghrdplrvvvvvvpdpcpvridmdgrggpsvddsvvsnvssvpvqqgmeteheddcllvvlsccqrppvnlvvvvvllqvvvvvvlvcllqlhgqdddddsphrydysncssnhrnsvsqlvlcqqffaalfaldrpllqlllqlqvvvvdpdkdqllvslvsscvqavddddsvvssvsllqladaawqgfldrqkhwadqsaflqqdrprrddsvptiihgdplnclqqppllvlvfafdadpnhthggpdthtlfrrcqarphwgdgrsataghahfdpvlrlvqlvcvlvvhardatfhdtppadfdwddldqqkikfffwkwaqdlwkikgwtagrppgnnnlcvvfvvclcppdpagdrlfpdkdflddrhigiiitggpsvsvvvcvvvpvcvnsvridmggrqrnwhqypsggidgdrgpsvssvsnsvvllvslvvslvsclqqlvlvllllvlqlvllvcvvvvvddpppddpvvvvvvcvvvpdaqnsnvvsvvvsvvvsvvvvcpsdddddddddddpddrhpcsnvvddpvcndpvnsvvsvvvsvvsvvvsvvsvpdrssvvsvvssvvsvvvvvvvvvvvvvvvvvvqddddppdpddpdpppndpcvvsdddpsthmddgdcpvvsvvvvvvvvvvvvvcppdddpdpddpppdddddddddddddddddddddddddddddddddddddddddddddddddddddddddddyddddddddddddddddddddddddddddddddddpdpddddpddddddddddddddddddddddddddddddddddddddydddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddppddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddd\n>P14598\nddakawqakawwfkkfdppphtfiwtwiwtqipvrdtaidtdgpvlvvvllvvlcvvpvqcccvphvvndqqdddddddpdddpvnrvvcnvvvrvnrvslcvddccrrvppsnvvspdhdpvspddqdppdpdgmdtgtddpppppdpvcvstpppffkkffqawdddpdpqadtdhhgdiwgfpdfdllqwtwiddppdihtarvqriagpndnedppwddwalvffkkffldfdddpdpqadgddggdiwtfggsgssqwtwiddplfihtdgnlrmdgppddsvvsvvvcpddrgdhnvpdpgthdpdpdhyddddrvrvsvvsvvvvvvvvvvpdddddddddddddddddddddddpdpppddppvrccvpddpvvvvvvvvvd\n>P10591\ndffewewelaqfkiwiwaddpnaidtfaapvrhsiftlkwfadlaaidgtprllvcclvfqqridgsllqqaldfcpdplnvvvclvgshdwdhdlrgtwtwdqhnndidthhsllsslvnvlssqvrvcvvvvhhhaefayedaqpddqsrlvsvclscvlnnhdhlfydhqqlllvclvclpdaddwaweweweaaaawtkiwiwidyqqetatlamdigngahnvvllvlqlvvvqvvccvvpvdglvvpsslsslssvlssvqqvvllvdqkdwrwaqcsdprdtdidmdgnvrscvscvvslvvscvrvvvncvssvhdlvrrpayeyaypvcsrpssqvvvcvvsvndhhddpddrdcsssrsrnlvrclsvpnpdpsnprhdywfwqnwfkawqdppqfgdgqrhgsdtpqdkgkdkdwapafqdfkdktwmfthddggrvvthtlaididggqggggtppwtkiwmwgqhpnrqiwikiagvvvrridtdsspppnsdqdpvrsvvssvvsvvsvvsvvqvvlfvvllvvllvvlvvqlvvlvplppladpvlscqlvvlsvvsvvvcvvcvpdgsvvsvvssvvscvsvvvrvvscvvsvndpdddddddddddddddddddddddddddddd\n>P61869\nddddddddddddddpddpddpdppwdqqdpvnvvvlqqqlllqllqqapcqqvdpffqfrddpvcsnvvsvvlrktfrhhdadpdqlrwtwtwiarppqlatetythddldllvcvvpdpqdwddlvldvpwtwrpslsvscvvcvvvvvvscvvvcvvrvpgayeqehaesrllnslvnqsvvcvvprlryararhlyafttapvnlvsslvsdnyayeaeaqalrncpppvvvrtfhrdwyfyadddapddgdsvrttthgdrddlvhnnvndhrdsvpcssrcgrngssnpshdpdddrddd\n>P10275\ndwddfdprdtfdafddpplpclvvvllvllvccvvdvdddddpdppdddpdpvvvvvvvvvvvvvvvvvvvvvvvvvvvvvppdpppdddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddyddddddddpvvvvvvpppdppdpdddddddddddddddddddddddddddddddddddddvlsvvvsvvlcvvqvhdddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddppqdddpdpddddddddddddddddddddddddddddddddpdddddddddddddddddddddddddddddddddpddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddrappvpvqppqraaqlarpgfpqdaqqgghdpvlnvllvcvvvdpddfddpdpldddhhppcvvvgssvsnvscvvrvgdpcpppppddddpdddddddddddddddddddddddddddpddadpcvqlvllvsldddfdalpqdpvdddalqrlqlsvqsllvvvlssvsssqvshplsvvfdsvqsvqlclvlsllllllvqlqcccppvvlqwgpsgpnridhpvscvrnvvvvlrvlssvlsnlcvvlvddpslssllsvlsslfkafpvggpcnvssvvvsvvslvvslvvlcsvdvdpvssvvssvsslvssvssvvsslvvlvvvlvclsccvvrvhdhdsvrnvsssppsscvssvsmdtsgsddd\n>P38398\ndpppvvvlvvvlvvlvvlqqvqaapqprgraaqwwaapvrdihhpvrqvvqqqpdpdfradppprdggdpvridgdpvsvvssvvsvvvqvvscvvpvdnsvsppdppvapppddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddadddddddddddddddddddddddddddddddddddyddyddydddadddddddddddddddddddddddddddddddddddddddpdddddddddddddddddddddddddddddddddddddddyddddddddddddddddddddddddddddfdddddyddddddddddddddddhdddddddddddddddddpvvvvvvvvvvvvvvvvvvppdddddddddddddddddddddddddddddddpddpddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddpppdddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddgddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddyddddddwddddddddddddddpddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddadadddddddddddddddddddpdadaddddddddddddddddddddddddddddddddddddddddddddddddddpppdpvpvvvvvlvvvlvpdpdppddddddddddddddddddddddddddddddddpdddddddddddddddddddddddddddddddddddddddyddddddddddddddddddddddddddddddddddddaddddpdppdpppppddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddydddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddydddddddddddddddddpddpdddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddpdddpdddddddddddddddddddddddddyyddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddydddddddddddddddddddddddddddddddddddadddddpvnvvvvvvvvvvvvvvvvvvvvvcvvvvpddddddddddddddddydddddddddddddddddddyhddddddddddddddddddddddddddddddhdddddddddddddddddddddddddddddddddddddddpdddddddddydddddddddddddddddddddddddddddddydddddddddddddddddddddddddddddddddddddddddddddyddddtdddddpdpdddcpdppvpppvplqaeeeedpddpvlvvlvvvqcvlvvhhydhdddlshqeyefadpplqagadalslllcllllhlyfyscqsvvcvvvvhrddsvvggrqayppldrhlcrsvcsnvpqvcqaqaqeeeeaddddppadvvslcvsvvsshhhydddlvpddpdpghayeyeyacvrddpdpclqavcvshvhfyfypvqsrscsrsvhrddrvsgtdhhdddddd\n>P02452\nddddddpvvvvvvvvvvvvppddddddddddddddpqeweddprdidgaqdwddpaqqkiwhghrhdtdidghdddppqpaddfpapvshddgdrpppppdpddddddddddddddddddddddddddddfddddddddddddddpddgdddddddddddddddddddddddddddddddddddddddddddpdddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddpdddddddddddddddddddddddddddddddddddddddddfaddfdddddddpdddddddddddddydddddddddwdddddddddpdpdddddfiddddddddddddddddddddddddddddddddddddddddddydddddddddydddddddddddddddddddddddddddddddddddyyddddddddddddddddddddyddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddyddddddddddddddddddddddddddddddddddddddddddddddddddddddafaddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddpddddddddddppddpddddddddddddddddddddddddddddddhdgddddddddddddddddddddddddddddpdpdppddddddddddddddddyddyddddddddddhdddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddgrddgdddddddddddddddddddddddddddddddddddddddddddddddddddddddtddddddddddddddpdddddddddddddddddddpdpppvvvvvvvvvvvvvvvvvvvvvvvcqvqppqalvsraqallsncvscvpdafdkgwhdlpdddsvltfiftrhsvvsktwwdwqdqkwdwaafaadpdlvdfdwdkqqpgtdvrrwtairypphdsvsvvvslvsllvvfqkkkwkkkkkwaqfaaaaapvpgggpqhkwwafqvrqiqgcddpvqrhkdwpdgqrrhhdrdidmimimdmdsnsrrddttimigrrhnhnsiiimmrtttmmgd\n>P12821\ndddddddddddpddpppppppddpddppqqppllaqddfaldlvrlvvllvsnlvnvlvlllqlqvlvlccqlfpdpvslvsnlvsqlsvlssllrslvvlcvryvvplvprpallsslssvlsvqsqlsqddpvlssllsvllsvllvqlqpfwfaadprrvdthgvppgllccllpdldplsllsslvrllvsqflvnlvslqsnlvsrqvrrvvvvanfnlsvqvclqvdpcvvvqlvvllvvlqllllllqllvlvlvcvrqnpqqgdlqafhaqrnqngsrnqfnlsclvsrqpcpvdfdlwllvvlvvvphdqvvllvvlqqqlvllvfdgqaplavvpeqrardpvprgrnqpwawddslplhhiyirhnadstlvsslvslqrvllvslcnqaspprpslsaalgllnnslssnlqslqcsalvncvvvpsgpdddpdlsvllsslssvcsgqssnllvlpllsvlvvcsrsvvqplqcqpvssqvsccvrnrhdhqdddhsstrssssdscssnvnrrnsnnlssqvsllqlvllcvqlvhddqssndhsrsvnsssvlvsvssnsrnsdnvlvnscsrrvgsdrdnvsvcsnsvsnsvvsvvvcvvvvgdrgrpvhvdhrdddppppplsqadldvvvlvvllvvllvvclvlllllqvlvlccqqfpdpvslvsnlvslvvvlvslvvslvvlvrhdlvsdppllssllsvlsvqsqlslddpvlssllsvllvvllvqqqpwwqddpvgdtdgppppllvqllpdpdpvslqsslvsllvrrflvslvrqvsnlvsnqvscvsspanwnlrvqcvlqvdpcvvvqlvvqlvvlqlvllllqllvlvlncvrvhplqgdlqafhaqsnqngsrllfrlsvlvsrqpcppdddlwllvvcvvvphdlvvllvvlllqlvllvfdraaplqvvpaqsardpppgghrqdwakddslplrhiyihhnddrtlvsslssqlrvqlvrlqnlqspdrpsqsaalgllrsslssnlqsllcsqlvncvvvvsdpdndddlsvllsslssvcsgqvsnllvlvvlsvlvvcvgsvvddlqcsvvvsqvsccvrnsydhqdqdhsstrnsssdscssnvprrnsvnsssqvsllllvllqvlcvhddqsssgrngsrnsssvlvsvlsnrrnsdnqlvsscsrrvgsdgdnvsvcsnsvsnsvvsvvvcvvvvrdrgnpvhvdhsvvsvvpdddddpqwdddssdtdgpvvnvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvpddddddddddddddddd\n>P00968\ndafdplfqeeeeeaqfwaefffalllllllllllvlcvvvnhqyeyeyqfllgpsqdcvshvfyfnffldllqvllvcvvvlgqeyeqfrhplsslvsvlvcvvvcscvvsnyhyfqddnqlqccfqvpvsvcvlcvvvvaaafdkdkdldpvrqvvgcvvqdpqkwkaagsdglsqliwtdrddvvssnsllsrlvphprsmmiighdpwlfkkkkwkkfaaqvrqietqwiktflfhpffapsqtkikthddpddpvlvvvvrvsvssscvssprhqamkmwmwtahpvprrihthgmdshdgpvnlvsclvqvasrsnrsssrrrpddqqvaarvflvsphtssdhfddqkmkikhfaadclvfvlfdqfggsrghglwmfmaiendplqrqqqrqllrpppdqgldapddlpppclvvvllcclqgpgscnsnsllvclvnpndlvnscvrrsrdsvvsvlsnvlsvllvvllvcapvvpdpvsvlvcvqsnrqlcsscvsrvhhsvvsvvvcvvsvqawakdanrryhppgdtpqqhiethrdddhpnpaddpfaeeeeeaqriddqfqgsllvslslllqqlvvvvpgqyeyeyggsshpssnsvsgshyyngrldlrgvvsvcvvhvhpeyeqqrnpdsslvcqvvcvvvvhhhfwqhnvlqvcflvpvsnvvlcvvvvaaafqkdwaqdlvvvqvscvvrddqkfkdfrpdpqlsltdgagdsvsssvscvppvvpddstimmighqpppwwkkkwkwadllpdiatlfiwiwldapffapsqtkikapddpddpvlvvvvvvvvgsscnssvgqfmkmwmwiddpngihtdgirsgdgscqsvscllqvpnrssqsscsrvpqdcvnvvnhyhfrfqkmkmkhfaanvssglsgqpdggshggtstmgiaignhlllrllrrcssnvaladleaeeedeadpvclvvvlvllvlnvvsphayeyedvsqvsnvvvvhhhhyadqpvgdpphllvclvvvraqeyeaeddddvrsvscssnrsscvssrrdydhgvsvsvssssnspydsspdtrgsvrrnvrtd\n>P42336\nddqfpalalevvhndddqwdwaweqepqrdtdtdidgqqdfqvvvlvvclvrcvvdppnvvdddsvqkwkwaqaslrdgqtcpprrggssvrrghdryiyidgddddpvlsvlqvllcllqshrpvvlvpdpdlqslcllqvlvvllvvllcqcpvppplsvccqvapfqfdpdldafpvlcvladvqkdwakeweqdpdvrdididididhqqafflvvqlrsqcrvcvvvpddpvrsvvvsvvlsvwkwkaffqwghtrsdggrncrrpqnvvcslvvhhgyiyidtpvvvsvvrdnfdfddhpsnpddrpvpppppdpqdeaalqpdfdfwkkwwffkpafdddlvffawkkkkkfkddqpdtlfdidifdtdrrvggggrdidttpftrnlqalqikmkmfiktwgddvpddididtrwiamagcadpvqwgdaakdkaftdgdfppdpdntdsqafhghqppppgiiimmggddpsgtyghddlvvllvllvvlvvvlvplpdvrqvvalladdpppddvvlvvvlvvqllfflpddddlnnlsslnncvvvclvvlscvsvpvvsfsssgsnrvsnvlvclvprdadallslllclfsnnqhqssvvsslvncvvhpdpnncllqvlllvlsvsshngldhsslssqlsslsshvqsvlvlsqslsqccvpsscvsssssssvsnssnngcvsvlsslvsqlvslvssllvccvpvvvvpdqvvnlvvvlvvcpdpsnqvsqafhadslgrrfhfhgwpsvqwgqdpdpqrwiwtkgagprpssvsphriwiktkgadadcpqvllllsvlvvlcvlcsnlvqsqdadswgwgrrydritmtgddpqkdflvrqqvvqdadpdphgdlcsllvllcvlqpdplsvvllvqllsnvlscllscqlfvqadddrhqwmagssnhiygddrldtlvvvcvvpvdpdrasrdadpssslcsqqvndppslpdpsnvvsllsnlvslvsvlsvvnsvlsslssclsvsrpqrpdsvsvvsncvlsvsvddslvssvvvlvnncvrhvhsrdppsvvsvvvvvvsvvd\n>P42345\ndddddddppvvvvvvvplvvllvcllvqlldpdpvsllvslvvvlvcvlpvlllddpvvnvvsvvvnlvsllcllpdpdpsslssslsnllsvqprdlcpvvvlvsllvslvsqqldddlvsllssllsllssllspalcslvslvvllvqllvlqvdpddpssllsslsnllsclvrpvqnslvcvvsllvrllsqlldqdpsslvsslssvlssvvsvlvfqvffpdrdcslvsllvslvvlqdpvvppdpdhdnvssllsslssvlsllcqqdvvlvvlvvvlvvllvlqvvlvvvvvvvvdddddddddddpdpdddpddddddddppppppddppppppssddppppsdprghrpvsnvvcvvcvvvvlvsllvqlpdpdvssvlsslsvvlsslqsdvvvcvvvvvvvsvcvsllvqcpppsrvlsslvsllsncqsnvpvcvvcvvvllvslvvlfddpppddddddprdrdlsslssvlsncvshqcpcvpsvvvvvvslcvvadgssslvslqsscvrhvvclvvsqqslvqrlccllvvagqddppddppvsvvpddpppndddpcsdpsrllsslvsllpddhpdpsllvvlvsllvpqcpdpdpssnlsslssnlsslvvllcqqpvdppdrdpvsvvsllsslvsllcslqpppdpssvlsnlvscdpsclvnclppsnvvsllnqcvdpdlssvlssllvllqscvvnvppslvslvvvlvvllcqlppdpdlvsllsslqslqsnlqshlvncqvvlvvlvvsllvllpdppnvsdcsnllsslssllsslvshfqvclvclvvlvvslvvqlvdppdpsnqlsslssllsnclrnlcalvccqvpvclvvsllvclqpppdlsnvlssllnlqsnfqfdvlvncdsvvvpdppqpplfdpppppcpddddddpdllvllllqlvddlllsllvlllvlllvllqdpvslvcnllslvlvllslllcqqvclvclvvplvsllvclvpddpvvnlvslqsvlsnllrnfqscvvcvvsvlvsllvpldldpssllsvlssllsncrrnplvclvsvvsclvsllvllvdvpddlspsllsslvsllsnfvscavpcvspvvsllcqlldpphhpssnlsslvsllsnlvgydcpvvcvvlvvsllsslvppvvcvvssvssvlsvllscqlvcpvccvvsvvscvvvvdddpsnvvsvvcsvvvhrpdpdpppvvvvvvvcvvvvvvcvpvpddsppdrrqfdaadcpqllvllpqssardlvslvvsllsnllsllcrfrrsssnscsvvcsnpvvsslscslssvvrvvvpddpvvlvsnlvsllcslpsdlqlvnlvssllnqlsqclhpvhgrcndvlsclvsnlvscvvllvllsvlvsllvvvvvpddpvsllvnllscvlvvfllvnllsvvlccvppvvdddalssclssvvlvvslvrlvvvcvvpppplvsllsnlssclslvvlvvnlvscvvcvvvddlvsllssllsnlssclsvvnlvsnvvslvshdllalsnlqsqllncllvvvnvsnvvslsssssslssvlssqvppdpvsssvsslvsnlsnvsvvsvvvvvpvvcvvvnlvslvvsvvsddldlsslsssvssvvsdddcqrcvvsllsslvsclvvvnlvvsqvslcsnqvhrcvvvlpdqrdcppvssllsvlvsccsvvvlvsslvslvvslvvlvvclvpppdpvcpvvslvslqvslvslqvnlvslcvvpnddpvnlvvslvslvsscvspvldlvslvsnlvslvvqlvclvvvvvvvvvvvvvvvvpddddddddddddddddddddddddddddydddddddddddddpvvvvvvvvsvvsslvsllssllsllssqvsddppnllsvlslvvsclppvldpsnvvsvvvslvvddlqvcllvlvllllcclpprvssnvvslvsllvclqqpvlsnqlsllqqcqdpppssvvssvvsvvsscvrcvqqnvvsnvvlvllqlvqddllrllqvllvvlccccvpvvnpvvscvslvvsvvvcvvpadfqssvvccvvqvvlqvqlvvlvvvcvvpvdvvsnvssvvssvvsnvvsvvcvvvpqkdfcngrpvvlqvdadgpradaqpddpvddgfdfrhwdrmwgfdpdpqrwtwtwtqtpvrdtftktkgfqaacvlvvlvlslqsvlqsllcpdpvsvvlvqgashhgwrdsyrgititgdhpqkdfqlvqlvvlcvvvvhdscqlvvqlcvvpvplvpddlvvllvslvrslvvddlqslqvslsslgsdsvqsvqllsqqlllqllcllvlllfqadppfrrqwiarsrrnhihgddrpdtllpqqlapalrfqashadfpssqsnhspsgcsgsnlvsnlvslvsclvcvvsslssllsvlpspsslvslcvsppppppddddddddddddddddddddddddddddddddddddddddddddpdpdpvvvsvvssvvssvcsvcssqvpnppnpdgddssvssvvnvcsnydssslsshdvssvssd\n>P0DTC2\nddddddddppdqadaddppfeddddwdwdfpaqdfaadpdpdfdaqakdkawdwtfdhrdiwtkawfafdpdpvrhtyddwdkfwqpqwkkkkkfaqfqddkkkkwadplalpawiwiftdqqwktkiwtfsfhfhpqfaweweqdpvvsaihtdhrnttsdthdipdidightdhthpdrhdaagfkmkiwiwgqdlqktwiwididtdghnddddhdddytytddidnhngdttimimhfywyfdpvdppdpptdtggrtmmmimghtdgwmktfgahrrrgghdiaglqaaplsvqcvllshpwdaffkawgddpqfdapaefefaapadaawvvvcqvvpldaaflldkdkdktfqhaddpvvvvpppdwpdkdfaqfdpscqqlaffrikikikhkdfqlclvqadaprddqsvqsvdhhhnrfgwiktkdfplvaqadpvhvqqrkgfphdpddddppdrppdqdwqadddddppvdpddrthgridglnahspddnrsgmmmmiiitghhqqarggyhhpddgddddapgwhfyransqtftfgwaadpddddrrdafdadplrfgqwgqrnyvrgitgghflaqftkmwgdnmvvvhpaikmfgfpealvcqcsrnvlvsrdpnhncnnphppwdqdprgiigfaaedadddaapyqpfpfkhkdwdqpdddppdddddrdggiymyghadyddddddddfdwdkafadwdkdkdkdwfflfddqdaaqlqllqqqpdsllsvlvcvvpclsvvlnclssvlvrvsrvlssllvnlvvvvvpppqpcddpndgcqqqqdrpvcnvdghvvcvvppqwdfaqapclvvqlcvvcpppllcnvvsqdddvnqgadapdddpvnvvcvvccvvvvpppddddddpdddddpdvvsvvvshvvvsvvsvppvpppccqvcvlvcllcvlvvvlvvlvvvlvvlvvclvvvvvvvvvvvvcssvscsrrsrsvlsslvrrddsvvssvvsvvssvvssvvsvvvsvvvvvvslsssvssvlssvcsvdaavhqhpdacsqhhagfdhwdwdddprtimittihihrpdidifgwcqddddpqwdkgfpaatftdpppatfgdhsrhpdthhddcvridtdhgrddppppdpdpppppcpvppcpvcnvvcvvpvddpdpdpcpppcvvpvsppppcvvsvvssvvsvvssvpdddddddddgdtdgppdpvvvvvvvvvvvvvvvvvvvvvcvvvphddddddddddddddddddddddddddddddddd\n>P13569\ndaaqpvvvddpvclllvvslvvllvdllvdfddlvvfhfddpcqflvnlqvqlvvlqvvqvvpdpqgasvvslcvvpvvllvvllvlqlvllvlllcllvlvlllqvlldlvdppslvcnvvslvvnlvslvsslsrllvsqlsqlssllsvlsnllllllllllfafplvlvvddlvvvcclsppqsvcssnlsslqslqpslvvllvsllvvlcvlqpplsvqlvvvlvvllvllvvlvvvlvvlvvvlvvllvvllqlllqclvplvvclllvclvlslvvsvvsvvvnlvsllvsvvsvlclvlsllqslvvslcssvvvrllppndelssslsvlssvnssscsrrpsnvsnvvsvvsnvvssvvssvssphdtqdaddadppaakkwwfffwfdshpvllvvvvvvvvvvvvvpddpdcppvvvvvsvvpdailaggdtdmghflfeeeeaaadslcpvvvvcvqlsstdtsdtdidhhkaefeqdpqqfadffwlvcllqvpddddpvllvllcvllvcpvvlvpdpvrgrdgaagrrpvdfplvslsssvsnrlsddiqeyeaecsclraflvsvvsncvspvcpssrrhhyyhhalapvvlqvgqwyfygdsnytpdighpvvvcvvcvplvclcvvcvvrnqqhsqqssqssnvvvvvvvvvppddddddddddddddddddddddddddddddpddpdddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddppddddddddddddddddddddddddddddddddpvvvvvvvvvvvppdpppdppdvvsvclhqpqpdpdpdladdpvlvvvlcppdvllvvlvvvlvvvlvvllvvvlvvlvcvlpdddddpdddddddddssphpadvvnnlsvlssclsslsnssrddpvnlssllvvlsvslvslvsllssllsqffvlvsvsqdsvnssclssplsclssnpqsvlvvvlvslvsslvsscsslcvvpvvlvvlcvvlvvvlvvllvslrsslssllvqlvslvrqlsslsvscsvcssscsssvppvvssvsnsvssssscssvssnsssvssslsvssvslsvlssvlsvvqsvpsdshrsvsnsssvssssssvsvsvssvsvvvnrsssvsssvsvvssphdgppddpddddddddddddddppvplvpddppddppdqfkkwwfqkwfdaddvddtlaggetdidgglfeeeedegrsscrvvvvcvqlvstdidgwmdtpnripnsdtssrsslleqeqdpdqsdrfffqvcllpspppddpvllvvlcvllvvnvvqvvddprrrdtapdrnpvdfpqvslsssvsnslvslhqeyeyaaspsghnvvsvvsvvscccpsrsrhyyyhyhsnvvvvvvgqkywyrdpsytdidrapvvvlvpdppvvvspdpvvvcvrdppdpppdddddddddddddddddddddddd\n>O60885\nddddddddddddddddddddddddddddddddddddddppqqaddddddqadppddqfdwqqlvlcqppllvvlcpdplqvcqadwdpcnvvvvsclcvqqvdtagsvslnsnsvnsvdpdnvvsvvrlvsslvsqprpdddpdvnnvssvvsvvsnvvsvvvgdpdidtdddpdddddfddddddddddddddddddddddddyddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddfaadddddddddddddddddddddddddddddddddddddddddddddddqqlvllvvllvvlvdpvlcvqlvvqadfppcvvvvvvclvvqqvdtagsvvlvvcsvvvvdpgnvvsvvrlvsnlvsvprpdpcpdvnnvssvvsvvsnvvssvprdddpdddpdddddddddddddddddddddddddddddpddddddpdpvvvvvvvvvvvvvvvvvvvvvvvvppdddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddpplppqdaadpvslvvlvvllvvadpvlvvvlvvlcvvqpvvcppddpvdddddpvpddssssvvsvvssvvrppddddddddddddddddddddddydddddddddddddddddddddddydyddddddddddddddddddddddddddddddddddddddddddpdpdcpddppddpddppddppddpdddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddpvvvvvvvvvvvpddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddydddddddddddddddddddddddddddddddddddddddddddddddppddpppppdddddddddddddpvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvppddddddddddcpvvvvvvvvvvvvvvvvvvvvvvvppddpvvvvvvvvvvvvvpd\n>P06756\nddddddddddddddddddppdddppppppfdfwdppdwdkafddppqlwqlakdkaaapvdpaiwiktwrqqdddqappahrltfiwtahpppprdtdtdpqdnyafdaldvvhgqwhshsqrwrnywdddhqkikiwrlqifgldpphgdtfslikmwidrphdifidrqlqapqddqqarsnqswqlewdawpqqkiwiwrqrgrqrltkikiahpvqrrpqgdppdrntdgdqmamdddddplshnqrwrlywewddlppppttwiwawrccppnrqtkiwtadnrhrhtqdmdghpdhplswrnykywdfqqlpphikikiwsqqdfdqdpvrdtfrfikiwiwprdpvshtdidmathddgqlrkrlykewaqqqqppngtkikiwsqqddvlrlikifiwhghnvgidsdtsdidgrpdhadpdrqswqnykywhcqnpqqqgikmkigrsrrrmiiithtfwekekdkdkdkpplefefvpqddaqppdrdtfgkikikikmamatdgdwdqkwkkkkkkakqvvqppphdrfkafrpvrhrididididghdddidmdmtmmggdhpvpdddqqdwtkmkmfmdtpqqvtadpvrrghdydsphdrmdidtghyqdfcppvsawakakdkdwdkpdqedeaqdwdkikikmkiwtqtawfaqkkkkkffdpqkdwpawddpdpladrwdwdwdddppttiiitgcdrtnghgdmgiimtimignhndqedfwtkmkikiagpgphngiydididtrgydydwpkdkdkekvvqedappappadadpqaaalrpfadkikmkmkiwtqdsfkwfkkkkkkkfqqfapphgfwqwqdkdkdaqkdkdkpddhppsnrddddppppdpppddpdddddppppppppppdppddaaergvvrgpiimmmmmggmdhngdmmmmmtithgrsvvcvppvndqhkykgkmkmkmwrdggsnvvhdddididmdmyiyiyhhpddddpddddvvvvvvvvvvvvvvvvvvvvvcvvvvvpdddddpdccppvddddddddddddddd\n>P04626\ndvvvvvvvvvvvvvpddpdqddqaefeaffpqapddpdlvcplvllllrqqshaeyfyeaaaadaeqprdqpslqnhaeyaaayhaehaqhaehnnlnhaeqarpdadvraareaayqffdpcpvpdpppgpdgtgyqfhnnqnqqeyaeheyhhdngqnypqpvldplvrryfpqhqrhdddhddhypddddfadpcqvsrharhndpsgrhdgfaplddppargaphndpqrgadpqagsgfpghaqlthpagpaadessghdldfawqwdqdpvprdtdgdprgfaddrrythndadfcweaenrrythndadpqwdwdadpvrhtytdhdpdrdddaqedacddpnvqpdadapvclvslalhqeyhyehehdpcrqppdpvvptdgndlvsllsqlnyaeysaayyyaeddlvdqeplsqlnhaeyqnlhadssaaraeahqyqhqwhnnqnhqahndheheyeqhanyeqpvlapvvrryddssydydydnyhdhvvcvvvvffedpqapprhfggyfqqthndgpaadeprghdpdaqfqhdpqhfdddprythgaapqfdrdhrggqahdhfqlrgpdgnaadepsghdpaaqpqddpdppggrwawdqdpvrytdtddpvdphhqpdadpvrddddddddddpdvvvvvvvvvvvvvvvvvvvvvvvvvvvvvcvvvsvvvvvqldddddddffqafqpqsqahadepvqkafdawpdadpfatwtfiwghpvpdpatatkikgwgdddpdpvvvvlvvqllrvqrrdddqqawhfnywyddstiitithdqqqafqlvnlqlcvladdllllllvllsvlvqllscvvsqkfqqqqasnqwghndsngihghdssvidrdfpfdqwdadpfdwhllllfalccvpvvtthslrvlssslvrsvcslqsrddrpppddsvcvnvclvvvhgdddrpffdpvsvvlsvqssdndsvshdgsvvssvvsvvcsvpscvggpdprnprdddflvsrpsnvvvvpdpppprhdypvvvvdpdpdppddddddddddddddddgtpdddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddydddddddddddddddddddddddddddddhdddddddddddddddddddddddgdsndsrddhpsstgdddddddddddddddddddddssidtddddd\n>Q13936\ndddfdddddpddddpddddddddddddddddddddddddddddddddddddpvvvvvvvvvvvpvvddddddddddddddddqdddqvhpppppvvvqadqdapnprgppdpllvvllvllpdpvnvvvvvvllvvllvllqpdfdaappdddpsnvvsvvvvvvslvvlvvvlvsqcshrhdpdgqrhqcfdvvsvlsvvlsvlsvvlvvvvvvvvvvvpdpdddpppdppslvsvlsslcslvcvcvvdvlsvvlvvllvvlvvvcvllvvvlvlllllllllqllpfapqfqkwkdappdpdpdgpdqqtagadappaqaddddprigidgdhcgglrvqcgsffslsssvnlvclqlvpclvvsvnrvcrrhnndpsvvssvvsnvvnhqvsvllsllqllqslvqvvllvvlqcvvvvvvvvvvvvvvvvvvvvvvlvvvlpdvpppppdddddddddddddddddddddddddddddddddddddppppddddvpvvvvvvvvlsvvlvvllvvcpdpvvvvvlvvllvvllvllqpfffpddpvsvvvsvvslvvslvvlvvvlvsccsnspvvslvvdpvsvlsvvlsvlsvvlvvcvvvvpddllvnlvsvlsslssvvvvcvvdvvsvlvvvlvvvcvvvlvslvvvlvsllssllssqcslwanllddddpdddqqhsrhslsssvnlvcllvvaclvvvlssqlvsqpgpdpvssvsvvssvcsrvvsvssvssslssslvsslvsvvvvvvvvvvvvvvvvqvvvqvvvddddddddddddddddddddddddpdddddddddddddddddddddddddddddfaaadddddddddddddppppddddpppppddddqdpfdqdaavnprgppdpvlvvlsvllpdpvnlvvvvvllvvllvllqqdfqffppdvsvvvsqvvlvvvvvvpddddppddrddplvvslvvlvvslvslcsnhhddddprhqcpdplsvlsvvlsvlsvvlvvppdppdpvslvsvlsvlsscvnvcvspvvlvvvvvlvvvlcvvvvvlvvvvvsvllsllssqlsqfrqfqkakppnvdwapvrqfdwdwddpppdpvrtdidtidtdgppqgssrsvsssvnlvllqlvypvvvvlqsslqgddhrthgdgrphsvcvvssvvsnvvsvssvssslssslvvvvvvvvvvvvvpapddpvlvvllvcllpdffdaadqdpdpvlnvllcvlpwpqvvvvlvvllvvlsvlssprrppddpvnvvvsvvslvvslvvlvvslvslcsnqppvgqvvdvlsvlsvvlsvlsvvlvvlvvvvvccvvvvvvvvvvvvvvcvvvvvpdddddddddddddppvvvvvsvvvssvscsssssssvsvcvvvvvsvvsvvllvvlcvvpvslvvvlvsllssllsvqlsqflffaadppacrhpqagssrsvnssvnlvsllsspplvrvlvsqawagathpsnddppppddghighhpcsvvssvvsnvvsvsssssssvsssvscscvsnddpladdcvlvvvlsvlvcvqcnrrpqkhalvcvlvslccrdpplgdhvpddslvsllvllllqqfadpvnmggnsqsslssscssvvqqsdddlqvslvvvlvsccvppvpsdvvsscnnrnhddddtdtssnvslvvvvvvvvvvvvvvvvvppddddddddddvpvvvvvvcvvcvvvscvssvsdcpvvvvvvvvvvvvvvvpddddddddddddddddddddddddddddddddddddddddyddddddddddyddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddyddddddddddddddddddddddadddddddddddddddddddddddddddddddddddydddddddddddddddddddddddddddddhddddddddddddddddddhdhdddddddyddddddddddddddpddddddddddddddddddddppppppppsppnddddddddddrddddddddddddddddddddydddddddddddddddddddddddddddddddddddddddddddddddddddyddddddddddddddpdlvvvlvvcvvvvvnvlspdpvssvvvsvvvvvvvpqdpvrvvvvvccvvvvddddddddddddddddyddddddddddddddddddddydddddddsddddddpd\n>P02458\ndddddppvvvvvvvvvvvvvvvvvvddpppqqweddprdidghqdwddpaqqkiwgrhrhdididghdeddpppapdfdapvshdytdrppppppddddddddddddddddddddddddddpdppdpddddappppdpdppdpaddfdpppdddddppdpdddddddddddddydddddddddddddddddddddddddddddddddddfdddddddddddddddddddddddddddddddddddddddddddddddddddaddddddddddddddddddddddddddddddddddddddddddddddddddddddddhdddddddddddddddddddddddddddpdddddppdpdpppdpdppdppdpdddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddpddddddddddddddddddddddddddddddpdddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddadddddddcddddpddddpddddhddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddpdpddpdddddddddppdddddddddddddddddydddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddpppdddddddddddddddddddddddddddddddddddddddddddddgdddddddddddddddddddddrddddddddddfdddddddddddddddddddddddhddddddddddddpddpddddddddddddddddddddddddddddddddddddddddddddddddpddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddpdddddddddddddddddddddddddddydddddddddddddddddddddddddddydddddddddddddddddddddddddddddddddddddddpddddddddddddddpddddddddddddddddddddddgdddddddddddddddpdddpdppppdddpddddpddpddddddddddddddddppdddddddddddpdddpdppdddddpddppvvvvvvvvvvvvvvlcqvqpplalvrralallsncvscvvdawdwgwhdlpddpsplififtrhsvvsktwwakvvqkfdkdqfaadpdpqfdwdkqcpgtdvgrwidttdpvhdsvssvvslvsllvvfqkkkwkkkkkwaqaaaaaqpvvrhgpqhkwwafqvrdiagcdddpqrhkdwpddqrrhndrditmtmimdmdsrsnrddttimigrghndpriiimmrtttmigd\n>Q61410\ndddddddddddddddddddpddpvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvppddddddddddddddddddddpvpppppddpdfdkdwppdldddddppddpdpdlvvlfddddpvqlvlllvqlcvdpqsvvddpqlsnsqsrrwdkdkdaafdwpaafpaqqfkkkawadakkwkaqpndtddiddhshidscccllqvdhdriiiggnhitmmtidgsvsslvslvvvlvvqlvvqlvllclqplrvpddlllslllslqwdkdwdaafdwpaafqdfffkkkawqdakkwkwaddppdpdtdtddmdghsdidplvslqdrdtdrtiiggrhggtmitmdgnvlscfrnrgsvvsvcvvvvvvvvvvvvvvdppdddppppdppcpvqppvlvvllvvcvpddlacsvvqkdfdawqdaddfftwtwidgppdqdiwiktkgwpvvcvvssclvlvvqlvvvlcpddapqakhfrdwhdyqttimttihqaqlafqlvvllvvqaddlvqllqqllsllvvlvrqvvqqkaqaeddsrqwghhqlrhthgddrsvmdhddnsdfdqiqmyrlllaalcnsssstdgnlssllsslqsslcnqrvdgqfddpdsvssnvvlvvwpvpddrdpsddpqssvlsnqsgdnsssshqqntpvhsvsvcpgpsnpppdsvcvrvvvdrnpsnddapgssgnvrgdhdddddddgdgddpcsvvvd\n>P13368\ndddddddddddddddddddddddddddddddddaddddppddddddhdddddddddddddddyddddddddpddddddpdpdpvvvvvvvvvvvvdddddddddvvvvvvvvvvrvvvrvprvppdpddddddddddddyyddddddddddddvpvvvvcvvvvpvvvpppppvvpddppdddddpdddddddddddddddddpddddlvvllvcllvvllvclvvclvvccpqavppppddpdpsvllnvllnqlsvqlvvcvvvvnqqpvsqddpdddpsnlsnslssvsslvsslvvvcvscqvvqqpqafdqpppappqgdddfgwgfrqdqddpvppppddkwkkkfwdpcvqvvvpppdpddddpdprqdtfidtqddwdadnrrgigrpdddqaqtkmkmwiwdqsdddpvsirihdiydidthhqdaafaffwdwpdwaalaqfkiktfiddgprgshqwqwkkkwkawppdrdididtggrpdrmdmggrdhgqtkmkmwmwigdprgiydididiythhhddpdddllvqfqwwwwdfqfwikidgppppddidtqgghpftwqekdadlvqqkiwtagpqqwiwifhddprhtpddidtadadpdvvdpadkgwryweaqpqvqkiktwiwgppdpfikifiwmdhprnydidgqadidrfdwrywyddnqqqkiwtdtqffiwiarsvvsdidtldgdgrfadkdaqnsnqwiwtddlvvqwtwtahlnsrhtdiaghdhqdddddpdppdrpqqdwddrpqkiwtddqfkiwiwhdddrdigtafmdtrpprrrtrdmgtsgpsvdddhdafaewadwakdaaffkikifthghdddpsghpshqaqwwkkkwkafpvvrdididtrhrdrmdmggrdhgqtkiwmkmwtagpvrhthdiddididtyfhddqwkkwfafqfqwiwidgsqqddtdtqpdgdhglwadwedlfpfkiwtahpqawidiagspprvddwppngtdhqwqewddlqalqwiwtqrnssqwiwianrhvrnidtdphgqfrywdadnvqlkiwtfhqfwtwidfldpvlidtlggddppqwgwqekdadppqqkiktwidgsrfiwiwmfgnddpppdsvrgdilddgdaarrrawdappsqcwikgagpvlqkiwiahpvgrvnidirgndppdggtsymdiddpdtrdhhnqqqewedfdqvqfawdaddqqwtkgagdgtgrnvraqkwkwkwkdfdpdididtgsdrmdthhpdpdafgkikmkiwihgssdiddihiymyghhddaaaewadkdkawdwcdappddtktkmkifgggddddpdddppfkkkwkwffqqhhtfdididdppdgmdmfigfdfagkfwtkmwidrrpppphthpidfididhqfaghqpkwwfdfqffiwmagfvvrdigtddtdpadwpawdadvsqckiwtagpqqwiwidhppddiatqehddadwadweadnqqgkiktwgddpvapqkiwiwidgplpppvdrddtdtldmddrqkdwhdwyddqqqqktktwidgppcpqwtfiwmarnpprdtdpppppfqwdadddpdddddddwgqtagprrttwiagpvvrdrdpcqqvlqvvvvpdvnpdwdweaaafwtwicdpqwtwidtrrpldtpgidghdprpgdhmydphddddpdplqqfaepdadqqwdwddaeqwwtktwaafdtrdpvnprghhakkkkkfkffdddpddpddppdphgpdididdhgdmdiggghhafgkikmwiwmdgdvcvvvvhdtdthdmdmdgydwdaffwffdkdkaddalwwmwmftgdgdddrddakwkkkwkdwddddddppddddddidmdtgrgttitidtghhrftkmkifmwmtddpvhtdgddmdmtghhdddddwdfddaeqfwgkifdddddaawdakwkwkdwpvdtdtdgppdpdgmdmddghhgftkmkmfmwiahdddsrddtdtgdmdmdtyhwaawaewaekrkaddddqkikmftgdtdgthfdwdkkwkkkkwfddppvvvvvppvpdddddddddpppddddgddididtddidpdgmdmdgdppqpikmkmwmwiagdphgtydiydiydidhhdddddpcvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvpddpcvvvvvvppdppppvvvvvplpdfdrvvvvvvfaedecvqkdfddwpdakpqgtktwifgddvvdpgtatwikrwgdqpdpcvrlvvsqsvvlcpdddpqakhfdgwypndstiitithdqpcafqlvnlqvldddpddddppddrddplvllqallsvlrqllvcvsvfhfqqqqasnqkgwhddppvdpddihihghdsrddddddvqqwdqdppdgthrqllfalcclpvvdtgslrvlssslvrsvcslqsnddrppvdgsvvsnsclvsvhdddarpnadplrvvlsvlssdndsvshdrsvrssvssvvvsvvvvvvvvvvvpppdddddddddppppdddcsrsydpdyrhrrsnhsppnhdddssdpdppdddddpvd\n>P04808\ndvvvvvvvvvvvvvvvvvvvvvvvvvqqqdwaadadpvvvvllvvlqvvvvvvvvvcvvppdpdddddddddddddpddvvvvvvvvvvvvvpddpvvvvvvpdddddddppppppdddpddpddpvvlvvvvvvvvvvppddddddpppppppvvppvlvvvvvsvnvccnrptdgsnvsnvnd\n>Q04721\ndddddddddvppvvppppddpddddaaapdqqppappswgwdddpsrhtftdedpqfaagrspegapppdpqappdwdwdadrhpndiatdddpqfddrsrddhddaccvppvqapppwdwadpdrpdiatdedpqfddrnsdagdpcvvvqapdpwdwdddrndiatdedqqfddrrrpdgddpqvdpdqapppwhwdddrsdifthedpqfddgrrddgdfdvvphqadppwdwddpdrhdiatdedqqfddrrspdgdapfdpdqadafwgwdgdrpdtfifgqpqadwrntfggddvcvvdvqqqffqfdwdddgpdiathfdaqfddgrfdagdppvfpddqdppwdwdddrndiatdadpqfaarrspegapcvvvqapdpwdwdqdrphrhiatredqqfddrrgddgppvcppdddgqqplpfhwdddrrdiagrddqqaddrsrpdgdfpcvvvqapppfdwagdrnhiagnadqqfddrrrphgdfpcvvvqapppfdwdgdrnatqthedpqfddrrrpdgdwpcvvvqqpdpwdwdtdrpdiathddpqfddrsrpdgdfppvppdadqfdwdtdrvdiagdhdqqfddrrgdhgdfpcpvvqapppwdwdgdrvdiatdedqqfddrsrpdgdfpclvvqaqqddwdgdrvdiaddedqqwwfsntddgdfpcplvleaffwdwdddrpdifidaddddddrhdddddwdlvvlladqfhwdddppdiatdhdqqfddrsrfdgdfvcvvvlapppwdwdtdrndiatdedpqfddrnspdgpfpcvvvqapppwdwdgdspdifthedppfddrnrpdgaadcvpppadppwdwdadpvrpdtatdedqqfddrrtddgdfcvvvlqadppwdwdddgsaiatdedqqaddrrspdgpppcvvvqadqpwdwdrdrnaiatnddqqfddrrspdgpfpcvvvqapppfdwdgfrndiqtrddaqfddrrspggdfqddpqqapppwdwdgdrndiathedqqfddrrrpdgdfpclvvqapppwdwdgdrndiatdedqqfddrrrphgddpcvvpqapppwdwdddgndiatheddqaddrrspwgqaalcvqqvvvvhdscpsapprwhwddpdstiftdddqqfddrrrpdgdfpcvvvqapppfdwdddrrdiathddqqfddrsspdgdspcsnpqadqpwdwatarnaiatdagaqfddrrspdrnfpcppdllapppfdwdggrhdiatrddqqfddrrsrhgppcvvvlladplffpdwhddrnaiqghgqqqfddrrspdrhdccvvllapppwrwdadsndrnriatrddpqfddrnspdgpvpdddpdqwdwdqdpngtdthrpddpvddapcpvvqapppwdwdadpdpvriathddppfddrrrpdghqdddddadalpdpclvvqqppldqdpsqldlnnqssvcnlqqndsallpqppasdpllvvldldddvsccdlsnlrslvspppdddalppqvvllsqqppldqspsqcdlnnlhslcsvvpvpdfdwaffkkkwwfqdapvvcvvcvsvvqrqlrssqsfgkdfdadpvrdgpkdfdqddpdppcvvpdddddddddppdrdrrtiitimitgcrsvvvvdprhrrhvsssssssssvssvvpdpthirhidgrrcppppppvvvvvvvvvvvvvvcvvvvvvdddddddddddddddddddddddddddddyddddddddpddpddddddddddddddddddddddddddddddddddddddpddpplpppdddpvlcvllvddddddddddddddddddddddqcrqdpqrdgsllsllqddpddddhdpdvvcnvvslvsnvvsvvvpydqcragppwrdgsllnnllnlvlssnvssvvsprqqldqipfqdgsllsnlvnvsvssnvvslvpprhdqlrqtpqqdgsllscllvvvlvsnvvsvvspdplcrdtnqqdgsllnnllvlvqnsvvvsvvsprdqcgatnfrqgsllnnlqnlnlnsnlvsvvsprdqcrdtpvndgslvnnvvvvsvvsvvvvvvsvdddddddddddddddddddddddddddddddddddddddddddpdddddddddddddddddddddddddddddpdpddddddddddddddddddddddddddddddddddddddddddddddddddddddtppdddddddddddddddddddddddppdddddddddddddddddddrhdddddddddddddddddddddddddddddddddddddvppvdstgrrhdrhdddddddddddddddpppdtddddddddddpddppppddddddddddddddddpdddpppdddqddppphsdppddddddggdddddddddddddrdddddddddddffddddddddddhddddddddddddydddgdddddddddddddddddddddddddddddddddddddd\n>P07949\nddddddddpppvvvvvvcvvvppppppqkfwqaqeaewedeafdafpqwadatdiagqdppdawfkdkdqfwafpprqgqdgdpfwdadgrgriiggrgghhpvvvvsscvscvvhpgqkikikmatdpdpddpvdadpggihiymygydhhhdddlvpddqvnqwddpdadetedeapgdkdwndaghrrnscvrpvvkdkdkdkpddppaqwgqdhprrtiignggddvlqpfkdwikmwmwidddpdidididihiygydfaqawawaadvraqedefeaefaqaffdwttkgkiwfqgpppfavccpvfwdkdwddddpvcvqfkdkdkdkdwdwddhpndtgiiimmmiiitthhgddladfdwdkikmktagdrhphpdgsigihiytytyhhddqdfdqedefefellaaflawtdkgggdplqshdswqkfkfkdwpdpqqpfktwgadpnsrmiiitghdsvslndpvnqktwikmkiatppprdidihiyiygnddddddfdppadqallqdqdpvsqqnaahhnarnngwdwqdadqpfwalratgiasdpppqspldcdpscvvdcrhhvqndfdadeaagffagvvghtrtahhtkgddsvvsyiygdhddpddddddpvvvvvvvvvvvvvvvvvvvvvvvvvvvcvvpdddddddddddddddddddddpdpdddddddddpppddddddpvppqddddvvfaddvvqkawddwpdadllatktwiwgacqvndhdidtwikgfhdppddplsvllvvqqvvvlvvdddqqakhwrhwhrpddtiittihdqpqafqlsllvvlvvldpppppdddddddppdddppvshddvllllvqlllvlvvllrcvvvqkfqqqqarnqwgqhpprhthggssspmdhppppdpddppdpddrplllfalccvpvvdtgslrvlssslvrsvcsqvsnddhpaqddsvcvsvcvvvvddddddpsddplsnvlsslsrdndsvsrdgsvvvsvssvvvsvvvvspddpvvpdpdppdpddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddhrdhdddddddd\n>P35498\nddpddddfaeqvlwaaddpvllvvlvvllvvvcvvpvddddpddpppdddwdpqqdafdahdcrnpdhdpvqalgwtdrptpvcsnaqkiwhahprrgihiagqaaavnprgcprpvlsvlssllrdpvnlvvlvvllvvvlvqlqdldrdpvcvvvvvvslvslvvvvvsqssnhhdpddprhqvppplsvlsvvlnvlvvclvvdppvvvlvsvlsplsslvcvcvrdppsvvvvvllvqlvvlcpllvvvlvlvllllllllllffflllqkfwfadddpvpddddppfpwpfdadpvrdtddtdgddpppqvrqadcvrtdddfpdpdgdgaepdpqaadddprtdidrgyagplrnlqgsffsvsssvllvclllvflvvvsvnsrcnrrglvcvvssvvscvcrnlfsvlvsllsslvslvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvppdddddddddddddddpdppdddppvvvvvvvvvvvpvpddddddddddddddddddddddddyddddddddddddppdddddddddddddddddddddpppppdpdpppddddddddfffddfdddddddddddddpdddddvpddddddddddddddddfdddddddddddadddddpdddddddgdddddddddddddddddgddddddddddddddddddddpvvppvvvvpdpvvvvvvvvvvvvvvvvvvvvvvvpppddpvvvvvcvvpqdpppdpvlvvvlvvllvvlpqlvvvvvlvvllvvllvllqpffppddpvsvvvsvvslvvslvvlvvslvsvcsnqtpvvlvrdpvsvlsvvlsvlvvvcvvcvvdppvllsvlsnlvsnlvcllvdvvsvllvclvvvlcrvcvslvvvlvsllssqlssqqsqclvlqqvvvcqpdvvsdrdqqgssgsvsssvqlvclllshnvsvlssrcrrrnnpvsvvssvvsnvsnnssvssslssslvvsvvvnsvvhddppppprsnvvvvqvvvvvvvvvvvvvvvvvvvvpddddddddddddddddddddddddddddddddddddddddddddddddddydddddddddddddddddddpddddddddpddddppppddddddddddddddddddddddddddddddddddddddddpdpcdvvpqfedqqddpvccvvdvvsdddcpddvnvvlrvvqsvlqcqlpppvnvvvvlvllvvllvllqppfpcclvvvvsvvvsvvslvvslvvlvvslvsccshhhdpslvvdplsvlsvvlsvlsvvlvvcvvvvvcvdpvslvsvlcslssnvnvcvvpvlsvlvvvlvvllvvvlvslvvslvsvlssllssqlsqwaqfqkfkdqvvsrdtddcvqfffpvslvvcvvvvgridgdgdlqgssrsvssvvllvclllvhpvvshqqssqqtddhgthghgrpnvvcvvssvvcnvvnrsvslssllssslvscvvvcsvvsvddrsfdpvrvvvlvvlvcvlvddfdfpqdwdppvvlnvllvvlpdpvnvvvlvvllvvlsvlssprgnpddpvsvvvsvvvlvvslvvlvvslvslcsnrpvplvvdpvsvlsvvlsvlsvvlvvvvvvcvvppddcsvssvsvssssssvvvvcvvdpssvlsvslvvslvnsvvslvvvvvsvlssllsvqlsaqlqadddfqddpsqgssgsssssvnlvcclsphclvrvlvslsfadppswfqadddpsdphgirhhdnvvscvssvvsccprpspsssnssssssvsvvvsvvqvppqdhsvqvvqlsvlvcvqcnvppqkhapvcccvslcsgdppshdnppclvvqlvlfffafppniggslssvlssvcvrvpddpvsvvvvvvvvvvvcsvdvcpprpdgdgtsnrvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvppddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddd\n>P00533\ndddddpdpvvvvvvvvvppddpdpqddaaefaaddcqadadddlvvsvvvlclrqqshaeyleeaaaepaeapddqvslcnhaeyaaayhaehfayaencvqrhaeyahlyadvraareeheqqadpvlhghaaylpqrqqeyeggeyedynnqrhaqpvqdpcpryydpvnpvvyyddhnhpdppadfddppadsshfrghdpsgrhdgfavqddppapggfphhdpqthadpqagrgfpghaqlthpdgnaadersghdpdfaaqwfqaqvvrdthgdprgfaddgrythpdadpqwaaeprrhtfnaedqqwhwdddpngtytdgdppqpededeeqcddvnvvppadapvplvrlashaeypeeheaapcrqvpdpvvnthhhdlvsllrllnyqyyaaayhhpehdlpdqesnsqlnhaeqanphadvnaaryyaeqyqhqfhnnqnhqahqdheyedaqnanhafpvqapcvrrynddpyyyhydnyhdnvvcvvvvfadepqadsrtfrghaqlthndgpaadavrgrdpdaqfqhdppafdddprythgedpqfdrhhrggqfddhfqlggpdgpaadedsgrdpaaqfqghhppragwaweqdpvrytdtedrvqrrhfhdhdcvrhdpdddddddddddddvpvvvvvvvvvvvvvvvvvvvvvvvvllvvlvvlvspqddpdpplqafgpsqadadepvqkdfdawqdaadqatktfiwghpppdpatetkikgwgdddpdpvvvvlqvsllrvqsrdddqaawhfphwyddptiititfdqlqffqlvvlqvqvvqdfllllllvllsvlrqlvscvvsqkflqqqaslqwghndsngihghnssvidhagppgqwdadddddhplllfalccvppvtthslrvlssslvrsvcslqsnddrpppddsvcsnvclvvvhddddrpffdpvsvvlsvlssdndsvshdhsvvssvvsvvcsvprcvgtpdppnvvnddddpvddvvvvvvcpppsrprydypccsncvdppppdddpdddddddddddddddddddddddddddddddddddddhhrrddpppppdddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddppdpddddddddddddd\n>P00441\ndkwkkkwwkddpapktwikiwidhdqqawikiwikiapaaafkwwkwwacaldqvvhppvlhfgaalppfwdfeqpdpghgltgpfifgahnrrmttdtdigrsagcddpnrqarikmfianagqcglpppdpvsrggspnhdtrimtgihtdd\n>P01375\nddpvvvvvvvvvvvppddpppddpvvvvvvvvvvvvvvvvvvvvvvvvvcvvvvvpddpppppdvpppcppvvvvvvvvvvpdddlwqkwkwfadpppafdgatdqpdpptdgdnqwdcdpqwtfdaawakkkkkwkfkfkdwfaaqdwdkkkkfkwkqapvhrdtdtpwmdidgqdpdgddppdgghmdmdmdmtidmdtghggmimhidmdtsnrtdppdppntiimmtgd\n>P0A7L0\ndddddplvvvqpvpadppdadallvqlvsqlvsppdpdfwfkkkkfaflfqlvdplqwfkekfqqqqaapdqfaeaeedddplqvllvvlphphydaqvvlvvvvvppdpgdeyayepvrvvrlvvccvpcvvvvrrddvrhvrhdpnnsvvsnsvrshiwiwtadnrrmtmttqytspddsvsslssvlrvvlvqvvsgrpprpdcrddwmwmhipphdtrthdpvnrpnpdd\n'


def _has_benchmark_data(prost_dir: Path) -> bool:
    data_dir = prost_dir / "prostT5_benchmarks" / "benchmark_data"
    return (data_dir / "test_set_AA.fasta").exists() and (data_dir / "test_set_3Di.fasta").exists()


def _candidate_prost_dirs(start: Path) -> list[Path]:
    candidates = []

    for env_name in ("PROSTT5_DIR", "PROST_DIR"):
        value = os.environ.get(env_name)
        if value:
            candidates.append(Path(value).expanduser())

    for base in (start, *start.parents):
        candidates.append(base)
        candidates.append(base / "prostT5")
        if base.name == "prostT5":
            candidates.append(base)

    candidates.extend([
        Path("/content/Speculative-Decoding-ProstT5/prostT5"),
        Path("/content/Speculative-Decoding-ProstT5") / "prostT5",
    ])

    seen = set()
    unique = []
    for cand in candidates:
        try:
            resolved = cand.resolve()
        except FileNotFoundError:
            resolved = cand.absolute()
        if resolved not in seen:
            seen.add(resolved)
            unique.append(resolved)
    return unique


def _materialize_embedded_benchmark(prost_dir: Path) -> None:
    data_dir = prost_dir / "prostT5_benchmarks" / "benchmark_data"
    data_dir.mkdir(parents=True, exist_ok=True)
    files = {
        data_dir / "test_set_AA.fasta": EMBEDDED_BENCHMARK_AA_FASTA,
        data_dir / "test_set_3Di.fasta": EMBEDDED_BENCHMARK_3DI_FASTA,
    }
    for file_path, text in files.items():
        if not file_path.exists() or file_path.read_text() != text:
            file_path.write_text(text)


PROST_DIR = next((cand for cand in _candidate_prost_dirs(NOTEBOOK_DIR) if _has_benchmark_data(cand)), None)
if PROST_DIR is None:
    # Colab often starts in /content with only the notebook uploaded. In that
    # case create the benchmark data locally, then route all outputs there.
    fallback_prost_dir = Path("/content/Speculative-Decoding-ProstT5/prostT5") if Path("/content").exists() else NOTEBOOK_DIR / "prostT5"
    _materialize_embedded_benchmark(fallback_prost_dir)
    PROST_DIR = fallback_prost_dir.resolve()

if not _has_benchmark_data(PROST_DIR):
    searched = "\n  ".join(str(cand) for cand in _candidate_prost_dirs(NOTEBOOK_DIR)[:20])
    raise FileNotFoundError(
        "Could not route to prostT5 benchmark data. Set PROSTT5_DIR to the repository's "
        f"prostT5 directory or run from the Speculative-Decoding-ProstT5 checkout. Searched:\n  {searched}"
    )

DATA_DIR = PROST_DIR / "prostT5_benchmarks" / "benchmark_data"
RESULTS_DIR = PROST_DIR / "results" / RESULTS_SUBDIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TEST_AA_FASTA = DATA_DIR / "test_set_AA.fasta"
TEST_3DI_FASTA = DATA_DIR / "test_set_3Di.fasta"

AA_LETTERS = "ACDEFGHIKLMNPQRSTVWY"
HMM_SMOOTHING = 0.5

print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Project dir : {PROST_DIR}")
print(f"AA FASTA    : {TEST_AA_FASTA}")
print(f"3Di FASTA   : {TEST_3DI_FASTA}")
print(f"Results dir : {RESULTS_DIR}")
print(f"Protein limit: {BENCHMARK_PROTEIN_LIMIT}")
print(f"K values    : {K_VALUES}")
print(f"Family MSA max homologs: {FAMILY_MSA_MAX_SEQS}")

assert TEST_AA_FASTA.exists(), TEST_AA_FASTA
assert TEST_3DI_FASTA.exists(), TEST_3DI_FASTA


Notebook dir: /content
Project dir : /content/Speculative-Decoding-ProstT5/prostT5
AA FASTA    : /content/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/benchmark_data/test_set_AA.fasta
3Di FASTA   : /content/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/benchmark_data/test_set_3Di.fasta
Results dir : /content/Speculative-Decoding-ProstT5/prostT5/results/prostT5_probabilistic_drafter_folding_100
Protein limit: 100
K values    : [1, 3, 5, 8, 11, 15]
Family MSA max homologs: 16


## Load ProstT5

Load the same full ProstT5 encoder-decoder model, but use the folding prompt `<AA2fold>` so the model generates 3Di tokens from AA input.


In [ ]:
#@title Load ProstT5. { display-mode: "form" }
PROSTT5_NAME = "Rostlab/ProstT5_fp16"
try:
    tokenizer = T5Tokenizer.from_pretrained(PROSTT5_NAME, do_lower_case=False, legacy=True)
except ImportError as exc:
    raise ImportError("T5Tokenizer needs sentencepiece. In Colab run: !pip -q install sentencepiece") from exc

dtype = torch.float16 if (USE_FP16 and device.type == "cuda") else torch.float32


def _load_prostt5(**kwargs):
    try:
        return AutoModelForSeq2SeqLM.from_pretrained(PROSTT5_NAME, dtype=dtype, **kwargs)
    except TypeError:
        # Older transformers versions use torch_dtype instead of dtype.
        return AutoModelForSeq2SeqLM.from_pretrained(PROSTT5_NAME, torch_dtype=dtype, **kwargs)


# This notebook is GPU-only by design. Use device_map on CUDA to avoid an
# unnecessary full CPU copy during model load.
model = _load_prostt5(low_cpu_mem_usage=True, device_map={"": str(device)})
model.eval()
model.config.use_cache = True
model.generation_config.use_cache = True

encoder = model.get_encoder()
DECODER_START_TOKEN_ID = model.config.decoder_start_token_id

print(f"ProstT5 loaded. dtype={dtype} params={sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print(f"Decoder start token ID: {DECODER_START_TOKEN_ID}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.40k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/283 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/5.64G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.64G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

ProstT5 loaded. dtype=torch.float16 params=2818.9M
Decoder start token ID: 0


## Load Paired Benchmark FASTA

Parse AA and 3Di FASTA files, match records by UniProt ID, and keep only proteins where AA and 3Di lengths match.


### What this cell does

Loads and pairs the benchmark AA and 3Di sequences.

- Parses `test_set_AA.fasta` and `test_set_3Di.fasta`.
- Matches records by protein ID.
- Keeps only proteins where AA length and 3Di length agree.
- Stores valid pairs in `paired`.


In [ ]:
#@title FASTA parsing + paired dataset. { display-mode: "form" }

def parse_fasta(path: Path) -> dict[str, str]:
    records = {}
    cur = None
    with path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                cur = line[1:].split()[0]
                records[cur] = ""
            else:
                records[cur] += line.strip()
    return records

AA_SEQS = parse_fasta(TEST_AA_FASTA)
DI_SEQS = {uid: seq.lower() for uid, seq in parse_fasta(TEST_3DI_FASTA).items()}

paired = {}
for uid in sorted(set(AA_SEQS) & set(DI_SEQS)):
    aa = AA_SEQS[uid].upper()
    di = DI_SEQS[uid].lower()
    if len(aa) != len(di):
        print(f"skip {uid}: length mismatch AA={len(aa)} 3Di={len(di)}")
        continue
    paired[uid] = {"aa": aa, "3di": di, "length": len(aa)}

print(f"Loaded paired benchmark records: {len(paired)}")
print("First records:")
for uid, rec in list(paired.items())[:2]:
    print(f"  {uid}: L={rec['length']} AA={rec['aa'][:12]} 3Di={rec['3di'][:12]}")


Loaded paired benchmark records: 100
First records:
  A0A6G0XC32: L=163 AA=MEKLSSLSFAYC 3Di=ddfdaedepacc
  O60885: L=1362 AA=MSAESGPGTRLR 3Di=dddddddddddd


## Tokenization Helpers

ProstT5 uses task prefixes. For folding we use `<AA2fold>` and generate lowercase 3Di tokens.


### What this cell does

Defines tokenization helpers for the folding direction.

Important objects:

- `_format_aa(seq)`: formats AA input with the `<AA2fold>` task prefix.
- `_decode_3di(token_ids)`: decodes model output back into a compact 3Di string.
- `THREEDI_TOKEN_IDS`: maps each 3Di symbol to its ProstT5 tokenizer ID.

`THREEDI_TOKEN_IDS` is the bridge from the drafter's 20-token 3Di distribution into the full ProstT5 vocabulary.


In [ ]:
#@title Folding format helpers. { display-mode: "form" }

def _format_aa(seq: str) -> str:
    return "<AA2fold> " + " ".join(list(seq.upper()))


def _decode_3di(token_ids: torch.Tensor) -> str:
    if token_ids.ndim > 1:
        token_ids = token_ids[0]
    s = tokenizer.decode(token_ids, skip_special_tokens=True)
    return "".join(s.split()).replace("<AA2fold>", "").lower()

THREEDI_ALPHABET = "".join(sorted(set("".join(rec["3di"] for rec in paired.values()))))
THREEDI_TOKEN_IDS = [tokenizer.encode(f" {tok}", add_special_tokens=False)[0] for tok in THREEDI_ALPHABET]
THREEDI_TOKEN_ID_TO_IDX = {tid: i for i, tid in enumerate(THREEDI_TOKEN_IDS)}
THREEDI_IDX_TO_TOKEN_ID = {i: tid for i, tid in enumerate(THREEDI_TOKEN_IDS)}
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_LETTERS)}

print(f"3Di alphabet ({len(THREEDI_ALPHABET)}): {THREEDI_ALPHABET}")
print("First 3Di token IDs:", list(zip(THREEDI_ALPHABET[:8], THREEDI_TOKEN_IDS[:8])))


3Di alphabet (20): acdefghiklmnpqrstvwy
First 3Di token IDs: [('a', 128), ('c', 147), ('d', 135), ('e', 134), ('f', 140), ('g', 130), ('h', 145), ('i', 137)]


## Reference Folding Generation

This is the plain ProstT5 `AA -> 3Di` greedy generation path. It is the verifier/reference output used for correctness checks.

#### Maybe we can cache it so we can reuse the 3Di

In [ ]:
#@title Plain enc-dec folding reference. { display-mode: "form" }

@torch.inference_mode()
def generate_folding_reference(aa_seq: str) -> str:
    L = len(aa_seq)
    enc = tokenizer([_format_aa(aa_seq)], add_special_tokens=True, return_tensors="pt").to(device)
    out = None
    try:
        out = model.generate(
            input_ids=enc.input_ids,
            attention_mask=enc.attention_mask,
            max_length=L + 2,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )
        seq = _decode_3di(out[0])[:L]
        return seq
    finally:
        # Building family HMMs folds many homologs; release transient generate() tensors promptly.
        del enc
        if out is not None:
            del out
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        elif device.type == "mps" and hasattr(torch, "mps"):
            torch.mps.empty_cache()


## Family-Specific AA-Conditioned 3Di HMM

Build a per-protein drafter from that protein's MMseqs2/ColabFold AA family MSA. Because the API returns AA homologs rather than 3Di labels, the notebook folds a capped number of homolog rows once with ProstT5, projects those pseudo-3Di labels back to query-aligned MSA columns, and estimates draft distributions only from that target family's projected rows.


### What this cell does

Builds the family-specific folding drafter machinery.

Flow:

- Fetch or load an A3M MSA for the query protein.
- Select homolog AA rows from that family.
- Fold homolog AA rows with ProstT5 to obtain pseudo-3Di labels.
- Project pseudo-3Di labels back onto query-aligned MSA columns.
- Count per-position 3Di tokens and convert them into draft logits.

This is not a true Profile HMM over 3Di states. It is a family-local pseudo-3Di drafter.


In [ ]:
#@title Family-specific AA-conditioned 3Di HMM from MMseqs2 MSA. { display-mode: "form" }

COLABFOLD_HOST = "https://api.colabfold.com"
COLABFOLD_HEADERS = {
    "User-Agent": "prostt5-folding-hmm-family-notebook/1.0 (educational use)"
}
COLABFOLD_RETRY_STATUS = {429, 500, 502, 503, 504}
THREEDI_SYMBOL_TO_IDX = {tok: i for i, tok in enumerate(THREEDI_ALPHABET)}


def _log_normalize(x: np.ndarray, axis=None) -> np.ndarray:
    m = np.max(x, axis=axis, keepdims=True)
    return x - (m + np.log(np.sum(np.exp(x - m), axis=axis, keepdims=True)))


def _cf_request(method: str, url: str, *, max_retries: int = 8,
                base_backoff: float = 10.0, max_backoff: float = 120.0,
                **kwargs) -> requests.Response:
    """ColabFold API request with quiet exponential backoff."""
    kwargs.setdefault("headers", COLABFOLD_HEADERS)
    resp = None
    for attempt in range(max_retries):
        resp = requests.request(method, url, **kwargs)
        if resp.status_code not in COLABFOLD_RETRY_STATUS:
            resp.raise_for_status()
            return resp
        retry_after = resp.headers.get("Retry-After")
        try:
            wait = float(retry_after) if retry_after is not None else None
        except ValueError:
            wait = None
        if wait is None:
            wait = min(base_backoff * (2 ** attempt), max_backoff)
        time.sleep(wait)
    resp.raise_for_status()
    return resp


def _fetch_msa_colabfold(aa_seq: str, cache_path: Path,
                         mode: str = FAMILY_MSA_MODE,
                         poll_interval: float = 5.0,
                         max_wait_s: float = 600.0) -> str | None:
    """Fetch/cache an A3M MSA for one protein family using the ColabFold MMseqs2 API."""
    if cache_path.exists():
        return cache_path.read_text()

    try:
        submit_attempts = 0
        while True:
            resp = _cf_request(
                "POST",
                f"{COLABFOLD_HOST}/ticket/msa",
                data={"q": f">query\n{aa_seq}\n", "mode": mode},
                timeout=30,
            )
            sub = resp.json()
            status = sub.get("status", "")
            if status != "RATELIMIT":
                break
            submit_attempts += 1
            if submit_attempts > 8:
                print("    MSA API rate-limited after retries; using uniform drafter")
                return None
            time.sleep(min(10.0 * (2 ** (submit_attempts - 1)), 120.0))

        job_id = sub.get("id")
        if not job_id:
            print("    MSA API returned no job id; using uniform drafter")
            return None

        waited = 0.0
        while status != "COMPLETE":
            if status in ("ERROR", "MAINTENANCE"):
                print(f"    MSA API status={status}; using uniform drafter")
                return None
            if waited >= max_wait_s:
                print("    MSA API timed out; using uniform drafter")
                return None
            time.sleep(poll_interval)
            waited += poll_interval
            sr = _cf_request("GET", f"{COLABFOLD_HOST}/ticket/{job_id}", timeout=15)
            status = sr.json().get("status", "")

        dl = _cf_request("GET", f"{COLABFOLD_HOST}/result/download/{job_id}", timeout=120)
        data = dl.content
        a3m_parts = []
        if data[:2] == b"PK":
            with zipfile.ZipFile(io.BytesIO(data)) as zf:
                for name in zf.namelist():
                    if name.endswith(".a3m"):
                        a3m_parts.append(zf.read(name).decode("utf-8", "ignore"))
        elif data[:2] == b"\x1f\x8b":
            with tarfile.open(fileobj=io.BytesIO(data)) as tf:
                for member in tf.getmembers():
                    if member.name.endswith(".a3m"):
                        fh = tf.extractfile(member)
                        if fh is not None:
                            a3m_parts.append(fh.read().decode("utf-8", "ignore"))
        else:
            txt = data.decode("utf-8", "ignore")
            if txt.lstrip().startswith(">"):
                a3m_parts.append(txt)

        a3m_text = "\n".join(part.strip() for part in a3m_parts if part.strip()) or None
        if a3m_text is None:
            print("    MSA API result had no A3M; using uniform drafter")
            return None
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        cache_path.write_text(a3m_text)
        return a3m_text
    except Exception as exc:
        print(f"    MSA fetch failed ({type(exc).__name__}); using uniform drafter")
        return None


def _parse_a3m_records(a3m: str) -> list[tuple[str, str]]:
    records = []
    name = None
    chunks = []
    for raw in a3m.splitlines():
        line = raw.replace("\x00", "").strip()
        if not line:
            continue
        if line.startswith(">"):
            if name is not None:
                records.append((name, "".join(chunks)))
            name = line[1:].split()[0] or f"seq{len(records)}"
            chunks = []
        else:
            chunks.append(line)
    if name is not None:
        records.append((name, "".join(chunks)))
    return records


def _strip_a3m_insertions(seq: str) -> str:
    return "".join(c.upper() for c in seq if not c.islower() and c not in ".*")


def _ungap_aa(seq: str) -> str:
    return "".join(c for c in seq.upper() if c in AA_TO_IDX)


def _project_homolog_3di_to_query(query_aln: str, homolog_aln: str, homolog_3di: str) -> list[str | None]:
    # Output coordinates are query residue positions, not raw MSA columns.
    L = sum(1 for c in query_aln if c != "-")
    projected = [None] * L

    # q_pos counts non-gap query residues; h_pos counts non-gap homolog residues.
    q_pos = 0
    h_pos = 0
    width = min(len(query_aln), len(homolog_aln))

    for i in range(width):
        q_char = query_aln[i]
        h_char = homolog_aln[i]
        h_token = None

        # Homolog 3Di tokens are indexed by homolog residues, so gaps do not
        # consume a 3Di token.
        if h_char != "-":
            if h_pos < len(homolog_3di):
                h_token = homolog_3di[h_pos]
            h_pos += 1

        # Only MSA columns with a query residue map into the output list. If
        # both query and homolog have residues in this column, copy the homolog
        # pseudo-3Di token to the corresponding query position.
        if q_char != "-":
            if h_token in THREEDI_SYMBOL_TO_IDX:
                projected[q_pos] = h_token
            q_pos += 1

    return projected


class FamilyFoldingHMMDrafter:
    """Protein-family-specific AA->3Di drafter built from one target's MSA family.

    The ColabFold API supplies only AA homologs. To keep the drafter family-local,
    homolog AA rows are folded once with ProstT5, projected to query MSA columns,
    and counted as pseudo-labeled family examples. No unrelated benchmark proteins
    are used for this target's HMM.
    """

    def __init__(self, uid: str, aa_seq: str, a3m: str | None,
                 max_family_seqs: int = FAMILY_MSA_MAX_SEQS,
                 smoothing: float = HMM_SMOOTHING):
        self.uid = uid
        self.aa_seq = aa_seq
        self.smoothing = smoothing
        self.max_family_seqs = max_family_seqs
        self.n_family_seqs = 0
        self.n_projected_seqs = 0
        self.context_counts = defaultdict(lambda: np.zeros(len(THREEDI_ALPHABET), dtype=np.float32))
        self.logits = self._build_logits(a3m)

    def _candidate_family_rows(self, a3m: str | None) -> list[tuple[str, str, str]]:
        if not a3m:
            return []
        records = _parse_a3m_records(a3m)
        if len(records) < 2:
            return []
        query_aln = _strip_a3m_insertions(records[0][1])
        rows = []
        seen = set()
        for name, raw_aln in records[1:]:
            aln = _strip_a3m_insertions(raw_aln)
            aa = _ungap_aa(aln)
            if len(aa) < FAMILY_MSA_MIN_SEQ_LEN or aa in seen:
                continue
            seen.add(aa)
            rows.append((name, query_aln, aln))
            if len(rows) >= self.max_family_seqs:
                break
        return rows

    def _build_logits(self, a3m: str | None) -> np.ndarray:
        L = len(self.aa_seq)
        S = len(THREEDI_ALPHABET)
        counts = np.full((L, S), self.smoothing, dtype=np.float32)
        rows = self._candidate_family_rows(a3m)
        self.n_family_seqs = len(rows)

        for _, query_aln, homolog_aln in rows:
            homolog_aa = _ungap_aa(homolog_aln)
            if not homolog_aa:
                continue
            homolog_3di = generate_folding_reference(homolog_aa)
            projected = _project_homolog_3di_to_query(query_aln, homolog_aln, homolog_3di)
            if len(projected) != L:
                continue
            observed = [(pos, tok) for pos, tok in enumerate(projected) if tok in THREEDI_SYMBOL_TO_IDX]
            if not observed:
                continue
            self.n_projected_seqs += 1
            for pos, tok in observed:
                counts[pos, THREEDI_SYMBOL_TO_IDX[tok]] += np.float32(1.0)
            for pos, tok in observed:
                for ctx_len in range(min(MAX_CONTEXT_P, pos), -1, -1):
                    ctx_start = pos - ctx_len
                    ctx = projected[ctx_start:pos]
                    if any(c not in THREEDI_SYMBOL_TO_IDX for c in ctx):
                        continue
                    self.context_counts[(pos, tuple(ctx))][THREEDI_SYMBOL_TO_IDX[tok]] += np.float32(1.0)

        return np.log(counts / counts.sum(axis=1, keepdims=True)).astype(np.float32)

    def get_draft_logits(self, pos: int, prefix=None, K: int | None = None):
        if pos >= self.logits.shape[0]:
            return self.logits[:0]
        if K is None:
            return self.logits[pos:]
        return self.logits[pos:pos + K]

    def emission_for_prefix(self, prefix_symbols: list[str], max_p: int) -> np.ndarray:
        pos = len(prefix_symbols)
        if pos >= self.logits.shape[0]:
            return np.full(len(THREEDI_ALPHABET), -np.log(len(THREEDI_ALPHABET)), dtype=np.float32)
        available = min(max_p, len(prefix_symbols))
        for ctx_len in range(available, -1, -1):
            ctx = tuple(prefix_symbols[-ctx_len:]) if ctx_len > 0 else tuple()
            key = (pos, ctx)
            if key in self.context_counts:
                counts = self.context_counts[key] + self.smoothing
                return np.log(counts / counts.sum()).astype(np.float32)
        return self.logits[pos]


class PrefixAwareFoldingHMMDrafter:
    """Prefix-aware view over a family-specific folding HMM drafter."""

    def __init__(self, base_drafter: FamilyFoldingHMMDrafter, max_p: int = 5):
        self.base_drafter = base_drafter
        self.uid = base_drafter.uid
        self.aa_seq = base_drafter.aa_seq
        self.max_p = max_p
        self.n_family_seqs = base_drafter.n_family_seqs
        self.n_projected_seqs = base_drafter.n_projected_seqs

    def emission_for_prefix(self, prefix_symbols: list[str]) -> np.ndarray:
        return self.base_drafter.emission_for_prefix(prefix_symbols, self.max_p)


def build_family_folding_drafter(uid: str, aa_seq: str, msa_dir: Path) -> FamilyFoldingHMMDrafter:
    cache_path = msa_dir / f"{uid}.a3m"
    cache_hit = cache_path.exists()
    a3m = _fetch_msa_colabfold(aa_seq, cache_path=cache_path)
    if a3m is not None and not cache_hit and FAMILY_API_DELAY_S > 0:
        time.sleep(FAMILY_API_DELAY_S)
    return FamilyFoldingHMMDrafter(uid, aa_seq, a3m)

print("Family-specific folding HMM helpers loaded (MMseqs2 MSA -> pseudo-3Di family counts).")


Family-specific folding HMM helpers loaded (MMseqs2 MSA -> pseudo-3Di family counts).


## Build Family HMM Drafters

Build one family-specific HMM drafter per benchmark protein. Each drafter uses only its own cached MSA family; if no usable homolog rows are available, it falls back to a uniform drafter for that protein.


### What this cell does

Builds the static drafter for each benchmark protein.

- One `FamilyFoldingHMMDrafter` is built per query protein.
- Each drafter uses only that protein's MSA family.
- The drafter returns position-specific 3Di distributions.
- It does not look back at the generated prefix.


In [ ]:
#@title Build family-specific folding HMM drafters. { display-mode: "form" }

FAMILY_MSA_DIR = PROST_DIR / PART2_MSA_CACHE_SUBDIR
FAMILY_MSA_DIR.mkdir(parents=True, exist_ok=True)

hmm_drafters = {}
drafter_items = sorted(paired.items(), key=lambda kv: kv[1]["length"])
if BENCHMARK_PROTEIN_LIMIT is not None:
    drafter_items = drafter_items[:max(BENCHMARK_PROTEIN_LIMIT, HELD_OUT_N)]

print(f"Building family HMM drafters for {len(drafter_items)} proteins...")
for i, (uid, rec) in enumerate(drafter_items, start=1):
    d = build_family_folding_drafter(uid, rec["aa"], FAMILY_MSA_DIR)
    hmm_drafters[uid] = d
    print(f"  [{i}/{len(drafter_items)}] {uid}: homologs={d.n_family_seqs}, projected={d.n_projected_seqs}")
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    elif device.type == "mps" and hasattr(torch, "mps"):
        torch.mps.empty_cache()
    

n_family = sum(d.n_projected_seqs > 0 for d in hmm_drafters.values())
print(f"Family HMM coverage: {n_family}/{len(hmm_drafters)} with projected homolog pseudo-labels")


Building family HMM drafters for 100 proteins...
  [1/100] P01542: homologs=16, projected=16
  [2/100] P0A7N4: homologs=16, projected=16
  [3/100] P02798: homologs=16, projected=16
  [4/100] P61583: homologs=16, projected=16
  [5/100] P24311: homologs=16, projected=16
  [6/100] P0A7U3: homologs=16, projected=16
  [7/100] P63165: homologs=16, projected=16
  [8/100] P10599: homologs=16, projected=16
  [9/100] P01308: homologs=16, projected=16
  [10/100] P61769: homologs=16, projected=16
  [11/100] P37840: homologs=16, projected=16
  [12/100] P00698: homologs=16, projected=16
  [13/100] P68871: homologs=16, projected=16
  [14/100] Q57733: homologs=16, projected=16
  [15/100] P61626: homologs=16, projected=16
  [16/100] P0DP27: homologs=16, projected=16
  [17/100] P16949: homologs=16, projected=16
  [18/100] P00441: homologs=16, projected=16
  [19/100] P68082: homologs=16, projected=16
  [20/100] P0A7Y4: homologs=16, projected=16
  [21/100] Q9R0Q7: homologs=16, projected=16
  [22/100] P182

## Prefix-Aware Family HMM

The prefix-aware view reuses the same family-specific projected pseudo-3Di rows, but conditions the next-token distribution on previously accepted 3Di tokens up to context length `p`.


### What this cell does

Defines the prefix-aware view of the family drafter.

- Reuses the same family pseudo-3Di counts.
- Conditions on recent accepted 3Di prefix tokens.
- Falls back to the static position-wise distribution when a prefix context is missing.


In [ ]:
#@title Prefix-aware context HMM for AA -> 3Di. { display-mode: "form" }

# PrefixAwareFoldingHMMDrafter is defined with the family-specific HMM helpers.
# It reuses each protein's MSA-derived pseudo-3Di family counts and conditions on
# the accepted 3Di prefix up to max_p tokens.
print("Prefix-aware family HMM drafter class ready.")


## Build Prefix-Aware Family Drafters

Build lightweight prefix-aware views for `p = 1..5` from the already-built family HMM drafters. This reuses the cached MSA-derived counts and does not call the API again.


### What this cell does

Builds prefix-aware drafters for multiple context lengths.

Parameter:

- `p`: number of previous generated 3Di tokens used as context.

The notebook builds `p = 1, 2, 3, 4, 5` views for each static family drafter.


In [ ]:
#@title Build prefix-aware family HMM drafters, p=1..5. { display-mode: "form" }

P_VALUES = P_VALUES_OVERRIDE
prefix_hmm_drafters = {p_val: {} for p_val in P_VALUES}

for p_val in P_VALUES:
    for uid, base_drafter in hmm_drafters.items():
        prefix_hmm_drafters[p_val][uid] = PrefixAwareFoldingHMMDrafter(base_drafter, max_p=p_val)

print(
    f"Prefix-aware family HMM drafters ready: "
    f"{len(hmm_drafters)} proteins x {len(P_VALUES)} context lengths"
)


## HuggingFace Assistant Model Wrapper

Wrap the folding HMM drafter as a `PreTrainedModel` so it can be passed to `model.generate(..., assistant_model=...)`.


### What this cell does

Wraps the static family drafter as a HuggingFace assistant model.

- Converts 20-dimensional 3Di logits into full ProstT5 vocabulary logits.
- Uses `THREEDI_TOKEN_IDS` to place logits in the correct vocabulary positions.
- Enables `model.generate(..., assistant_model=hmm_assistant)`.


In [ ]:
#@title Folding HMM assistant_model wrapper. { display-mode: "form" }

class FoldingHMMAssistantModel(PreTrainedModel, GenerationMixin):
    """HF-compatible assistant for AA -> 3Di HMM drafting."""

    config_class = T5Config

    def __init__(self, config, prostt5_encoder, three_di_token_ids, device):
        super().__init__(config)
        self._encoder = prostt5_encoder
        self._di_ids = torch.tensor(three_di_token_ids, device=device, dtype=torch.long)
        self._device = device
        self.config.is_encoder_decoder = True
        self.config.decoder_start_token_id = config.decoder_start_token_id
        self.generation_config = GenerationConfig(
            num_assistant_tokens=5,
            num_assistant_tokens_schedule="constant",
            do_sample=False,
            max_length=3000,
        )
        self._active = None

    def set_drafter(self, drafter):
        self._active = drafter

    def get_encoder(self):
        return self._encoder

    def _validate_model_kwargs(self, model_kwargs):
        return

    def prepare_inputs_for_generation(self, decoder_input_ids, encoder_outputs=None, **kwargs):
        return {"decoder_input_ids": decoder_input_ids, "encoder_outputs": encoder_outputs}

    def forward(self, decoder_input_ids=None, encoder_outputs=None, **kwargs):
        seq_len = decoder_input_ids.shape[1]
        vocab = self.config.vocab_size
        logits = torch.full((1, seq_len, vocab), -1e4, device=decoder_input_ids.device)

        if self._active is not None and seq_len > 0:
            rows = self._active.get_draft_logits(0, K=seq_len)
            if rows.shape[0] > 0:
                rows = torch.from_numpy(rows).to(self._device, dtype=logits.dtype)
                logits[0, :rows.shape[0], self._di_ids] = rows

        return Seq2SeqLMOutput(logits=logits)


hmm_assistant = FoldingHMMAssistantModel(
    config=model.config,
    prostt5_encoder=encoder,
    three_di_token_ids=THREEDI_TOKEN_IDS,
    device=device,
).to(device).eval()

print("Folding HMM assistant created.")

## Prefix-Aware Assistant Model

This assistant reads the current decoder prefix, converts previous generated token IDs back to 3Di symbols, and asks the active prefix-aware drafter for the next-token distribution.


### What this cell does

Wraps the prefix-aware family drafter as a HuggingFace assistant model.

- Reads the current decoder prefix.
- Converts generated token IDs back to 3Di symbols.
- Requests a prefix-conditioned next-token distribution from the active drafter.


In [ ]:
#@title Prefix-aware folding HMM assistant_model wrapper. { display-mode: "form" }

TOKEN_ID_TO_3DI = {tid: tok for tok, tid in zip(THREEDI_ALPHABET, THREEDI_TOKEN_IDS)}

class PrefixAwareFoldingHMMAssistantModel(PreTrainedModel, GenerationMixin):
    """HF-compatible assistant for prefix-aware AA -> 3Di HMM drafting."""

    config_class = T5Config

    def __init__(self, config, prostt5_encoder, three_di_token_ids, device):
        super().__init__(config)
        self._encoder = prostt5_encoder
        self._di_ids = torch.tensor(three_di_token_ids, device=device, dtype=torch.long)
        self._device = device
        self.config.is_encoder_decoder = True
        self.config.decoder_start_token_id = config.decoder_start_token_id
        self.generation_config = GenerationConfig(
            num_assistant_tokens=5,
            num_assistant_tokens_schedule="constant",
            do_sample=False,
            max_length=3000,
        )
        self._active = None

    def set_drafter(self, drafter):
        self._active = drafter

    def get_encoder(self):
        return self._encoder

    def _validate_model_kwargs(self, model_kwargs):
        return

    def prepare_inputs_for_generation(self, decoder_input_ids, encoder_outputs=None, **kwargs):
        return {"decoder_input_ids": decoder_input_ids, "encoder_outputs": encoder_outputs}

    def forward(self, decoder_input_ids=None, encoder_outputs=None, **kwargs):
        seq_len = decoder_input_ids.shape[1]
        vocab = self.config.vocab_size
        logits = torch.full((1, seq_len, vocab), -1e4, device=decoder_input_ids.device)

        if self._active is None or seq_len == 0:
            return Seq2SeqLMOutput(logits=logits)

        prefix_symbols = []
        decoded_ids = decoder_input_ids[0, 1:].tolist()

        for pos in range(seq_len):
            row = self._active.emission_for_prefix(prefix_symbols)
            row = torch.from_numpy(row).to(self._device, dtype=logits.dtype)
            logits[0, pos, self._di_ids] = row

            if pos < len(decoded_ids):
                token_id = decoded_ids[pos]
                if token_id in TOKEN_ID_TO_3DI:
                    prefix_symbols.append(TOKEN_ID_TO_3DI[token_id])

        return Seq2SeqLMOutput(logits=logits)


prefix_hmm_assistant = PrefixAwareFoldingHMMAssistantModel(
    config=model.config,
    prostt5_encoder=encoder,
    three_di_token_ids=THREEDI_TOKEN_IDS,
    device=device,
).to(device).eval()

print("Prefix-aware folding HMM assistant created.")

## Timing Helpers

Measure plain enc-dec folding and HMM-assisted folding with the same warmup/repeat pattern as the inverse-folding notebook.


### What this cell does

Defines benchmark timing functions for plain enc-dec and static HMM assisted folding.

Metrics collected:

- `wall_s`: runtime in seconds.
- `peak_vram_gb`: peak GPU memory.
- generated 3Di output for exact-match comparison.


In [ ]:
#@title Timing helpers. { display-mode: "form" }

def _sync():
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def _reset_peak_mem():
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    elif device.type == "mps":
        torch.mps.empty_cache()


def _peak_mem_gb() -> float:
    if device.type == "cuda":
        return torch.cuda.max_memory_allocated(device) / 1e9
    elif device.type == "mps":
        return torch.mps.current_allocated_memory() / 1e9
    return 0.0


def _safe_mean(values):
    return float(np.mean(values)) if values else float("nan")


def _ensure_metric_columns(df: pd.DataFrame) -> pd.DataFrame:
    for col in [
        "static_hmm_token_accuracy",
        "prefix_mean_accepted_tokens",
        "prefix_acceptance_rate",
        "prefix_decode_steps",
    ]:
        if col not in df.columns:
            df[col] = np.nan
    return df


def static_hmm_token_accuracy(uid: str, reference_seq: str) -> float:
    if not reference_seq:
        return float("nan")
    rows = hmm_drafters[uid].get_draft_logits(0, K=len(reference_seq))
    if rows.shape[0] == 0:
        return float("nan")
    n = min(len(reference_seq), rows.shape[0])
    pred_idx = np.argmax(rows[:n], axis=1)
    pred = "".join(THREEDI_ALPHABET[int(i)] for i in pred_idx)
    return sum(a == b for a, b in zip(pred, reference_seq[:n])) / n


def prefix_hmm_acceptance_stats(uid: str, reference_seq: str, p_val: int, K: int) -> dict:
    if not reference_seq or K <= 0:
        return dict(
            prefix_mean_accepted_tokens=float("nan"),
            prefix_acceptance_rate=float("nan"),
            prefix_decode_steps=0,
        )

    drafter = prefix_hmm_drafters[p_val][uid]
    accepted_counts = []
    pos = 0
    L = len(reference_seq)

    while pos < L:
        prefix = list(reference_seq[:pos])
        draft = []
        for _ in range(min(K, L - pos)):
            row = drafter.emission_for_prefix(prefix + draft)
            draft.append(THREEDI_ALPHABET[int(np.argmax(row))])

        accepted = 0
        for draft_tok, ref_tok in zip(draft, reference_seq[pos:pos + len(draft)]):
            if draft_tok != ref_tok:
                break
            accepted += 1

        accepted_counts.append(accepted)
        pos += accepted + 1

    return dict(
        prefix_mean_accepted_tokens=_safe_mean(accepted_counts),
        prefix_acceptance_rate=(sum(accepted_counts) / (K * len(accepted_counts)) if accepted_counts else float("nan")),
        prefix_decode_steps=len(accepted_counts),
    )


@torch.inference_mode()
def time_encdec_folding(aa_seq: str):
    L = len(aa_seq)
    enc = tokenizer([_format_aa(aa_seq)], add_special_tokens=True, return_tensors="pt").to(device)
    gen_kwargs = dict(
        input_ids=enc.input_ids,
        attention_mask=enc.attention_mask,
        max_length=L + 2,
        do_sample=False,
        num_beams=1,
        use_cache=False,
    )
    for _ in range(NUM_WARMUP):
        model.generate(**gen_kwargs)
    _sync()
    times = []
    _reset_peak_mem()
    for _ in range(NUM_REPEATS):
        _sync()
        t0 = time.perf_counter()
        out = model.generate(**gen_kwargs)
        _sync()
        times.append(time.perf_counter() - t0)
    return statistics.median(times), _decode_3di(out[0])[:L], _peak_mem_gb()


@torch.inference_mode()
def time_hmm_assisted_folding(uid: str, aa_seq: str, K: int):
    drafter = hmm_drafters[uid]
    hmm_assistant.set_drafter(drafter)
    hmm_assistant.generation_config.num_assistant_tokens = float(K)
    hmm_assistant.generation_config.num_assistant_tokens_schedule = "constant"
    hmm_assistant.generation_config.do_sample = False

    L = len(aa_seq)
    enc = tokenizer([_format_aa(aa_seq)], add_special_tokens=True, return_tensors="pt").to(device)
    gen_kwargs = dict(
        input_ids=enc.input_ids,
        attention_mask=enc.attention_mask,
        max_length=L + 2,
        do_sample=False,
        num_beams=1,
        use_cache=True,
        assistant_model=hmm_assistant,
    )
    for _ in range(NUM_WARMUP):
        model.generate(**gen_kwargs)
    _sync()
    times = []
    _reset_peak_mem()
    for _ in range(NUM_REPEATS):
        _sync()
        t0 = time.perf_counter()
        out = model.generate(**gen_kwargs)
        _sync()
        times.append(time.perf_counter() - t0)
    return statistics.median(times), _decode_3di(out[0])[:L], _peak_mem_gb()

print("Timing helpers ready.")


## Prefix-Aware Timing Helper

Time the prefix-aware assistant for a selected context length `p` and draft length `K`.


### What this cell does

Defines the timing function for prefix-aware assisted folding.

Inputs:

- protein ID.
- AA sequence.
- prefix context length `p_val`.
- draft length `K`.


In [ ]:
#@title Prefix-aware timing helper. { display-mode: "form" }

@torch.inference_mode()
def time_prefix_hmm_assisted_folding(uid: str, aa_seq: str, p_val: int, K: int):
    drafter = prefix_hmm_drafters[p_val][uid]
    prefix_hmm_assistant.set_drafter(drafter)
    prefix_hmm_assistant.generation_config.num_assistant_tokens = float(K)
    prefix_hmm_assistant.generation_config.num_assistant_tokens_schedule = "constant"
    prefix_hmm_assistant.generation_config.do_sample = False

    L = len(aa_seq)
    enc = tokenizer([_format_aa(aa_seq)], add_special_tokens=True, return_tensors="pt").to(device)
    gen_kwargs = dict(
        input_ids=enc.input_ids,
        attention_mask=enc.attention_mask,
        max_length=L + 2,
        do_sample=False,
        num_beams=1,
        use_cache=True,
        assistant_model=prefix_hmm_assistant,
    )
    for _ in range(NUM_WARMUP):
        model.generate(**gen_kwargs)
    _sync()
    times = []
    _reset_peak_mem()
    for _ in range(NUM_REPEATS):
        _sync()
        t0 = time.perf_counter()
        out = model.generate(**gen_kwargs)
        _sync()
        times.append(time.perf_counter() - t0)
    return statistics.median(times), _decode_3di(out[0])[:L], _peak_mem_gb()

print("Prefix-aware timing helper ready.")


## Full Benchmark Settings

The notebook now has one full-evaluation configuration: 100 benchmark proteins, the configured `K_VALUES`, and outputs under the local result directory. No dev/evaluate switch is needed.


## Single Benchmark Flow

Run the following benchmark cells in order. They reuse the same 100-protein selection, compare assisted generation against the plain ProstT5 baseline, and save raw tables plus summaries for later plotting.


## Benchmark Loop

Run the same benchmark proteins for plain ProstT5 folding and HMM-assisted folding over multiple K values.


### What this cell does

Runs the static HMM benchmark.

For each protein:

- time plain ProstT5 folding.
- time HMM-assisted folding for each `K`.
- record speedup, exact match, peak vRAM, and family coverage.


In [ ]:
#@title Main folding HMM benchmark loop. { display-mode: "form" }

if not RUN_BENCHMARKS:
    print("Skipping static HMM timing benchmark (RUN_BENCHMARKS=False).")
    results_df = pd.DataFrame()
else:

    rows = []
    bench_exact = 0
    bench_total = 0
    proteins = sorted(paired.items(), key=lambda kv: kv[1]["length"])
    if BENCHMARK_PROTEIN_LIMIT is not None:
        proteins = proteins[:BENCHMARK_PROTEIN_LIMIT]

    for idx, (uid, rec) in enumerate(proteins, start=1):
        aa = rec["aa"]
        L = rec["length"]

        t_ref, seq_ref, mem_ref = time_encdec_folding(aa)
        static_accuracy = static_hmm_token_accuracy(uid, seq_ref)
        rows.append(dict(
            protein_id=uid,
            length=L,
            pipeline="enc_dec_folding",
            K=0,
            wall_s=t_ref,
            speedup=1.0,
            peak_vram_gb=mem_ref,
            exact_match=True,
            static_hmm_token_accuracy=np.nan,
            family_homologs=hmm_drafters[uid].n_family_seqs,
            family_projected=hmm_drafters[uid].n_projected_seqs,
        ))

        protein_exact = 0
        protein_speedups = []
        for K in K_VALUES:
            t_hmm, seq_hmm, mem_hmm = time_hmm_assisted_folding(uid, aa, K=K)
            exact = seq_hmm == seq_ref
            speedup = t_ref / t_hmm if t_hmm > 0 else float("nan")
            bench_total += 1
            bench_exact += int(exact)
            protein_exact += int(exact)
            protein_speedups.append(speedup)
            rows.append(dict(
                protein_id=uid,
                length=L,
                pipeline="hmm_assisted_folding",
                K=K,
                wall_s=t_hmm,
                speedup=speedup,
                peak_vram_gb=mem_hmm,
                exact_match=exact,
                static_hmm_token_accuracy=static_accuracy,
                family_homologs=hmm_drafters[uid].n_family_seqs,
                family_projected=hmm_drafters[uid].n_projected_seqs,
            ))
        print(
            f"[{idx}/{len(proteins)}] {uid} L={L}: "
            f"exact={protein_exact}/{len(K_VALUES)}, "
            f"static_acc={static_accuracy:.3f}, "
            f"median_speedup={np.nanmedian(protein_speedups):.2f}x, "
            f"family_projected={hmm_drafters[uid].n_projected_seqs}"
        )

    results_df = pd.DataFrame(rows)
    results_path = RESULTS_DIR / "folding_hmm_results.csv"
    results_df.to_csv(results_path, index=False)
    print(f"HMM-assisted bit-exact: {bench_exact}/{bench_total} ({bench_exact / bench_total:.1%})")
    print(f"Saved {len(results_df)} rows to {results_path}")
    results_df.head()


## Prefix-Aware Benchmark Loop

Benchmark prefix-aware folding HMM drafters for `p = 1..5` and `K = 1, 3, 5, 8` using the same proteins.


### What this cell does

Runs the prefix-aware HMM benchmark.

For each protein, it tests all combinations of:

- `p` in `P_VALUES`.
- `K` in `K_VALUES`.

This is where outputs such as `prefix_exact=8/20` are produced.


### Prefix-Aware Timing Scope

This cell keeps all 100 benchmark proteins but restricts the timed prefix-aware benchmark to `p = 3` and `p = 5`. The full set of prefix-aware drafter views is still built earlier, so this only controls which views are timed in the next benchmark loop.


In [ ]:
#@title Run prefix-aware benchmark only for p=3 and p=5. { display-mode: "form" }

# Keep the same benchmark proteins already loaded in `paired`, but restrict the
# prefix-aware context lengths used by the next benchmark cell.
P_VALUES = [3, 5]

# Rebuild only the requested prefix-aware views if they are missing.
if "prefix_hmm_drafters" not in globals():
    prefix_hmm_drafters = {}
for p_val in P_VALUES:
    if p_val not in prefix_hmm_drafters:
        prefix_hmm_drafters[p_val] = {}
    for uid, base_drafter in hmm_drafters.items():
        if uid not in prefix_hmm_drafters[p_val]:
            prefix_hmm_drafters[p_val][uid] = PrefixAwareFoldingHMMDrafter(base_drafter, max_p=p_val)

proteins_preview = sorted(paired.items(), key=lambda kv: kv[1]["length"])
if BENCHMARK_PROTEIN_LIMIT is not None:
    proteins_preview = proteins_preview[:BENCHMARK_PROTEIN_LIMIT]

print(f"Prefix-aware benchmark restricted to P_VALUES={P_VALUES}")
print(f"Same benchmark proteins: {len(proteins_preview)}")
print("Protein IDs:", ", ".join(uid for uid, _ in proteins_preview))


### Prefix-Aware Benchmark Loop Details

This cell times prefix-aware HMM-assisted folding on the same 100 proteins. It reuses one cached plain ProstT5 baseline per protein, computes speedup as baseline time divided by assisted time, records exact-match status, and saves `folding_prefix_hmm_results.csv` locally.


In [ ]:
#@title Main prefix-aware folding HMM benchmark loop. { display-mode: "form" }

if not RUN_BENCHMARKS:
    print("Skipping prefix-aware HMM timing benchmark (RUN_BENCHMARKS=False).")
    prefix_results_df = pd.DataFrame()
else:

    prefix_rows = []
    prefix_bench_exact = 0
    prefix_bench_total = 0
    proteins = sorted(paired.items(), key=lambda kv: kv[1]["length"])
    if BENCHMARK_PROTEIN_LIMIT is not None:
        proteins = proteins[:BENCHMARK_PROTEIN_LIMIT]
    ref_cache = {}

    for idx, (uid, rec) in enumerate(proteins, start=1):
        aa = rec["aa"]
        L = rec["length"]

        if uid not in ref_cache:
            t_ref, seq_ref, mem_ref = time_encdec_folding(aa)
            ref_cache[uid] = (t_ref, seq_ref, mem_ref)
        else:
            t_ref, seq_ref, mem_ref = ref_cache[uid]

        static_accuracy = static_hmm_token_accuracy(uid, seq_ref)
        protein_exact = 0
        protein_total = 0
        protein_speedups = []
        protein_accepted = []
        for p_val in P_VALUES:
            for K in K_VALUES:
                t_hmm, seq_hmm, mem_hmm = time_prefix_hmm_assisted_folding(uid, aa, p_val=p_val, K=K)
                accept_stats = prefix_hmm_acceptance_stats(uid, seq_ref, p_val=p_val, K=K)
                exact = seq_hmm == seq_ref
                speedup = t_ref / t_hmm if t_hmm > 0 else float("nan")
                prefix_bench_total += 1
                prefix_bench_exact += int(exact)
                protein_total += 1
                protein_exact += int(exact)
                protein_speedups.append(speedup)
                protein_accepted.append(accept_stats["prefix_mean_accepted_tokens"])
                prefix_rows.append(dict(
                    protein_id=uid,
                    length=L,
                    pipeline="prefix_hmm_assisted_folding",
                    p=p_val,
                    K=K,
                    wall_s=t_hmm,
                    speedup=speedup,
                    peak_vram_gb=mem_hmm,
                    exact_match=exact,
                    static_hmm_token_accuracy=static_accuracy,
                    **accept_stats,
                    family_homologs=hmm_drafters[uid].n_family_seqs,
                    family_projected=hmm_drafters[uid].n_projected_seqs,
                ))
        print(
            f"[{idx}/{len(proteins)}] {uid} L={L}: "
            f"prefix_exact={protein_exact}/{protein_total}, "
            f"mean_accepted={np.nanmean(protein_accepted):.2f}, "
            f"median_speedup={np.nanmedian(protein_speedups):.2f}x, "
            f"family_projected={hmm_drafters[uid].n_projected_seqs}"
        )

    prefix_results_df = pd.DataFrame(prefix_rows)
    prefix_results_path = RESULTS_DIR / "folding_prefix_hmm_results.csv"
    prefix_results_df.to_csv(prefix_results_path, index=False)
    print(f"Prefix-aware HMM bit-exact: {prefix_bench_exact}/{prefix_bench_total} ({prefix_bench_exact / prefix_bench_total:.1%})")
    print(f"Saved {len(prefix_results_df)} rows to {prefix_results_path}")
    prefix_results_df.head()


## Aggregate Results

Summarize speedup and memory behavior for the folding HMM drafter.


### What this cell does

Summarizes the static HMM benchmark table.

Grouped by:

- pipeline.
- draft length `K`.

Saved as `folding_hmm_summary.csv`.


In [ ]:
#@title Aggregate folding HMM results. { display-mode: "form" }

results_df = _ensure_metric_columns(results_df)
summary = (
    results_df
    .groupby(["pipeline", "K"], dropna=False)
    .agg(
        n=("protein_id", "count"),
        median_wall_s=("wall_s", "median"),
        median_speedup=("speedup", "median"),
        median_peak_vram_gb=("peak_vram_gb", "median"),
        exact_match_rate=("exact_match", "mean"),
        mean_static_hmm_token_accuracy=("static_hmm_token_accuracy", "mean"),
    )
    .reset_index()
)
summary_path = RESULTS_DIR / "folding_hmm_summary.csv"
summary.to_csv(summary_path, index=False)
print(f"Saved summary to {summary_path}")
summary


## Static vs Prefix-Aware Comparison Plot

This optional comparison cell reruns the same 100 proteins for static HMM and prefix-aware HMM with `p = 3` and `p = 5`, then saves a focused CSV summary and plot in the local results directory.


In [ ]:
#@title Run and plot static HMM plus prefix-aware p=3 and p=5. { display-mode: "form" }

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

P_COMPARE_VALUES = [3, 5]
COMBINED_RESULTS_PATH = RESULTS_DIR / "folding_hmm_static_prefix_p3_p5_results.csv"
COMBINED_SUMMARY_PATH = RESULTS_DIR / "folding_hmm_static_prefix_p3_p5_summary.csv"
COMBINED_PLOT_PATH = RESULTS_DIR / "folding_hmm_static_prefix_p3_p5_comparison.png"

# Keep the same benchmark protein selection as the main benchmark cells.
proteins = sorted(paired.items(), key=lambda kv: kv[1]["length"])
if BENCHMARK_PROTEIN_LIMIT is not None:
    proteins = proteins[:BENCHMARK_PROTEIN_LIMIT]

# Ensure only the requested prefix-aware views exist for this run.
if "prefix_hmm_drafters" not in globals():
    prefix_hmm_drafters = {}
for p_val in P_COMPARE_VALUES:
    if p_val not in prefix_hmm_drafters:
        prefix_hmm_drafters[p_val] = {}
    for uid, base_drafter in hmm_drafters.items():
        if uid not in prefix_hmm_drafters[p_val]:
            prefix_hmm_drafters[p_val][uid] = PrefixAwareFoldingHMMDrafter(base_drafter, max_p=p_val)

print(f"Running static HMM and prefix-aware HMM for p={P_COMPARE_VALUES}")
print(f"Proteins: {len(proteins)}")
print("Protein IDs:", ", ".join(uid for uid, _ in proteins))

combined_rows = []
ref_cache = {}

for idx, (uid, rec) in enumerate(proteins, start=1):
    aa = rec["aa"]
    L = rec["length"]

    if uid not in ref_cache:
        t_ref, seq_ref, mem_ref = time_encdec_folding(aa)
        ref_cache[uid] = (t_ref, seq_ref, mem_ref)
    else:
        t_ref, seq_ref, mem_ref = ref_cache[uid]

    static_accuracy = static_hmm_token_accuracy(uid, seq_ref)
    protein_exact = 0
    protein_total = 0
    protein_speedups = []
    protein_accepted = []

    # Static family HMM drafter: same proteins and same K grid, no prefix context p.
    for K in K_VALUES:
        t_hmm, seq_hmm, mem_hmm = time_hmm_assisted_folding(uid, aa, K=K)
        exact = seq_hmm == seq_ref
        speedup = t_ref / t_hmm if t_hmm > 0 else float("nan")
        protein_exact += int(exact)
        protein_total += 1
        protein_speedups.append(speedup)
        combined_rows.append(dict(
            protein_id=uid,
            length=L,
            drafter="static_hmm",
            p="static",
            K=K,
            wall_s=t_hmm,
            speedup=speedup,
            peak_vram_gb=mem_hmm,
            exact_match=exact,
            static_hmm_token_accuracy=static_accuracy,
            prefix_mean_accepted_tokens=np.nan,
            prefix_acceptance_rate=np.nan,
            prefix_decode_steps=np.nan,
            family_homologs=hmm_drafters[uid].n_family_seqs,
            family_projected=hmm_drafters[uid].n_projected_seqs,
        ))

    # Prefix-aware family HMM drafters for p=3 and p=5.
    for p_val in P_COMPARE_VALUES:
        for K in K_VALUES:
            t_hmm, seq_hmm, mem_hmm = time_prefix_hmm_assisted_folding(uid, aa, p_val=p_val, K=K)
            accept_stats = prefix_hmm_acceptance_stats(uid, seq_ref, p_val=p_val, K=K)
            exact = seq_hmm == seq_ref
            speedup = t_ref / t_hmm if t_hmm > 0 else float("nan")
            protein_exact += int(exact)
            protein_total += 1
            protein_speedups.append(speedup)
            protein_accepted.append(accept_stats["prefix_mean_accepted_tokens"])
            combined_rows.append(dict(
                protein_id=uid,
                length=L,
                drafter="prefix_hmm",
                p=p_val,
                K=K,
                wall_s=t_hmm,
                speedup=speedup,
                peak_vram_gb=mem_hmm,
                exact_match=exact,
                static_hmm_token_accuracy=static_accuracy,
                **accept_stats,
                family_homologs=hmm_drafters[uid].n_family_seqs,
                family_projected=hmm_drafters[uid].n_projected_seqs,
            ))

    print(
        f"[{idx}/{len(proteins)}] {uid} L={L}: "
        f"exact={protein_exact}/{protein_total}, "
        f"static_acc={static_accuracy:.3f}, "
        f"mean_accepted={np.nanmean(protein_accepted):.2f}, "
        f"median_speedup={np.nanmedian(protein_speedups):.2f}x, "
        f"family_projected={hmm_drafters[uid].n_projected_seqs}"
    )

combined_compare_df = pd.DataFrame(combined_rows)
combined_compare_df = _ensure_metric_columns(combined_compare_df)
combined_compare_df.to_csv(COMBINED_RESULTS_PATH, index=False)

combined_summary = (
    combined_compare_df
    .groupby(["drafter", "p", "K"], dropna=False)
    .agg(
        n=("protein_id", "count"),
        proteins=("protein_id", "nunique"),
        median_speedup=("speedup", "median"),
        mean_speedup=("speedup", "mean"),
        exact_match_rate=("exact_match", "mean"),
        mean_static_hmm_token_accuracy=("static_hmm_token_accuracy", "mean"),
        mean_prefix_accepted_tokens=("prefix_mean_accepted_tokens", "mean"),
        median_prefix_accepted_tokens=("prefix_mean_accepted_tokens", "median"),
        mean_prefix_acceptance_rate=("prefix_acceptance_rate", "mean"),
        median_wall_s=("wall_s", "median"),
        median_peak_vram_gb=("peak_vram_gb", "median"),
    )
    .reset_index()
)

label_order = ["static HMM", "prefix HMM p=3", "prefix HMM p=5"]
label_map = {("static_hmm", "static"): "static HMM"}
for p_val in P_COMPARE_VALUES:
    label_map[("prefix_hmm", p_val)] = f"prefix HMM p={p_val}"
combined_summary["label"] = combined_summary.apply(lambda r: label_map[(r["drafter"], r["p"])], axis=1)
combined_summary["label"] = pd.Categorical(combined_summary["label"], categories=label_order, ordered=True)
combined_summary = combined_summary.sort_values(["label", "K"])
combined_summary.to_csv(COMBINED_SUMMARY_PATH, index=False)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.7), constrained_layout=True)
colors = {
    "static HMM": "#ea580c",
    "prefix HMM p=3": "#2563eb",
    "prefix HMM p=5": "#059669",
}
markers = {
    "static HMM": "s",
    "prefix HMM p=3": "o",
    "prefix HMM p=5": "^",
}

for label in label_order:
    sub = combined_summary[combined_summary["label"] == label].sort_values("K")
    axes[0].plot(sub["K"], sub["median_speedup"], marker=markers[label], linewidth=2, color=colors[label], label=label)
    axes[1].plot(sub["K"], sub["exact_match_rate"], marker=markers[label], linewidth=2, color=colors[label], label=label)
    axes[2].plot(sub["K"], sub["median_wall_s"], marker=markers[label], linewidth=2, color=colors[label], label=label)

axes[0].axhline(1.0, color="red", linestyle="--", alpha=0.4)
axes[0].set_title("Median speedup vs K")
axes[0].set_xlabel("Draft length K")
axes[0].set_ylabel("Median speedup")
axes[1].set_title("Exact-match rate vs K")
axes[1].set_xlabel("Draft length K")
axes[1].set_ylabel("Assisted output == ProstT5 greedy")
axes[1].set_ylim(0, 1.05)
axes[2].set_title("Median wall time vs K")
axes[2].set_xlabel("Draft length K")
axes[2].set_ylabel("Median wall time (s)")

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend(title="Drafter")

fig.suptitle("Static HMM vs prefix-aware HMM p=3 and p=5", fontsize=14, fontweight="bold")
fig.savefig(COMBINED_PLOT_PATH, dpi=180, bbox_inches="tight")

print(f"Saved raw results to {COMBINED_RESULTS_PATH}")
print(f"Saved summary to {COMBINED_SUMMARY_PATH}")
print(f"Saved plot to {COMBINED_PLOT_PATH}")
display(combined_summary)
plt.show()


## Prefix-Aware Aggregate Results

Summarize the effect of previous-token context length `p` on folding assisted generation.


### What this cell does

Summarizes prefix-aware benchmark results.

Grouped by:

- prefix context length `p`.
- draft length `K`.

Saved as `folding_prefix_hmm_summary.csv`.


In [ ]:
#@title Aggregate prefix-aware folding HMM results. { display-mode: "form" }

prefix_results_df = _ensure_metric_columns(prefix_results_df)
prefix_summary = (
    prefix_results_df
    .groupby(["p", "K"], dropna=False)
    .agg(
        n=("protein_id", "count"),
        median_wall_s=("wall_s", "median"),
        median_speedup=("speedup", "median"),
        median_peak_vram_gb=("peak_vram_gb", "median"),
        exact_match_rate=("exact_match", "mean"),
        mean_static_hmm_token_accuracy=("static_hmm_token_accuracy", "mean"),
        mean_prefix_accepted_tokens=("prefix_mean_accepted_tokens", "mean"),
        median_prefix_accepted_tokens=("prefix_mean_accepted_tokens", "median"),
        mean_prefix_acceptance_rate=("prefix_acceptance_rate", "mean"),
    )
    .reset_index()
)
prefix_summary_path = RESULTS_DIR / "folding_prefix_hmm_summary.csv"
prefix_summary.to_csv(prefix_summary_path, index=False)
print(f"Saved prefix summary to {prefix_summary_path}")
prefix_summary


## Compare Prefix Context Lengths

Aggregate prefix-aware results by context length `p` and draft length `K`, then plot speedup, exact-match rate, and wall time so different `p` values can be compared directly.


In [ ]:
#@title Compare prefix-aware HMM results across p values. { display-mode: "form" }

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

prefix_results_path = RESULTS_DIR / "folding_prefix_hmm_results.csv"

if "prefix_results_df" in globals() and len(prefix_results_df) > 0:
    prefix_compare_df = prefix_results_df.copy()
elif prefix_results_path.exists():
    prefix_compare_df = pd.read_csv(prefix_results_path)
else:
    raise FileNotFoundError(
        f"No prefix-aware results found. Run the prefix-aware benchmark first: {prefix_results_path}"
    )

if "pipeline" in prefix_compare_df.columns:
    prefix_compare_df = prefix_compare_df[
        prefix_compare_df["pipeline"] == "prefix_hmm_assisted_folding"
    ].copy()

prefix_compare_df = _ensure_metric_columns(prefix_compare_df)

prefix_p_summary = (
    prefix_compare_df
    .groupby(["p", "K"], dropna=False)
    .agg(
        n=("protein_id", "count"),
        proteins=("protein_id", "nunique"),
        median_speedup=("speedup", "median"),
        mean_speedup=("speedup", "mean"),
        exact_match_rate=("exact_match", "mean"),
        mean_static_hmm_token_accuracy=("static_hmm_token_accuracy", "mean"),
        mean_prefix_accepted_tokens=("prefix_mean_accepted_tokens", "mean"),
        median_prefix_accepted_tokens=("prefix_mean_accepted_tokens", "median"),
        mean_prefix_acceptance_rate=("prefix_acceptance_rate", "mean"),
        median_wall_s=("wall_s", "median"),
        median_peak_vram_gb=("peak_vram_gb", "median"),
    )
    .reset_index()
    .sort_values(["p", "K"])
)

prefix_p_summary_path = RESULTS_DIR / "folding_prefix_hmm_p_comparison_summary.csv"
prefix_p_summary.to_csv(prefix_p_summary_path, index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
colors = plt.cm.viridis(np.linspace(0.15, 0.85, prefix_p_summary["p"].nunique()))

for color, p_val in zip(colors, sorted(prefix_p_summary["p"].unique())):
    sub = prefix_p_summary[prefix_p_summary["p"] == p_val].sort_values("K")

    axes[0].plot(sub["K"], sub["median_speedup"], marker="o", linewidth=2, color=color, label=f"p={p_val}")
    axes[1].plot(sub["K"], sub["exact_match_rate"], marker="o", linewidth=2, color=color, label=f"p={p_val}")
    axes[2].plot(sub["K"], sub["median_wall_s"], marker="o", linewidth=2, color=color, label=f"p={p_val}")

axes[0].axhline(1.0, color="red", linestyle="--", alpha=0.4)
axes[0].set_title("Median speedup vs K")
axes[0].set_xlabel("Draft length K")
axes[0].set_ylabel("Median speedup")

axes[1].set_title("Exact-match rate vs K")
axes[1].set_xlabel("Draft length K")
axes[1].set_ylabel("Assisted output == ProstT5 greedy")
axes[1].set_ylim(0, 1.05)

axes[2].set_title("Median wall time vs K")
axes[2].set_xlabel("Draft length K")
axes[2].set_ylabel("Median wall time (s)")

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend(title="Context length")

fig.suptitle("Prefix-aware HMM drafter: comparison across p", fontsize=14, fontweight="bold")
prefix_p_plot_path = RESULTS_DIR / "folding_prefix_hmm_p_comparison.png"
fig.savefig(prefix_p_plot_path, dpi=180, bbox_inches="tight")

print(f"Compared p values: {sorted(prefix_p_summary['p'].unique())}")
print(f"Proteins per p/K row: {sorted(prefix_p_summary['proteins'].unique())}")
print(f"Saved p comparison summary to {prefix_p_summary_path}")
print(f"Saved p comparison plot to {prefix_p_plot_path}")
display(prefix_p_summary)
plt.show()


## Collect, Aggregate, And Plot Benchmark Results

These final result cells read the benchmark tables produced above, write compact summary CSVs, and save the overview plot locally. They do not build new drafters.


In [ ]:
#@title Collect and aggregate benchmark result tables. { display-mode: "form" }

import pandas as pd
import numpy as np

folding_results_path = RESULTS_DIR / "folding_hmm_results.csv"
prefix_results_path = RESULTS_DIR / "folding_prefix_hmm_results.csv"

folding_plot_df = results_df.copy() if "results_df" in globals() and len(results_df) > 0 else pd.read_csv(folding_results_path)
prefix_plot_df = prefix_results_df.copy() if "prefix_results_df" in globals() and len(prefix_results_df) > 0 else pd.read_csv(prefix_results_path)
folding_plot_df = _ensure_metric_columns(folding_plot_df)
prefix_plot_df = _ensure_metric_columns(prefix_plot_df)

fixed_hmm_df = folding_plot_df[folding_plot_df["pipeline"] == "hmm_assisted_folding"].copy()
fixed_hmm_df["drafter"] = "hmm"
fixed_hmm_df["p"] = np.nan

prefix_hmm_df = prefix_plot_df[prefix_plot_df["pipeline"] == "prefix_hmm_assisted_folding"].copy()
prefix_hmm_df["drafter"] = "prefix_hmm"

plot_df = pd.concat([fixed_hmm_df, prefix_hmm_df], ignore_index=True)

benchmark_summary = (
    plot_df
    .groupby(["drafter", "K"], dropna=False)
    .agg(
        n=("protein_id", "count"),
        proteins=("protein_id", "nunique"),
        median_speedup=("speedup", "median"),
        mean_speedup=("speedup", "mean"),
        exact_match_rate=("exact_match", "mean"),
        mean_static_hmm_token_accuracy=("static_hmm_token_accuracy", "mean"),
        mean_prefix_accepted_tokens=("prefix_mean_accepted_tokens", "mean"),
        median_prefix_accepted_tokens=("prefix_mean_accepted_tokens", "median"),
        mean_prefix_acceptance_rate=("prefix_acceptance_rate", "mean"),
        median_wall_s=("wall_s", "median"),
        median_peak_vram_gb=("peak_vram_gb", "median"),
        median_family_projected=("family_projected", "median"),
    )
    .reset_index()
)

prefix_pk_summary = (
    prefix_hmm_df
    .groupby(["p", "K"], dropna=False)
    .agg(
        n=("protein_id", "count"),
        median_speedup=("speedup", "median"),
        exact_match_rate=("exact_match", "mean"),
        mean_static_hmm_token_accuracy=("static_hmm_token_accuracy", "mean"),
        mean_prefix_accepted_tokens=("prefix_mean_accepted_tokens", "mean"),
        median_prefix_accepted_tokens=("prefix_mean_accepted_tokens", "median"),
        mean_prefix_acceptance_rate=("prefix_acceptance_rate", "mean"),
        median_peak_vram_gb=("peak_vram_gb", "median"),
    )
    .reset_index()
)

benchmark_summary_path = RESULTS_DIR / "probabilistic_drafter_folding_summary.csv"
prefix_pk_summary_path = RESULTS_DIR / "probabilistic_drafter_folding_prefix_p_k_summary.csv"
benchmark_summary.to_csv(benchmark_summary_path, index=False)
prefix_pk_summary.to_csv(prefix_pk_summary_path, index=False)

print(f"Mode: {MODE}")
print(f"Static rows : {len(folding_plot_df)}")
print(f"Prefix rows : {len(prefix_plot_df)}")
print(f"Combined rows: {len(plot_df)}")
print(f"Saved benchmark summary: {benchmark_summary_path}")
print(f"Saved prefix p/K summary: {prefix_pk_summary_path}")

display(benchmark_summary)
display(prefix_pk_summary.head())


### Overview Plot

This cell plots the aggregated benchmark results: median speedup, exact-match rate, peak memory, and prefix-aware exactness by `p` and `K`. The image is saved in the local results directory.


In [ ]:
#@title Plot benchmark result summary. { display-mode: "form" }

import matplotlib.pyplot as plt

colors = {"hmm": "darkorange", "prefix_hmm": "forestgreen"}
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Speedup vs K
ax = axes[0, 0]
for drafter in ["hmm", "prefix_hmm"]:
    sub = plot_df[plot_df["drafter"] == drafter]
    by_k = sub.groupby("K")["speedup"].median()
    q1 = sub.groupby("K")["speedup"].quantile(0.25)
    q3 = sub.groupby("K")["speedup"].quantile(0.75)
    ax.errorbar(by_k.index, by_k.values, yerr=[by_k.values - q1.values, q3.values - by_k.values], fmt="o-", label=drafter, color=colors[drafter], capsize=3)
ax.axhline(1.0, color="red", linestyle="--", alpha=0.4)
ax.set_xlabel("Draft length K")
ax.set_ylabel("Median speedup")
ax.set_title("Speedup vs K")
ax.legend()
ax.grid(True, alpha=0.3)

# Exactness vs K
ax = axes[0, 1]
for drafter in ["hmm", "prefix_hmm"]:
    sub = plot_df[plot_df["drafter"] == drafter]
    exact = sub.groupby("K")["exact_match"].mean()
    ax.plot(exact.index, exact.values, "o-", label=drafter, color=colors[drafter])
ax.set_xlabel("Draft length K")
ax.set_ylabel("Exact-match rate")
ax.set_ylim(0, 1.05)
ax.set_title("Assisted output == ProstT5 greedy")
ax.legend()
ax.grid(True, alpha=0.3)

# Peak vRAM vs K
ax = axes[1, 0]
for drafter in ["hmm", "prefix_hmm"]:
    sub = plot_df[(plot_df["drafter"] == drafter) & plot_df["peak_vram_gb"].notna()]
    mem = sub.groupby("K")["peak_vram_gb"].median()
    ax.plot(mem.index, mem.values, "o-", label=drafter, color=colors[drafter])
ax.set_xlabel("Draft length K")
ax.set_ylabel("Median peak vRAM (GB)")
ax.set_title("Memory vs K")
ax.legend()
ax.grid(True, alpha=0.3)

# Prefix p/K heatmap
ax = axes[1, 1]
heat = prefix_hmm_df.pivot_table(index="p", columns="K", values="exact_match", aggfunc="mean")
im = ax.imshow(heat.values, aspect="auto", origin="lower", vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns)
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)
ax.set_xlabel("Draft length K")
ax.set_ylabel("Prefix context p")
ax.set_title("Prefix-HMM exact-match rate")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(f"ProstT5 Probabilistic Folding Drafter ({MODE})", fontsize=13)
fig.tight_layout()
plot_path = RESULTS_DIR / "probabilistic_drafter_folding_plots.png"
fig.savefig(plot_path, dpi=150)
plt.show()
print(f"Saved plot to {plot_path}")


### Inspect One Family Alignment

This diagnostic cell prints the first query protein, its AA alignment row, and up to 16 family rows with their generated/projected 3Di. It is for visual inspection only and does not change benchmark outputs.


In [ ]:
#@title Print first protein, 16 family AA alignments, and projected 3Di rows. { display-mode: "form" }

first_uid, first_rec = sorted(paired.items(), key=lambda kv: kv[1]["length"])[0]
first_aa = first_rec["aa"]
first_a3m_path = FAMILY_MSA_DIR / f"{first_uid}.a3m"
first_a3m = _fetch_msa_colabfold(first_aa, cache_path=first_a3m_path)

if first_a3m is None:
    raise RuntimeError(f"No cached/fetched A3M alignment available for {first_uid}")

records = _parse_a3m_records(first_a3m)
if not records:
    raise RuntimeError(f"A3M alignment for {first_uid} has no records")

query_name, query_raw = records[0]
query_aln = _strip_a3m_insertions(query_raw)
query_len = len(_ungap_aa(query_aln))
query_3di = generate_folding_reference(first_aa)

family_rows = []
seen_aa = set()
for name, raw_aln in records[1:]:
    homolog_aln = _strip_a3m_insertions(raw_aln)
    homolog_aa = _ungap_aa(homolog_aln)
    if len(homolog_aa) < FAMILY_MSA_MIN_SEQ_LEN or homolog_aa in seen_aa:
        continue
    seen_aa.add(homolog_aa)
    homolog_3di = generate_folding_reference(homolog_aa)
    projected_3di = _project_homolog_3di_to_query(query_aln, homolog_aln, homolog_3di)
    aligned_projected_3di = "".join(tok if tok is not None else "-" for tok in projected_3di)
    family_rows.append((name, homolog_aln, homolog_aa, homolog_3di, aligned_projected_3di))
    if len(family_rows) >= FAMILY_MSA_MAX_SEQS:
        break

print(f"First protein: {first_uid}")
print(f"A3M path: {first_a3m_path}")
print(f"Query name: {query_name}")
print(f"Query AA length: {len(first_aa)}")
print(f"Query alignment width: {len(query_aln)}")
print(f"Query 3Di length: {len(query_3di)}")
print(f"Family rows shown: {len(family_rows)}/{FAMILY_MSA_MAX_SEQS}")
print()
print("QUERY_AA_ALN:")
print(query_aln)
print("QUERY_3DI:")
print(query_3di[:query_len])
print()

for i, (name, homolog_aln, homolog_aa, homolog_3di, aligned_projected_3di) in enumerate(family_rows, start=1):
    print(f"[{i:02d}] {name}")
    print(f"  homolog aligned AA width : {len(homolog_aln)}")
    print(f"  homolog ungapped AA len  : {len(homolog_aa)}")
    print(f"  homolog raw 3Di len      : {len(homolog_3di)}")
    print(f"  projected query 3Di len  : {len(aligned_projected_3di)}")
    print("  HOMOLOG_AA_ALN:")
    print(f"  {homolog_aln}")
    print("  PROJECTED_3DI_ON_QUERY:")
    print(f"  {aligned_projected_3di}")
    print()

alignment_3di_df = pd.DataFrame([{
    "idx": i,
    "name": name,
    "homolog_aligned_width": len(homolog_aln),
    "homolog_aa_len": len(homolog_aa),
    "homolog_3di_len": len(homolog_3di),
    "projected_3di_len": len(aligned_projected_3di),
    "homolog_AA_alignment": homolog_aln,
    "projected_3Di_on_query": aligned_projected_3di,
} for i, (name, homolog_aln, homolog_aa, homolog_3di, aligned_projected_3di) in enumerate(family_rows, start=1)])

alignment_3di_df



### Export Query/Family AA And 3Di Alignments

This export cell writes a CSV containing, for each of the 100 query proteins, the query AA alignment, up to 16 family AA alignments, projected family 3Di rows, and a length table. The CSV is saved in the local results directory.


In [ ]:
#@title Save all query/family aligned AA and 3Di rows to CSV. { display-mode: "form" }

aligned_family_csv_path = RESULTS_DIR / "query_family_aligned_aa_3di.csv"
aligned_family_rows = []

proteins = sorted(paired.items(), key=lambda kv: kv[1]["length"])
if BENCHMARK_PROTEIN_LIMIT is not None:
    proteins = proteins[:BENCHMARK_PROTEIN_LIMIT]

for protein_index, (uid, rec) in enumerate(proteins, start=1):
    query_aa = rec["aa"]
    a3m_path = FAMILY_MSA_DIR / f"{uid}.a3m"
    a3m = _fetch_msa_colabfold(query_aa, cache_path=a3m_path)
    if a3m is None:
        print(f"[{protein_index}] {uid}: no A3M found, skipping")
        continue

    records = _parse_a3m_records(a3m)
    if not records:
        print(f"[{protein_index}] {uid}: empty A3M, skipping")
        continue

    query_name, query_raw = records[0]
    query_aln = _strip_a3m_insertions(query_raw)
    query_3di = generate_folding_reference(query_aa)[:len(query_aa)]

    family_entries = []
    seen_aa = set()
    for name, raw_aln in records[1:]:
        homolog_aln = _strip_a3m_insertions(raw_aln)
        homolog_aa = _ungap_aa(homolog_aln)
        if len(homolog_aa) < FAMILY_MSA_MIN_SEQ_LEN or homolog_aa in seen_aa:
            continue
        seen_aa.add(homolog_aa)

        homolog_3di = generate_folding_reference(homolog_aa)
        projected = _project_homolog_3di_to_query(query_aln, homolog_aln, homolog_3di)
        projected_3di = "".join(tok if tok is not None else "-" for tok in projected)
        family_entries.append((name, homolog_aln, homolog_aa, homolog_3di, projected_3di))
        if len(family_entries) >= FAMILY_MSA_MAX_SEQS:
            break

    def add_row(section, role, member_index, member_name, sequence_type, aligned_sequence,
                member_aa_length, member_alignment_width, member_3di_length, projected_3di_length):
        aligned_family_rows.append({
            "protein_index": protein_index,
            "query_id": uid,
            "query_name": query_name,
            "section": section,
            "role": role,
            "member_index": member_index,
            "member_name": member_name,
            "sequence_type": sequence_type,
            "aligned_sequence": aligned_sequence,
            "query_aa_length": len(query_aa),
            "query_alignment_width": len(query_aln),
            "member_aa_length": member_aa_length,
            "member_alignment_width": member_alignment_width,
            "member_3di_length": member_3di_length,
            "projected_3di_length": projected_3di_length,
            "a3m_path": str(a3m_path),
        })

    add_row("header", "query", 0, query_name, "summary", f"Query Protein {protein_index}: {uid}",
            len(query_aa), len(query_aln), len(query_3di), len(query_3di))

    add_row("query_aa_family_aa_aligned", "query", 0, query_name, "AA_alignment", query_aln,
            len(query_aa), len(query_aln), len(query_3di), len(query_3di))
    for member_index, (name, homolog_aln, homolog_aa, homolog_3di, projected_3di) in enumerate(family_entries, start=1):
        add_row("query_aa_family_aa_aligned", "family", member_index, name, "AA_alignment", homolog_aln,
                len(homolog_aa), len(homolog_aln), len(homolog_3di), len(projected_3di))

    add_row("query_3di_family_3di_aligned", "query", 0, query_name, "3Di_projected_to_query", query_3di,
            len(query_aa), len(query_aln), len(query_3di), len(query_3di))
    for member_index, (name, homolog_aln, homolog_aa, homolog_3di, projected_3di) in enumerate(family_entries, start=1):
        add_row("query_3di_family_3di_aligned", "family", member_index, name, "3Di_projected_to_query", projected_3di,
                len(homolog_aa), len(homolog_aln), len(homolog_3di), len(projected_3di))

    add_row("length_table", "query", 0, query_name, "lengths", "",
            len(query_aa), len(query_aln), len(query_3di), len(query_3di))
    for member_index, (name, homolog_aln, homolog_aa, homolog_3di, projected_3di) in enumerate(family_entries, start=1):
        add_row("length_table", "family", member_index, name, "lengths", "",
                len(homolog_aa), len(homolog_aln), len(homolog_3di), len(projected_3di))

aligned_family_df = pd.DataFrame(aligned_family_rows)
aligned_family_df.to_csv(aligned_family_csv_path, index=False)
print(f"Saved {len(aligned_family_df)} rows to {aligned_family_csv_path}")
aligned_family_df.head(30)

